In [2]:
 %pip install anthropic

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import anthropic

os.environ['ANTHROPIC_API_KEY'] = 
client = anthropic.Anthropic()
print(os.getcwd())

/Users/matt/Desktop/Patent Litigation Analysis/notebooks


In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re as re

In [5]:
names=pd.read_csv('/Users/matt/Desktop/Patent Litigation Analysis/data/processed/names/names_cleaned_v4.csv')
cases=pd.read_csv('/Users/matt/Desktop/Patent Litigation Analysis/data/processed/cases/cases_processed_v4.csv')

In [6]:
individual_mask = names['name'].str.contains(r'\ban individual\b|\bDoes\b', case=False, na=False)
names.loc[individual_mask, 'entity_type'] = 'Independent'


In [7]:
import re

SUFFIX_MAP = [
    (r',?\s+Incorporated\.?$',  'Inc.'),
    (r',?\s+Inc[,\.]?$',        'Inc.'),
    (r',?\s+Corporation\.?$',   'Corp.'),
    (r',?\s+Corp[,\.]?$',       'Corp.'),
    (r',?\s+LLC\.?$',           'LLC'),
    (r',?\s+L\.L\.C\.?$',       'LLC'),
    (r',?\s+Limited\.?$',       'Ltd.'),
    (r',?\s+Ltd[,\.]?$',        'Ltd.'),
    (r',?\s+L\.P\.?$',          'L.P.'),
    (r',?\s+Company\.?$',       'Co.'),
    (r',?\s+Co[,\.]?$',         'Co.'),
]

def standardize_org_name(name: str) -> str:
    if not isinstance(name, str):
        return name

    name = name.strip()

    if name == name.upper():
        name = name.title()

    for pattern, replacement in SUFFIX_MAP:
        new_name = re.sub(pattern, ', ' + replacement, name, flags=re.IGNORECASE)
        if new_name != name:
            name = new_name
            break

    return name

org_mask = names['entity_type'] == 'Organization'
names.loc[org_mask, 'standardized_org_name'] = names.loc[org_mask, 'name'].apply(standardize_org_name)

In [8]:
PARENT_MAP = {

    "Apple, Inc.":                          "Apple Inc.",
    "Apple, Corp.":                         "Apple Inc.",
    "Apple Computer, Inc.":                 "Apple Inc.",
    "Apple Computer Company, Inc.":         "Apple Inc.",
    "Apple Computers, Inc.":                "Apple Inc.",
    "Apple Computer inc":                   "Apple Inc.",
    "Apple Incorporated":                   "Apple Inc.",
    "Apple Inc,":                           "Apple Inc.",
    "APPLE INC.":                           "Apple Inc.",
    "APPLE, INC.":                          "Apple Inc.",
    "Apple Inc":                            "Apple Inc.",

    "Samsung Electronics America, Inc.":            "Samsung Electronics",
    "Samsung Electronics Co., Ltd.":                "Samsung Electronics",
    "Samsung Telecommunications America, LLC":      "Samsung Electronics",
    "Samsung Semiconductor, Inc.":                  "Samsung Electronics",
    "Samsung Electronics Co, Ltd.":                 "Samsung Electronics",
    "Samsung TeleCommunications America, LLC":       "Samsung Electronics",
    "Samsung Electronics Co., LTD.,":               "Samsung Electronics",
    "Samsung Austin Semiconductor, LLC":            "Samsung Electronics",
    "Samsung Telecommunications America LLP":       "Samsung Electronics",
    "Samsung Electronics America, Inc.,":           "Samsung Electronics",
    "Samsung Electronics Company, Ltd.":            "Samsung Electronics",
    "Samsung Electronics, Co., Ltd.":               "Samsung Electronics",
    "Samsung SDI Co., Ltd.":                        "Samsung Electronics",
    "Samsung Opto-Electronics America, Inc.":       "Samsung Electronics",
    "Samsung Electronics USA, Inc.":                "Samsung Electronics",
    "Samsung Techwin Co., Ltd.":                    "Samsung Electronics",
    "Samsung SDI Co, Ltd.":                         "Samsung Electronics",
    "Samsung Telecommunications America, Inc.":     "Samsung Electronics",
    "Samsung America, Inc.":                        "Samsung Electronics",
    "Samsung Electronics America":                  "Samsung Electronics",
    "Samsung Electronics":                          "Samsung Electronics",
    "Samsung, Inc.":                                "Samsung Electronics",
    "Samsung, Corp.":                               "Samsung Electronics",
    "Samsung Electronics, Inc.":                    "Samsung Electronics",
    "Samsung Display Co., Ltd.":                    "Samsung Electronics",
    "Samsung Mobile Display Co., Ltd.":             "Samsung Electronics",
    "Samsung Electro-Mechanics Co., Ltd.":          "Samsung Electronics",
    "Samsung TeleCommunications America, Inc.":     "Samsung Electronics",
    "Samsung Mobile":                               "Samsung Electronics",

    "Sony Electronics, Inc.":                       "Sony Corp.",
    "Sony, Corp.":                                  "Sony Corp.",
    "Sony Corporation of America":                  "Sony Corp.",
    "Sony Ericsson Mobile Communications (USA), Inc.": "Sony Corp.",
    "Sony Computer Entertainment America, Inc.":    "Sony Corp.",
    "Sony Mobile Communications (USA), Inc.":       "Sony Corp.",
    "Sony Computer Entertainment America, LLC":     "Sony Corp.",
    "Sony Computer Entertainment, Inc.":            "Sony Corp.",
    "Sony Mobile Communications AB":                "Sony Corp.",
    "Sony BMG Music Entertainment":                 "Sony Corp.",
    "Sony DADC US, Inc.":                           "Sony Corp.",
    "Sony Online Entertainment, LLC":               "Sony Corp.",
    "Sony Music Entertainment, Inc.":               "Sony Corp.",
    "Sony Pictures Entertainment, Inc.":            "Sony Corp.",
    "Sony Corp of America":                         "Sony Corp.",
    "Sony Corporation Of America":                  "Sony Corp.",
    "Sony Interactive Entertainment, Inc.":         "Sony Corp.",

    "HTC America, Inc.":                    "HTC Corp.",
    "HTC, Corp.":                           "HTC Corp.",
    "HTC (BVI), Corp.":                     "HTC Corp.",
    "HTC BVI, Corp.":                       "HTC Corp.",
    "HTC America, Inc.,":                   "HTC Corp.",
    "HTC (B.V.I.), Corp.":                  "HTC Corp.",
    "HTC America Holding, Inc.":            "HTC Corp.",
    "HTC USA, Inc.":                        "HTC Corp.",
    "HTC America Innovation, Inc.":         "HTC Corp.",
    "HTC, Inc.":                            "HTC Corp.",
    "HTC America":                          "HTC Corp.",
    "HTC America Holdings, Inc.":           "HTC Corp.",

    "LG Electronics, Inc.":                 "LG Electronics",
    "LG Electronics USA, Inc.":             "LG Electronics",
    "LG Electronics U.S.A., Inc.":          "LG Electronics",
    "LG Electronics Mobilecomm U.S.A., Inc.": "LG Electronics",
    "LG Electronics Mobilecomm USA, Inc.":  "LG Electronics",
    "LG Electronics MobileComm U.S.A., Inc.": "LG Electronics",
    "LG Electronics MobileComm USA, Inc.":  "LG Electronics",
    "LG Display Co., Ltd.":                 "LG Electronics",
    "LG Display America, Inc.":             "LG Electronics",
    "LG Electronics U.S.A., Inc.,":         "LG Electronics",
    "LG Electronics":                       "LG Electronics",
    "LG Electronics, U.S.A., Inc.":         "LG Electronics",

    "Microsoft, Corp.":                     "Microsoft Corp.",
    "Microsoft, Corp":                      "Microsoft Corp.",

    "Google, Inc.":                         "Google Inc.",

    "Amazon.com, Inc.":                     "Amazon.com Inc.",

    "Dell, Inc.":                           "Dell Inc.",

    "Intel, Corp.":                         "Intel Corp.",

    "Cisco Systems, Inc.":                  "Cisco Systems Inc.",

    "Qualcomm, Inc.":                       "Qualcomm Inc.",
    "Qualcomm Technologies, Inc.":          "Qualcomm Inc.",
    "Qualcomm Atheros, Inc.":               "Qualcomm Inc.",
    "QUALCOMM, Inc.":                       "Qualcomm Inc.",
    "Qualcomm-Atheros, Inc.":               "Qualcomm Inc.",
    "Qualcomm Innovation Center, Inc.":     "Qualcomm Inc.",

    "Broadcom, Corp.":                      "Broadcom Corp.",
    "BROADCOM, Corp.":                      "Broadcom Corp.",

    "Motorola, Inc.":                       "Motorola Inc.",
    "Motorola Mobility, Inc.":              "Motorola Inc.",
    "Motorola Mobility, LLC":               "Motorola Inc.",
    "Motorola Solutions, Inc.":             "Motorola Inc.",
    "Motorola Mobility Holdings, Inc.":     "Motorola Inc.",
    "Motorola, Corp.":                      "Motorola Inc.",
    "Motorola Mobility Holdings, LLC":      "Motorola Inc.",
    "Motorola Mobility International, Ltd.": "Motorola Inc.",

    "Nokia, Inc.":                          "Nokia Corp.",
    "Nokia, Corp.":                         "Nokia Corp.",
    "Nokia Mobile Phones, Inc.":            "Nokia Corp.",
    "Nokia Holding, Inc.":                  "Nokia Corp.",
    "Nokia Siemens Networks US, LLC":       "Nokia Corp.",
    "Nokia Mobile Phones Americas, Inc.":   "Nokia Corp.",
    "Nokia USA, Inc.":                      "Nokia Corp.",
    "Nokia Mobile Phones, Ltd.":            "Nokia Corp.",
    "Nokia Networks, Inc.":                 "Nokia Corp.",
    "Nokia Products, Corp.":                "Nokia Corp.",
    "Nokia Mobile Phones":                  "Nokia Corp.",

    "BlackBerry, Corp.":                    "BlackBerry Ltd.",
    "BlackBerry, Ltd.":                     "BlackBerry Ltd.",
    "Blackberry, Ltd.":                     "BlackBerry Ltd.",
    "Blackberry, Corp.":                    "BlackBerry Ltd.",
    "Research In Motion, Corp.":            "BlackBerry Ltd.",
    "Research In Motion, Ltd.":             "BlackBerry Ltd.",
    "Research in Motion, Ltd.":             "BlackBerry Ltd.",
    "Research in Motion, Corp.":            "BlackBerry Ltd.",
    "Research In Motion, Inc.":             "BlackBerry Ltd.",

    "Huawei Device USA, Inc.":              "Huawei Technologies",
    "Huawei Technologies USA, Inc.":        "Huawei Technologies",
    "Huawei Technologies Co., Ltd.":        "Huawei Technologies",
    "Huawei Devices USA, Inc.":             "Huawei Technologies",
    "Huawei Technologies Co, Ltd.":         "Huawei Technologies",
    "Huawei Device Co., Ltd.":              "Huawei Technologies",
    "Huawei America, Inc.":                 "Huawei Technologies",
    "Huawei Technologies, Co., Ltd.":       "Huawei Technologies",
    "Huawei Technologies USA":              "Huawei Technologies",

    "Toshiba, Corp.":                       "Toshiba Corp.",
    "Toshiba America Information Systems, Inc.": "Toshiba Corp.",
    "Toshiba America, Inc.":                "Toshiba Corp.",
    "Toshiba America Electronic Components, Inc.": "Toshiba Corp.",
    "Toshiba America Consumer Products, LLC": "Toshiba Corp.",
    "Toshiba America Medical Systems, Inc.": "Toshiba Corp.",
    "Toshiba America Business Solutions, Inc.": "Toshiba Corp.",
    "Toshiba International, Corp.":         "Toshiba Corp.",

    "Panasonic Corporation of North America": "Panasonic Corp.",
    "Panasonic, Corp.":                     "Panasonic Corp.",
    "Panasonic Corp. of North America":     "Panasonic Corp.",
    "Panasonic Corporation Of North America": "Panasonic Corp.",
    "Panasonic Consumer Electronics, Co.":  "Panasonic Corp.",
    "Panasonic Corporation of North America, Inc.": "Panasonic Corp.",
    "Panasonic Corporation of America":     "Panasonic Corp.",

    "AT&T Mobility, LLC":                   "AT&T Inc.",
    "AT&T, Inc.":                           "AT&T Inc.",
    "AT&T, Corp.":                          "AT&T Inc.",
    "AT&T Services, Inc.":                  "AT&T Inc.",
    "AT&T Operations, Inc.":                "AT&T Inc.",
    "AT&T Wireless Services, Inc.":         "AT&T Inc.",
    "AT&T Mobility II, LLC":                "AT&T Inc.",
    "AT&T Internet Services":               "AT&T Inc.",
    "At&T, Corp.":                          "AT&T Inc.",
    "At&T Mobility, LLC":                   "AT&T Inc.",
    "AT&T Mobility, Corp.":                 "AT&T Inc.",

    "Verizon Communications, Inc.":         "Verizon Communications",
    "Cellco Partnership d/b/a Verizon Wireless": "Verizon Communications",
    "Verizon Services, Corp.":              "Verizon Communications",
    "Verizon Wireless":                     "Verizon Communications",
    "Verizon Business Network Services, Inc.": "Verizon Communications",
    "Verizon Online, LLC":                  "Verizon Communications",
    "Verizon Data Services, LLC":           "Verizon Communications",
    "Verizon South, Inc.":                  "Verizon Communications",
    "Verizon Wireless, Inc.":               "Verizon Communications",
    "Verizon Virginia, Inc.":               "Verizon Communications",
    "Verizon California, Inc.":             "Verizon Communications",

    "Sprint Spectrum, L.P.":                "Sprint Corp.",
    "Sprint Nextel, Corp.":                 "Sprint Corp.",
    "Sprint Solutions, Inc.":               "Sprint Corp.",
    "Sprint, Corp.":                        "Sprint Corp.",
    "Sprint Communications Company, L.P.":  "Sprint Corp.",
    "Sprint Communications, Inc.":          "Sprint Corp.",
    "Sprint Spectrum LP":                   "Sprint Corp.",

    "T-Mobile USA, Inc.":                   "T-Mobile USA Inc.",

    "Comcast, Corp.":                       "Comcast Corp.",
    "Comcast Cable Communications, LLC":    "Comcast Corp.",
    "Comcast Interactive Media, LLC":       "Comcast Corp.",

    "Hewlett-Packard, Co.":                 "HP Inc.",
    "Hewlett Packard, Co.":                 "HP Inc.",
    "Hewlett-Packard Development Company, L.P.": "HP Inc.",
    "Hewlett-Packard, Corp.":               "HP Inc.",
    "Hewlett Packard Enterprise, Co.":      "HP Inc.",
    "Hewlett Packard, Corp.":               "HP Inc.",

    "International Business Machines, Corp.": "IBM Corp.",

    "Oracle, Corp.":                        "Oracle Corp.",

    "Adobe Systems, Inc.":                  "Adobe Inc.",

    "Symantec, Corp.":                      "Symantec Corp.",

    "Canon, Inc.":                          "Canon Inc.",
    "Canon U.S.A., Inc.":                   "Canon Inc.",
    "Canon USA, Inc.":                      "Canon Inc.",
    "Canon Solutions America, Inc.":        "Canon Inc.",
    "Canon Computer Systems, Inc.":         "Canon Inc.",

    "Fujitsu, Ltd.":                        "Fujitsu Ltd.",
    "Fujitsu America, Inc.":                "Fujitsu Ltd.",
    "Fujitsu Computer Products of America, Inc.": "Fujitsu Ltd.",
    "Fujitsu Network Communications, Inc.": "Fujitsu Ltd.",
    "Fujitsu Computer Systems, Corp.":      "Fujitsu Ltd.",
    "Fujitsu Microelectronics, Inc.":       "Fujitsu Ltd.",
    "Fujitsu Semiconductor, Ltd.":          "Fujitsu Ltd.",
    "Fujitsu Semiconductor America, Inc.":  "Fujitsu Ltd.",

    "Epson America, Inc.":                  "Seiko Epson Corp.",
    "Seiko Epson, Corp.":                   "Seiko Epson Corp.",

    "Nintendo of America, Inc.":            "Nintendo Co. Ltd.",
    "Nintendo Co., Ltd.":                   "Nintendo Co. Ltd.",
    "Nintendo Co, Ltd.":                    "Nintendo Co. Ltd.",
    "Nintendo Company, Ltd.":               "Nintendo Co. Ltd.",
    "Nintendo Of America, Inc.":            "Nintendo Co. Ltd.",
    "Nintendo of America":                  "Nintendo Co. Ltd.",
    "Nintendo of North America, Inc.":      "Nintendo Co. Ltd.",

    "Acer America, Corp.":                  "Acer Inc.",
    "Acer, Inc.":                           "Acer Inc.",

    "General Electric, Co.":               "General Electric Co.",
    "General Electric Capital, Corp.":     "General Electric Co.",
    "General Electric, Corp.":             "General Electric Co.",
    "The General Electric, Co.":           "General Electric Co.",

    "General Motors, Corp.":               "General Motors Co.",
    "General Motors, LLC":                 "General Motors Co.",
    "General Motors, Co.":                 "General Motors Co.",
    "General Motors Acceptance, Corp.":    "General Motors Co.",
    "General Motors of Canada, Ltd.":      "General Motors Co.",

    "Ford Motor, Co.":                     "Ford Motor Co.",

    "BMW of North America, LLC":           "BMW AG",
    "Bayerische Motoren Werke AG":         "BMW AG",

    "Mercedes-Benz USA, LLC":              "Mercedes-Benz Group",
    "Mercedes-Benz U.S. International, Inc.": "Mercedes-Benz Group",
    "Mercedes Benz USA, LLC":              "Mercedes-Benz Group",
    "Mercedes-Benz Usa, LLC":              "Mercedes-Benz Group",
    "Mercedes-Benz of North America, Inc.": "Mercedes-Benz Group",

    "Nissan North America, Inc.":          "Nissan Motor Co.",
    "Nissan Motor Co., Ltd.":              "Nissan Motor Co.",
    "Nissan Motor Co, Ltd.":               "Nissan Motor Co.",
    "Nissan Motor Company, Ltd.":          "Nissan Motor Co.",
    "Nissan Motor, Co.":                   "Nissan Motor Co.",

    "Hyundai Motor America":               "Hyundai Motor Co.",
    "Hyundai Motor, Co.":                  "Hyundai Motor Co.",
    "Hyundai Motor America, Inc.":         "Hyundai Motor Co.",
    "Hyundai Motor Manufacturing Alabama, LLC": "Hyundai Motor Co.",

    "Wal-Mart Stores, Inc.":               "Walmart Inc.",
    "Wal-Mart Stores Texas, LLC":          "Walmart Inc.",
    "Wal-Mart, Inc.":                      "Walmart Inc.",
    "Wal-Mart Stores East, LP":            "Walmart Inc.",
    "Wal-Mart Stores":                     "Walmart Inc.",

    "Target, Corp.":                       "Target Corp.",

    "Best Buy Co., Inc.":                  "Best Buy Co.",
    "Best Buy Stores, L.P.":               "Best Buy Co.",
    "Best Buy Co, Inc.":                   "Best Buy Co.",
    "Best Buy Company, Inc.":              "Best Buy Co.",
    "Best Buy, Co.":                       "Best Buy Co.",

    "Costco Wholesale, Corp.":             "Costco Wholesale Corp.",

    "Nike, Inc.":                          "Nike Inc.",

    "3M, Co.":                             "3M Co.",
    "3M Innovative Properties, Co.":       "3M Co.",
    "Minnesota Mining and Manufacturing, Co.": "3M Co.",

    "Facebook, Inc.":                      "Meta Platforms Inc.",

    "Yahoo!, Inc.":                        "Yahoo Inc.",

    "eBay, Inc.":                          "eBay Inc.",

    "Netflix, Inc.":                       "Netflix Inc.",

    "Pfizer, Inc.":                        "Pfizer Inc.",
    "Pfizer Ireland Pharmaceuticals":      "Pfizer Inc.",
    "Pfizer, Ltd.":                        "Pfizer Inc.",
    "Pfizer Pharmaceuticals, LLC":         "Pfizer Inc.",

    "Novartis Pharmaceuticals, Corp.":     "Novartis AG",
    "Novartis AG":                         "Novartis AG",
    "Novartis, Corp.":                     "Novartis AG",
    "Novartis Pharma AG":                  "Novartis AG",
    "Novartis Pharma Ag":                  "Novartis AG",
    "Novartis International Pharmaceutical, Ltd.": "Novartis AG",
    "Novartis Consumer Health, Inc.":      "Novartis AG",
    "Novartis International AG":           "Novartis AG",

    "Teva Pharmaceuticals USA, Inc.":      "Teva Pharmaceutical Industries",
    "Teva Pharmaceuticals Usa, Inc.":      "Teva Pharmaceutical Industries",
    "Teva Pharmaceutical Industries, Ltd.": "Teva Pharmaceutical Industries",
    "Teva Pharmaceuticals Industries, Ltd.": "Teva Pharmaceutical Industries",
    "Teva Neuroscience, Inc.":             "Teva Pharmaceutical Industries",
    "Teva Pharmaceuticals U.S.A., Inc.":   "Teva Pharmaceutical Industries",
    "Teva Pharmaceutical USA, Inc.":       "Teva Pharmaceutical Industries",

    "Mylan Pharmaceuticals, Inc.":         "Mylan Inc.",
    "Mylan, Inc.":                         "Mylan Inc.",
    "Mylan Laboratories, Inc.":            "Mylan Inc.",
    "Mylan Technologies, Inc.":            "Mylan Inc.",
    "Mylan N.V.":                          "Mylan Inc.",

    "Astrazeneca Ab":                      "AstraZeneca PLC",
    "AstraZeneca Pharmaceuticals LP":      "AstraZeneca PLC",
    "Astrazeneca Pharmaceuticals Lp":      "AstraZeneca PLC",
    "AstraZeneca AB":                      "AstraZeneca PLC",
    "Astrazeneca Uk, Ltd.":                "AstraZeneca PLC",
    "Astrazeneca Lp":                      "AstraZeneca PLC",
    "AstraZeneca UK, Ltd.":                "AstraZeneca PLC",
    "Astrazeneca AB":                      "AstraZeneca PLC",
    "AstraZeneca LP":                      "AstraZeneca PLC",
    "Astrazeneca Pharmaceuticals LP":      "AstraZeneca PLC",

    "Abbott Laboratories":                 "Abbott Laboratories",
    "Abbott Laboratories, Inc.":           "Abbott Laboratories",
    "Abbott Labs":                         "Abbott Laboratories",
    "Abbott Cardiovascular Systems, Inc.": "Abbott Laboratories",
    "Abbott Diabetes Care, Inc.":          "Abbott Laboratories",
    "Abbott Vascular, Inc.":               "Abbott Laboratories",

    "Merck & Co., Inc.":                   "Merck & Co.",
    "Merck Sharp & Dohme, Corp.":          "Merck & Co.",
    "Merck & Co, Inc.":                    "Merck & Co.",
    "Merck Sharp & Dohme B.V.":            "Merck & Co.",
    "Merck & Company, Inc.":               "Merck & Co.",

    "Johnson & Johnson":                   "Johnson & Johnson",
    "Johnson & Johnson, Inc.":             "Johnson & Johnson",
    "Johnson & Johnson Vision Care, Inc.": "Johnson & Johnson",
    "Johnson & Johnson Pharmaceutical Research & Development, LLC": "Johnson & Johnson",

    "Glaxo Group, Ltd.":                   "GlaxoSmithKline PLC",
    "Glaxo Wellcome, Inc.":                "GlaxoSmithKline PLC",
    "GlaxoSmithKline, LLC":                "GlaxoSmithKline PLC",
    "Glaxosmithkline, LLC":                "GlaxoSmithKline PLC",
    "GlaxoSmithKline":                     "GlaxoSmithKline PLC",
    "Glaxosmithkline, Inc.":               "GlaxoSmithKline PLC",
    "Glaxosmithkline PLC":                 "GlaxoSmithKline PLC",
    "GlaxoSmithKline PLC":                 "GlaxoSmithKline PLC",
    "Glaxo, Inc.":                         "GlaxoSmithKline PLC",

    "Sanofi-Aventis U.S., LLC":            "Sanofi S.A.",
    "Sanofi-Aventis":                      "Sanofi S.A.",
    "Sanofi":                              "Sanofi S.A.",
    "Sanofi-Aventis US, LLC":              "Sanofi S.A.",
    "Sanofi-Aventis U.S., Inc.":           "Sanofi S.A.",
    "Sanofi-Synthelabo, Inc.":             "Sanofi S.A.",

    "Hoffmann-La Roche, Inc.":             "Roche Holding AG",
    "Roche Palo Alto, LLC":                "Roche Holding AG",
    "Roche Molecular Systems, Inc.":       "Roche Holding AG",
    "Hoffman-La Roche, Inc.":              "Roche Holding AG",
    "Roche Diagnostics Operations, Inc.":  "Roche Holding AG",
    "Roche Diagnostics, Corp.":            "Roche Holding AG",
    "Roche Laboratories, Inc.":            "Roche Holding AG",
    "F. Hoffmann-La Roche, Ltd.":          "Roche Holding AG",
    "Hoffman-LaRoche, Inc.":               "Roche Holding AG",

    "Bristol-Myers Squibb, Co.":           "Bristol-Myers Squibb Co.",
    "Bristol-Meyers Squibb, Co.":          "Bristol-Myers Squibb Co.",
    "Bristol-Myers Squibb Pharma, Co.":    "Bristol-Myers Squibb Co.",
    "Bristol Myers Squibb, Co.":           "Bristol-Myers Squibb Co.",
    "Bristol-Myers, Co.":                  "Bristol-Myers Squibb Co.",

    "Bayer, Corp.":                        "Bayer AG",
    "Bayer Healthcare Pharmaceuticals, Inc.": "Bayer AG",
    "Bayer AG":                            "Bayer AG",
    "Bayer Schering Pharma AG":            "Bayer AG",
    "Bayer Pharma AG":                     "Bayer AG",
    "Bayer Healthcare, LLC":               "Bayer AG",
    "Bayer HealthCare Pharmaceuticals, Inc.": "Bayer AG",

    "Lupin, Ltd.":                         "Lupin Ltd.",
    "Lupin Pharmaceuticals, Inc.":         "Lupin Ltd.",
    "Lupin Atlantis Holdings S.A.":        "Lupin Ltd.",
    "Lupin, Inc.":                         "Lupin Ltd.",

    "Ranbaxy Laboratories, Ltd.":          "Ranbaxy Laboratories",
    "Ranbaxy Pharmaceuticals, Inc.":       "Ranbaxy Laboratories",
    "Ranbaxy, Inc.":                       "Ranbaxy Laboratories",
    "Ranbaxy Laboratories, Inc.":          "Ranbaxy Laboratories",

    "Watson Laboratories, Inc.":           "Actavis Inc.",
    "Watson Pharmaceuticals, Inc.":        "Actavis Inc.",
    "Watson Pharma, Inc.":                 "Actavis Inc.",
    "Actavis, Inc.":                       "Actavis Inc.",
    "Watson Pharmaceutical, Inc.":         "Actavis Inc.",

    "Sandoz, Inc.":                        "Sandoz Inc.",

    "Apotex, Inc.":                        "Apotex Inc.",
    "Apotex, Corp.":                       "Apotex Inc.",

    "Philips Electronics North America, Corp.": "Koninklijke Philips N.V.",
    "U.S. Philips, Corp.":                 "Koninklijke Philips N.V.",
    "Koninklijke Philips Electronics N.V.": "Koninklijke Philips N.V.",
    "Koninklijke Philips N.V.":            "Koninklijke Philips N.V.",
    "US Philips, Corp.":                   "Koninklijke Philips N.V.",
    "North American Philips, Corp.":       "Koninklijke Philips N.V.",

    "Xerox, Corp.":                        "Xerox Corp.",

    "NCR, Corp.":                          "NCR Corp.",

    "Texas Instruments, Inc.":             "Texas Instruments Inc.",

    "Advanced Micro Devices, Inc.":        "AMD Inc.",

    "Whirlpool, Corp.":                    "Whirlpool Corp.",

    "Emerson Electric, Co.":               "Emerson Electric Co.",

    "Honeywell International, Inc.":       "Honeywell International Inc.",

    "Stryker, Corp.":                      "Stryker Corp.",

    "Medtronic, Inc.":                     "Medtronic Inc.",

    "Boston Scientific, Corp.":            "Boston Scientific Corp.",
    "Boston Scientific Scimed, Inc.":      "Boston Scientific Corp.",

    "Genentech, Inc.":                     "Genentech Inc.",

    "Monsanto, Co.":                       "Monsanto Co.",
    "Monsanto Technology, LLC":            "Monsanto Co.",

    "Eastman Kodak, Co.":                  "Eastman Kodak Co.",

    "Garmin International, Inc.":          "Garmin Ltd.",

    "Juniper Networks, Inc.":              "Juniper Networks Inc.",

    "NEC, Corp.":                          "NEC Corp.",

    "Sharp Electronics, Corp.":            "Sharp Corp.",
    "Sharp, Corp.":                        "Sharp Corp.",

    "Hitachi, Ltd.":                       "Hitachi Ltd.",

    "Ericsson, Inc.":                      "Ericsson AB",
    "Telefonaktiebolaget LM Ericsson":     "Ericsson AB",

    "ZTE (USA), Inc.":                     "ZTE Corp.",
    "ZTE, Corp.":                          "ZTE Corp.",

    "Celgene, Corp.":                      "Celgene Corp.",

    "Allergan, Inc.":                      "Allergan Inc.",

    "Genzyme, Corp.":                      "Genzyme Corp.",

    "Illinois Tool Works, Inc.":           "Illinois Tool Works Inc.",

    "Kimberly-Clark, Corp.":               "Kimberly-Clark Corp.",

    "Ecolab, Inc.":                        "Ecolab Inc.",

    "American Express, Co.":               "American Express Co.",

    "Sears Holdings, Corp.":               "Sears Holdings Corp.",

    "Office Depot, Inc.":                  "Office Depot Inc.",

    "Staples, Inc.":                       "Staples Inc.",

    "Walgreen, Co.":                       "Walgreens Co.",

    "Rite Aid, Corp.":                     "Rite Aid Corp.",

    "Barnes & Noble, Inc.":                "Barnes & Noble Inc.",

    "GeoTag, Inc.":                        "GeoTag Inc.",
    "Geotag, Inc.":                        "GeoTag Inc.",

    "Orion Ip, LLC":                       "Orion IP, LLC",
    "Orion IP, LLC Orion IP, LLC":         "Orion IP, LLC",

    "Amazon.Com, Inc.":                    "Amazon.com Inc.",
    "Amazon.Com, Inc.,":                   "Amazon.com Inc.",
    "Amazon Web Services, LLC":            "Amazon.com Inc.",
    "Amazon.com, LLC":                     "Amazon.com Inc.",
    "Amazon Digital Services, Inc.":       "Amazon.com Inc.",
    "Amazon Services, LLC":                "Amazon.com Inc.",
    "Amazon Web Services, Inc.":           "Amazon.com Inc.",
    "Amazon Technologies, Inc.":           "Amazon.com Inc.",
    "Amazon Payments, Inc.":               "Amazon.com Inc.",
    "Amazon Services, Inc.":               "Amazon.com Inc.",
    "Amazon Com, Inc.":                    "Amazon.com Inc.",
    "Amazon, Inc.":                        "Amazon.com Inc.",
    "Amazon":                              "Amazon.com Inc.",
    "Amazon Technologies":                 "Amazon.com Inc.",
    "Amazon.Com,Inc.":                     "Amazon.com Inc.",
    "AmazonFresh, LLC":                    "Amazon.com Inc.",

    "ATT Mobility, LLC":                   "AT&T Inc.",
    "ATT, Inc.":                           "AT&T Inc.",
    "ATT, Corp.":                          "AT&T Inc.",
    "ATT Services, Inc.":                  "AT&T Inc.",
    "ATT Mobility, Corp.":                 "AT&T Inc.",
    "ATT Wireless Services, Inc.":         "AT&T Inc.",
    "ATT Intellectual Property I, L.P.":   "AT&T Inc.",
    "ATT Intellectual Property II, L.P.":  "AT&T Inc.",
    "ATT Mobility II, LLC":                "AT&T Inc.",
    "ATT Video Services, Inc.":            "AT&T Inc.",
    "Att Mobility, LLC":                   "AT&T Inc.",
    "Att":                                 "AT&T Inc.",
    "At&T, Inc.":                          "AT&T Inc.",
    "At&T Services, Inc.":                 "AT&T Inc.",

    "BestBuy.com, LLC":                    "Best Buy Co.",
    "Bestbuy.com, LLC":                    "Best Buy Co.",
    "BestBuy.Com, LLC":                    "Best Buy Co.",
    "Bestbuy.Com, LLC":                    "Best Buy Co.",
    "BestBuy Co., Inc.":                   "Best Buy Co.",
    "BestBuy, Inc.":                       "Best Buy Co.",

    "Dr. Reddy'S Laboratories, Ltd.":      "Dr. Reddy's Laboratories",
    "Dr. Reddy'S Laboratories, Inc.":      "Dr. Reddy's Laboratories",
    "Dr. Reddy's Laboratories, Inc.":      "Dr. Reddy's Laboratories",
    "Dr. Reddy's Laboratories, Ltd.":      "Dr. Reddy's Laboratories",
    "Dr. Reddys Laboratories, Ltd.":       "Dr. Reddy's Laboratories",
    "Dr. Reddys Laboratories, Inc.":       "Dr. Reddy's Laboratories",
    "Dr. Reddy'S Pharmaceuticals, Ltd.":   "Dr. Reddy's Laboratories",
    "Dr. Reddy'S Pharmaceuticals, Inc.":   "Dr. Reddy's Laboratories",

    "Actavis Elizabeth, LLC":              "Actavis Inc.",
    "Actavis Pharma, Inc.":                "Actavis Inc.",
    "Actavis Laboratories Fl, Inc.":       "Actavis Inc.",
    "Actavis Laboratories FL, Inc.":       "Actavis Inc.",
    "Actavis, LLC":                        "Actavis Inc.",
    "Actavis South Atlantic, LLC":         "Actavis Inc.",
    "Actavis Mid Atlantic, LLC":           "Actavis Inc.",
    "Actavis Plc":                         "Actavis Inc.",
    "Actavis plc":                         "Actavis Inc.",
    "Actavis Group":                       "Actavis Inc.",
    "Actavis Totowa, LLC":                 "Actavis Inc.",

    "Cellco Partnership":                  "Verizon Communications",
    "Cellco Partnership, Inc.":            "Verizon Communications",
    "Cellco Partnership d/b/a/ Verizon Wireless": "Verizon Communications",
    "Cellco Partnership, d/b/a Verizon Wireless": "Verizon Communications",

    "Kyocera Communications, Inc.":        "Kyocera Corp.",
    "Kyocera Wireless, Corp.":             "Kyocera Corp.",
    "Kyocera, Corp.":                      "Kyocera Corp.",
    "Kyocera International, Inc.":         "Kyocera Corp.",
    "Kyocera America, Inc.":               "Kyocera Corp.",
    "Kyocera Mita America, Inc.":          "Kyocera Corp.",
    "Kyocera Optics, Inc.":                "Kyocera Corp.",
    "Kyocera Document Solutions America, Inc.": "Kyocera Corp.",
    "Kyocera Document Solutions, Inc.":    "Kyocera Corp.",

    "Ricoh Company, Ltd.":                 "Ricoh Co. Ltd.",
    "Ricoh Americas, Corp.":               "Ricoh Co. Ltd.",
    "Ricoh, Corp.":                        "Ricoh Co. Ltd.",
    "Ricoh Electronics, Inc.":             "Ricoh Co. Ltd.",
    "Ricoh Innovations, Inc.":             "Ricoh Co. Ltd.",
    "Ricoh USA, Inc.":                     "Ricoh Co. Ltd.",
    "Ricoh Co, Ltd.":                      "Ricoh Co. Ltd.",
    "RICOH Company, Ltd.":                 "Ricoh Co. Ltd.",
    "Ricoh Company, Inc.":                 "Ricoh Co. Ltd.",

    "Matsushita Electric Industrial Co., Ltd.":     "Panasonic Corp.",
    "Matsushita Electric Corporation of America":   "Panasonic Corp.",
    "Matsushita Electric Industrial Company, Ltd.": "Panasonic Corp.",
    "Matsushita Electric Corp. Of America":         "Panasonic Corp.",
    "Matsushita Electric Industrial Co, Ltd.":      "Panasonic Corp.",
    "Matsushita Electric, Corp.":                   "Panasonic Corp.",

    "Sanyo North America, Corp.":          "Sanyo Electric Co.",
    "Sanyo Electric Co., Ltd.":            "Sanyo Electric Co.",
    "Sanyo Electric Co, Ltd.":             "Sanyo Electric Co.",
    "Sanyo Manufacturing, Corp.":          "Sanyo Electric Co.",
    "Sanyo Fisher, Co.":                   "Sanyo Electric Co.",
    "Sanyo North America":                 "Sanyo Electric Co.",
    "Sanyo Electric Company, Ltd.":        "Sanyo Electric Co.",
    "Sanyo, Corp.":                        "Sanyo Electric Co.",

    "UCB, Inc.":                           "UCB S.A.",
    "UCB Pharma GmbH":                     "UCB S.A.",
    "Ucb S.A.":                            "UCB S.A.",
    "UCB Pharma, Inc.":                    "UCB S.A.",
    "UCB Societe Anonyme":                 "UCB S.A.",
    "Ucb, Inc.":                           "UCB S.A.",
    "UCB Manufacturing, Inc.":             "UCB S.A.",
    "UCB Pharma S.A.":                     "UCB S.A.",

    "Aventis Pharmaceuticals, Inc.":       "Sanofi S.A.",
    "Aventis Pharma S.A.":                 "Sanofi S.A.",
    "Aventis, Inc.":                       "Sanofi S.A.",
    "Aventis Pharma SA":                   "Sanofi S.A.",
    "Aventis Pasteur, Inc.":               "Sanofi S.A.",
    "Aventis Pasteur, Ltd.":               "Sanofi S.A.",

    "Schering, Corp.":                     "Merck & Co.",
    "Schering-Plough, Corp.":              "Merck & Co.",
    "Schering-Plough Healthcare Products, Inc.": "Merck & Co.",
    "Schering-Plough Research Institute":  "Merck & Co.",
    "Schering Plough, Corp.":              "Merck & Co.",

    "Wyeth":                               "Pfizer Inc.",
    "Wyeth, LLC":                          "Pfizer Inc.",
    "Wyeth Pharmaceuticals, Inc.":         "Pfizer Inc.",
    "Wyeth Holdings, Corp.":               "Pfizer Inc.",
    "Wyeth Laboratories, Inc.":            "Pfizer Inc.",
    "Wyeth-Ayerst Pharmaceuticals, Inc.":  "Pfizer Inc.",

    "Lucent Technologies, Inc.":           "Nokia Corp.",
    "Alcatel-Lucent USA, Inc.":            "Nokia Corp.",
    "Alcatel-Lucent Holdings, Inc.":       "Nokia Corp.",
    "Alcatel-Lucent":                      "Nokia Corp.",
    "Alcatel-Lucent S.A.":                 "Nokia Corp.",
    "Alcatel-Lucent, Inc.":                "Nokia Corp.",
    "Alcatel Lucent USA, Inc.":            "Nokia Corp.",
    "Alcatel Lucent SA":                   "Nokia Corp.",

    "Sun Microsystems, Inc.":              "Oracle Corp.",
    "Sun microsystems, Inc.":              "Oracle Corp.",
    "Sun Microsystems of California, Inc.": "Oracle Corp.",
    "Sun Microsystems Federal, Inc.":      "Oracle Corp.",

    "Palm, Inc.":                          "Palm Inc.",
    "Palm, Inc.,":                         "Palm Inc.",
    "PalmOne, Inc.":                       "Palm Inc.",
    "palmOne, Inc.":                       "Palm Inc.",
    "Palmone, Inc.":                       "Palm Inc.",
    "Palm Computing, Inc.":                "Palm Inc.",

    "Medtronic Vascular, Inc.":            "Medtronic Inc.",
    "Medtronic USA, Inc.":                 "Medtronic Inc.",
    "Medtronic Sofamor Danek USA, Inc.":   "Medtronic Inc.",
    "Medtronic Xomed, Inc.":               "Medtronic Inc.",
    "Medtronic Minimed, Inc.":             "Medtronic Inc.",
    "Medtronic MiniMed, Inc.":             "Medtronic Inc.",
    "Medtronic Navigation, Inc.":          "Medtronic Inc.",
    "Medtronics, Inc.":                    "Medtronic Inc.",

    "Alcon Laboratories, Inc.":            "Alcon Inc.",
    "Alcon Research, Ltd.":                "Alcon Inc.",
    "Alcon Pharmaceuticals, Ltd.":         "Alcon Inc.",
    "Alcon, Inc.":                         "Alcon Inc.",
    "Alcon Surgical, Inc.":                "Alcon Inc.",
    "Alcon Manufacturing, Ltd.":           "Alcon Inc.",
    "Alcon Laboratories Holding, Corp.":   "Alcon Inc.",
    "Alcon Universal, Ltd.":               "Alcon Inc.",

    "Eli Lilly And, Co.":                  "Eli Lilly and Co.",
    "Eli Lilly and, Co.":                  "Eli Lilly and Co.",
    "Eli Lilly &, Co.":                    "Eli Lilly and Co.",
    "Eli Lilly & Co., Inc.":               "Eli Lilly and Co.",
    "Eli Lilly & Company, Inc.":           "Eli Lilly and Co.",

    "Toyota Motor, Corp.":                 "Toyota Motor Corp.",
    "Toyota Motor North America, Inc.":    "Toyota Motor Corp.",
    "Toyota Motor Sales, U.S.A., Inc.":    "Toyota Motor Corp.",
    "Toyota Motor Sales USA, Inc.":        "Toyota Motor Corp.",
    "Toyota Motor Engineering & Manufacturing North America, Inc.": "Toyota Motor Corp.",
    "Toyota Motor Manufacturing, Kentucky, Inc.": "Toyota Motor Corp.",
    "Toyota Motor, Co.":                   "Toyota Motor Corp.",
    "Toyota Motors, Corp.":                "Toyota Motor Corp.",
    "Toyota Motor Manufacturing USA, Inc.": "Toyota Motor Corp.",

    "Volkswagen Group of America, Inc.":   "Volkswagen AG",
    "Volkswagen of America, Inc.":         "Volkswagen AG",
    "Volkswagen of America":               "Volkswagen AG",
    "Volkswagen Group Of America, Inc.":   "Volkswagen AG",
    "Volkswagen Of America, Inc.":         "Volkswagen AG",

    "Kia Motors America, Inc.":            "Kia Motors Corp.",
    "Kia Motors, Corp.":                   "Kia Motors Corp.",
    "Kia Motors Manufacturing Georgia, Inc.": "Kia Motors Corp.",
    "KIA Motors America, Inc.":            "Kia Motors Corp.",
    "Kia Motors Co, Ltd.":                 "Kia Motors Corp.",
    "KIA Motors, Corp.":                   "Kia Motors Corp.",
    "Kia Motors, Co.":                     "Kia Motors Corp.",

    "ASUS Computer International":         "ASUSTeK Computer Inc.",
    "Asustek Computer, Inc.":              "ASUSTeK Computer Inc.",
    "Asus Computer International":         "ASUSTeK Computer Inc.",
    "Asus Computer International, Inc.":   "ASUSTeK Computer Inc.",
    "ASUSTeK Computer, Inc.":              "ASUSTeK Computer Inc.",
    "ASUSTek Computer, Inc.":              "ASUSTeK Computer Inc.",
    "Asus Computer Intl":                  "ASUSTeK Computer Inc.",
    "ASUS Computer International, Inc.":   "ASUSTeK Computer Inc.",
    "ASUS Computer International, Corp.":  "ASUSTeK Computer Inc.",
    "Asus Computer International, Corp.":  "ASUSTeK Computer Inc.",
    "ASUSteK Computer, Inc.":              "ASUSTeK Computer Inc.",
    "ASUSTEK Computer, Inc.":              "ASUSTeK Computer Inc.",
    "Asustek Computer,Inc":                "ASUSTeK Computer Inc.",
    "ASUSTeK COMPUTER, Inc.":              "ASUSTeK Computer Inc.",

    "Lenovo (United States), Inc.":        "Lenovo Group Ltd.",
    "Lenovo Group, Ltd.":                  "Lenovo Group Ltd.",
    "Lenovo Holding Company, Inc.":        "Lenovo Group Ltd.",
    "Lenovo, Inc.":                        "Lenovo Group Ltd.",
    "Lenovo United States, Inc.":          "Lenovo Group Ltd.",
    "Lenovo (USA), Inc.":                  "Lenovo Group Ltd.",
    "Lenovo Group":                        "Lenovo Group Ltd.",
    "Lenovo International":                "Lenovo Group Ltd.",
    "Lenovo Group., Ltd.":                 "Lenovo Group Ltd.",
    "Lenovo (United States) Inc. d/b/a Lenovo International": "Lenovo Group Ltd.",
    "LENOVO (UNITED STATES) INC., a Delaware, Corp.": "Lenovo Group Ltd.",

    "Sears, Roebuck and, Co.":             "Sears Holdings Corp.",
    "Sears Roebuck &, Co.":                "Sears Holdings Corp.",
    "Sears Brands, LLC":                   "Sears Holdings Corp.",
    "Sears Holdings Management, Corp.":    "Sears Holdings Corp.",
    "Sears Holding, Corp.":                "Sears Holdings Corp.",
    "Sears Roebuck And, Co.":              "Sears Holdings Corp.",

    "Time Warner Cable, Inc.":             "Time Warner Inc.",
    "Time Warner, Inc.":                   "Time Warner Inc.",
    "Time Warner Cable, LLC":              "Time Warner Inc.",
    "Time Warner Entertainment Company L P": "Time Warner Inc.",
    "AOL Time Warner, Inc.":               "Time Warner Inc.",

    "Warner-Lambert Company, LLC":         "Pfizer Inc.",
    "Warner-Lambert, Co.":                 "Pfizer Inc.",
    "Warner Lambert Company, LLC":         "Pfizer Inc.",
    "Warner Lambert, Co.":                 "Pfizer Inc.",

    "Par Pharmaceutical, Inc.":            "Par Pharmaceutical Inc.",
    "Par Pharmaceutical Companies, Inc.":  "Par Pharmaceutical Inc.",

    "Procter & Gamble, Co.":               "Procter & Gamble Co.",

    "Mitsubishi Electric, Corp.":          "Mitsubishi Electric Corp.",

    "Colgate-Palmolive, Co.":              "Colgate-Palmolive Co.",
    "Colgate-Palmolive Co., Inc.":         "Colgate-Palmolive Co.",
    "Colgate Palmolive, Co.":              "Colgate-Palmolive Co.",
    "Colgate Palmolive Co., Inc.":         "Colgate-Palmolive Co.",
    "Colgate/Palmolive, Co.":              "Colgate-Palmolive Co.",
    "Colgate-Palmolive Company, Inc.":     "Colgate-Palmolive Co.",

    "Smithkline Beecham, Corp.":           "GlaxoSmithKline PLC",
    "SmithKline Beecham, Corp.":           "GlaxoSmithKline PLC",
    "Smithkline Beecham Plc":              "GlaxoSmithKline PLC",
    "Smithkline Beecham, P.L.C.":          "GlaxoSmithKline PLC",
    "SmithKline Beecham (Cork), Ltd.":     "GlaxoSmithKline PLC",
    "Smithkline Beecham Consumer Healthcare, L.P.": "GlaxoSmithKline PLC",
    "Glaxosmithkline":                     "GlaxoSmithKline PLC",
    "Glaxosmithkline Plc":                 "GlaxoSmithKline PLC",
    "Glaxosmithkline Consumer Healthcare, L.P.": "GlaxoSmithKline PLC",
    "Glaxosmithkline, Corp.":              "GlaxoSmithKline PLC",
    "GLAXOSMITHKLINE p.l.c.":             "GlaxoSmithKline PLC",

    "Forest Laboratories Holdings, Ltd.":  "Forest Laboratories Inc.",
    "Forest Laboratories, Inc.":           "Forest Laboratories Inc.",
    "Forest Laboratories, LLC":            "Forest Laboratories Inc.",
    "Forest Laboratories Holding, Ltd.":   "Forest Laboratories Inc.",

    "Barr Laboratories, Inc.":             "Barr Laboratories Inc.",
    "Barr Laboratories, Inc.,":            "Barr Laboratories Inc.",
    "Barr Laboratories":                   "Barr Laboratories Inc.",

    "Hospira, Inc.":                       "Hospira Inc.",
    "Hospira Australia Pty, Ltd.":         "Hospira Inc.",
    "Hospira Worldwide, Inc.":             "Hospira Inc.",

    "Cordis, Corp.":                       "Cordis Corp.",
    "Cordis, LLC":                         "Cordis Corp.",
    "Cordis Endovascular Systems, Inc.":   "Cordis Corp.",

    "Gateway, Inc.":                       "Gateway Inc.",
    "Gateway Companies, Inc.":             "Gateway Inc.",
    "Gateway Country Stores, LLC":         "Gateway Inc.",
    "Gateway 2000, Inc.":                  "Gateway Inc.",

    "Freescale Semiconductor, Inc.":       "Freescale Semiconductor Inc.",
    "Freescale Semiconductor Holdings I, Ltd.": "Freescale Semiconductor Inc.",
    "Freescale Semiconductor Japan, Ltd.": "Freescale Semiconductor Inc.",
    "Freescale Semiconductor Pte., Ltd.":  "Freescale Semiconductor Inc.",
    "Freescale Semiconductor Taiwan, Ltd.": "Freescale Semiconductor Inc.",

    "Endo Pharmaceuticals, Inc.":          "Endo Pharmaceuticals Inc.",
    "Endo Pharmaceuticals Solutions, Inc.": "Endo Pharmaceuticals Inc.",
    "Endo Pharmaceuticals Holding, Inc.":  "Endo Pharmaceuticals Inc.",
    "Endo Pharmaceuticals Holdings, Inc.": "Endo Pharmaceuticals Inc.",

    "Baxter Healthcare, Corp.":            "Baxter International Inc.",
    "Baxter International, Inc.":          "Baxter International Inc.",
    "Baxter Healthcare S.A.":              "Baxter International Inc.",
    "Baxter Diagnostics, Inc.":            "Baxter International Inc.",
    "Baxter Pharmaceutical Products, Inc.": "Baxter International Inc.",
    "Baxter Healthcare, Co.":              "Baxter International Inc.",

    "Black & Decker (U.S.), Inc.":         "Stanley Black & Decker Inc.",
    "Black & Decker, Inc.":                "Stanley Black & Decker Inc.",
    "Black & Decker, Corp.":               "Stanley Black & Decker Inc.",
    "Stanley Black & Decker, Inc.":        "Stanley Black & Decker Inc.",
    "Black & Decker (US), Inc.":           "Stanley Black & Decker Inc.",
    "The Black & Decker, Corp.":           "Stanley Black & Decker Inc.",
    "Black & Decker U.S., Inc.":           "Stanley Black & Decker Inc.",

    "Robert Bosch, LLC":                   "Robert Bosch GmbH",
    "Robert Bosch GmbH":                   "Robert Bosch GmbH",
    "Robert Bosch, Corp.":                 "Robert Bosch GmbH",
    "Robert Bosch Healthcare Systems, Inc.": "Robert Bosch GmbH",
    "Robert Bosch Tool, Corp.":            "Robert Bosch GmbH",
    "Robert Bosch North America, Corp.":   "Robert Bosch GmbH",
    "Robert Bosch Power Tool, Corp.":      "Robert Bosch GmbH",
    "Robert Bosch Gmbh":                   "Robert Bosch GmbH",
    "Bosch Security Systems, Inc.":        "Robert Bosch GmbH",

    "The Procter & Gamble, Co.":           "Procter & Gamble Co.",
    "Procter & Gamble Distributing, Co.":  "Procter & Gamble Co.",
    "Procter & Gamble Manufacturing, Co.": "Procter & Gamble Co.",
    "Procter and Gamble, Co.":             "Procter & Gamble Co.",

    "Allergan Sales, LLC":                 "Allergan Inc.",
    "Allergan USA, Inc.":                  "Allergan Inc.",
    "Allergan Sales, Inc.":                "Allergan Inc.",
    "Allergan Plc":                        "Allergan Inc.",
    "Allergan Optical, Inc.":              "Allergan Inc.",
    "Allergan Usa, Inc.":                  "Allergan Inc.",

    "Sandisk, Corp.":                      "SanDisk Corp.",
    "SanDisk, Corp.":                      "SanDisk Corp.",

    "Logitech, Inc.":                      "Logitech International S.A.",
    "Logitech International SA":           "Logitech International S.A.",
    "Logitech Europe S A":                 "Logitech International S.A.",
    "Logitech International S.A":          "Logitech International S.A.",
    "Logitech, LLC":                       "Logitech International S.A.",
    "Logitech Europe S.A.":                "Logitech International S.A.",

    "Garmin USA, Inc.":                    "Garmin Ltd.",
    "Garmin, Ltd.":                        "Garmin Ltd.",
    "Garmin, Corp.":                       "Garmin Ltd.",
    "Garmin International":                "Garmin Ltd.",
    "Garmin North America, Inc.":          "Garmin Ltd.",

    "Raytheon, Co.":                       "Raytheon Co.",
    "Raytheon Applied Signal Technology, Inc.": "Raytheon Co.",
    "Raytheon BBN Technologies, Corp.":    "Raytheon Co.",

    "Siemens, Corp.":                      "Siemens AG",
    "Siemens AG":                          "Siemens AG",
    "Siemens Medical Solutions USA, Inc.": "Siemens AG",
    "Siemens Enterprise Communications, Inc.": "Siemens AG",
    "Siemens Industry, Inc.":              "Siemens AG",
    "Siemens Medical Systems, Inc.":       "Siemens AG",
    "Siemens Energy & Automation, Inc.":   "Siemens AG",
    "Siemens Communications, Inc.":        "Siemens AG",

    "Koninklijke Philips Electronics NV":  "Koninklijke Philips N.V.",
    "U S Philips, Corp.":                  "Koninklijke Philips N.V.",
    "Koninklijke Philips Electronics N V": "Koninklijke Philips N.V.",

    "Cephalon, Inc.":                      "Teva Pharmaceutical Industries",
    "Cephalon France":                     "Teva Pharmaceutical Industries",

    "Genzyme Corp.":                       "Sanofi S.A.",
    "Genzyme Surgical Products, Corp.":    "Sanofi S.A.",

    "Ronald A Katz Technology Licensing L P":    "Ronald A. Katz Technology Licensing L.P.",
    "Ronald A Katz Technology Licensing LP":     "Ronald A. Katz Technology Licensing L.P.",
    "Ronald A Katz Technology Licensing, LP":    "Ronald A. Katz Technology Licensing L.P.",
    "Ronald A Katz Technology":                  "Ronald A. Katz Technology Licensing L.P.",
    "Ronald A. Katz Technology Licensing, L.P.": "Ronald A. Katz Technology Licensing L.P.",

    "WI-Lan, Inc.":                        "WI-LAN Inc.",
    "WI-LAN, Inc.":                        "WI-LAN Inc.",
    "Wi-Lan, Inc.":                        "WI-LAN Inc.",
    "Wi-Lan USA, Inc.":                    "WI-LAN Inc.",
    "Wi-LAN, Inc.":                        "WI-LAN Inc.",
    "Wi-LAN USA, Inc.":                    "WI-LAN Inc.",
    "Wi-Lan Technologies, Corp.":          "WI-LAN Inc.",

    "Andrx, Corp.":                        "Actavis Inc.",
    "Andrx Pharmaceuticals, Inc.":         "Actavis Inc.",
    "Andrx Pharmaceuticals, LLC":          "Actavis Inc.",
    "Andrx Labs, LLC":                     "Actavis Inc.",
    "Andrx Corporation, Inc.":             "Actavis Inc.",

    "Barr Pharmaceuticals, Inc.":          "Teva Pharmaceutical Industries",
    "Barr Pharmaceuticals, LLC":           "Teva Pharmaceutical Industries",

    "ZTE Solutions, Inc.":                 "ZTE Corp.",
    "ZTE USA, Inc.":                       "ZTE Corp.",
    "ZTE (TX), Inc.":                      "ZTE Corp.",
    "ZTE (United States), Inc.":           "ZTE Corp.",

    "Sony Ericsson Mobile Communications AB":        "Sony Corp.",
    "Sony Ericsson Mobile Communications, Inc.":     "Sony Corp.",
    "Sony Ericsson Mobile Communications, AB":       "Sony Corp.",
    "Sony Ericsson Mobile Commications (USA), Inc.": "Sony Corp.",
    "Sony Ericsson Communications, (USA), Inc.":     "Sony Corp.",
    "Sony Ericsson Mobile Communication USA, Inc.":  "Sony Corp.",

    "JVC Americas, Corp.":                 "JVC Kenwood Corp.",
    "US JVC, Corp.":                       "JVC Kenwood Corp.",
    "JVC Kenwood, Corp.":                  "JVC Kenwood Corp.",
    "Jvc Americas, Corp.":                 "JVC Kenwood Corp.",
    "U.S. JVC, Corp.":                     "JVC Kenwood Corp.",
    "JVC America, Inc.":                   "JVC Kenwood Corp.",
    "JVC Company of America":              "JVC Kenwood Corp.",

    "SAP America, Inc.":                   "SAP AG",
    "Sap Ag":                              "SAP AG",
    "Sap America, Inc.":                   "SAP AG",
    "Sap, Ag":                             "SAP AG",

    "NEC Corporation of America":          "NEC Corp.",
    "NEC Corporation of America, Inc.":    "NEC Corp.",
    "NEC Corp. of America":                "NEC Corp.",
    "NEC Corp. of America, Inc.":          "NEC Corp.",

    "Mylan Laboratories, Ltd.":            "Mylan Inc.",

    "Kimberly-Clark Worldwide, Inc.":      "Kimberly-Clark Corp.",
    "Kimberly-Clark Global Sales, LLC":    "Kimberly-Clark Corp.",
    "Kimberly-Clark Global Sales, Inc.":   "Kimberly-Clark Corp.",
    "Kimberly-clark, Corp.":               "Kimberly-Clark Corp.",

    "Dell Computer, Corp.":                "Dell Inc.",
    "Dell Computer, Inc.":                 "Dell Inc.",
    "Dell Marketing, Corp.":               "Dell Inc.",
    "Dell, Inc.,":                         "Dell Inc.",

    "DataTreasury, Corp.":                 "Datatreasury Corp.",
    "Datatreasury, Corp.":                 "Datatreasury Corp.",

    "Sun Pharmaceutical Industries, Ltd.": "Sun Pharmaceutical Industries",
    "Sun Pharmaceutical Industries, Inc.": "Sun Pharmaceutical Industries",
    "Sun Pharma Global FZE":               "Sun Pharmaceutical Industries",
    "Sun Pharmaceuticals Industries, Inc.": "Sun Pharmaceutical Industries",
    "Sun Pharmaceuticals Industries, Ltd.": "Sun Pharmaceutical Industries",

    "Janssen Pharmaceuticals, Inc.":        "Johnson & Johnson",
    "Janssen Pharmaceutica N.V.":           "Johnson & Johnson",
    "Janssen Products, L.P.":               "Johnson & Johnson",
    "Janssen Biotech, Inc.":                "Johnson & Johnson",
    "Janssen Pharmaceutica Products, L.P.": "Johnson & Johnson",
    "Janssen, L.P.":                        "Johnson & Johnson",

    "J.C. Penney Corporation, Inc.":       "J.C. Penney Co.",
    "J.C. Penney Company, Inc.":           "J.C. Penney Co.",
    "J.C. Penney, Co.":                    "J.C. Penney Co.",
    "J.C. Penney CO., Inc.":               "J.C. Penney Co.",
    "J.C. Penney Co., Inc.":               "J.C. Penney Co.",

    "Lowe's Companies, Inc.":              "Lowe's Companies Inc.",
    "Lowe's Home Centers, Inc.":           "Lowe's Companies Inc.",
    "Lowe's HIW, Inc.":                    "Lowe's Companies Inc.",
    "Lowes Home Centers, Inc.":            "Lowe's Companies Inc.",
    "Lowe's Home Centers, LLC":            "Lowe's Companies Inc.",

    "American Honda Motor Co., Inc.":       "Honda Motor Co. Ltd.",
    "American Honda Motor Co, Inc.":        "Honda Motor Co. Ltd.",
    "Honda North America, Inc.":            "Honda Motor Co. Ltd.",
    "American Honda Motor Company, Inc.":   "Honda Motor Co. Ltd.",
    "Honda Manufacturing of Alabama, LLC":  "Honda Motor Co. Ltd.",
    "Honda of America Manufacturing, Inc.": "Honda Motor Co. Ltd.",
    "Honda Motor Co., Ltd.":                "Honda Motor Co. Ltd.",

    "Reebok International, Ltd.":          "Adidas AG",
    "Reebok Interanation, Ltd.":           "Adidas AG",

    "Scimed Life Systems, Inc.":           "Boston Scientific Corp.",
    "SciMed Life Systems, Inc.":           "Boston Scientific Corp.",
    "Boston Scientific SciMed, Inc.":      "Boston Scientific Corp.",

    "Warner Chilcott Company, LLC":        "Actavis Inc.",
    "Warner Chilcott (Us), LLC":           "Actavis Inc.",
    "Warner Chilcott (US), LLC":           "Actavis Inc.",
    "Warner Chilcott Company, Inc.":       "Actavis Inc.",
    "Warner Chilcott, Inc.":               "Actavis Inc.",
    "Warner Chilcott PLC":                 "Actavis Inc.",

    "Purdue Pharmaceuticals, L.P.":         "Purdue Pharma L.P.",
    "Purdue Pharma, L.P.":                  "Purdue Pharma L.P.",
    "Purdue Pharma Products LP":            "Purdue Pharma L.P.",
    "Purdue Pharmaceutical Products, L.P.": "Purdue Pharma L.P.",
    "The Purdue Pharma, Co.":               "Purdue Pharma L.P.",
    "Purdue Pharma LP":                     "Purdue Pharma L.P.",
    "Purdue Pharmaceuticals LP":            "Purdue Pharma L.P.",

    "AOL, Inc.":                           "AOL Inc.",
    "Aol, LLC":                            "AOL Inc.",
    "AOL Advertising, Inc.":               "AOL Inc.",
    "Aol, Inc.":                           "AOL Inc.",
    "Aol":                                 "AOL Inc.",

    "Hitachi America, Ltd.":                        "Hitachi Ltd.",
    "Hitachi Global Storage Technologies, Inc.":    "Hitachi Ltd.",
    "Hitachi Data Systems, Corp.":                  "Hitachi Ltd.",
    "Hitachi Home Electronics (America), Inc.":     "Hitachi Ltd.",
    "Hitachi Semiconductor (America), Inc.":        "Hitachi Ltd.",

    "Casio America, Inc.":                 "Casio Computer Co. Ltd.",
    "Casio Computer Co., Ltd.":            "Casio Computer Co. Ltd.",
    "Casio, Inc.":                         "Casio Computer Co. Ltd.",
    "Casio Computer Co, Ltd.":             "Casio Computer Co. Ltd.",
    "Casio Phonemate, Inc.":               "Casio Computer Co. Ltd.",

    "Nikon, Inc.":                         "Nikon Corp.",
    "Nikon, Corp.":                        "Nikon Corp.",
    "Nikon Precision, Inc.":               "Nikon Corp.",
    "Nikon Americas, Inc.":                "Nikon Corp.",
    "Nikon Instruments, Inc.":             "Nikon Corp.",

    "Brother International, Corp.":        "Brother Industries Ltd.",
    "Brother Industries, Ltd.":            "Brother Industries Ltd.",

    "Pantech Wireless, Inc.":              "Pantech Co. Ltd.",
    "Pantech Co., Ltd.":                   "Pantech Co. Ltd.",
    "Pantech & Curitel Communications, Inc.": "Pantech Co. Ltd.",
    "Pantech, Corp.":                      "Pantech Co. Ltd.",
    "Pantech Co, Ltd.":                    "Pantech Co. Ltd.",
    "Pantech Company, Ltd.":               "Pantech Co. Ltd.",

    "Becton Dickinson and, Co.":           "Becton Dickinson and Co.",

    "JPMorgan Chase &, Co.":               "JPMorgan Chase & Co.",

    "Bank of America, Corp.":              "Bank of America Corp.",

    "Callaway Golf, Co.":                  "Callaway Golf Co.",

    "Avery Dennison, Corp.":               "Avery Dennison Corp.",

    "Infineon Technologies North America, Corp.": "Infineon Technologies AG",

    "Halliburton Energy Services, Inc.":   "Halliburton Co.",

    "Seagate Technology, LLC":             "Seagate Technology",

    "Western Digital, Corp.":              "Western Digital Corp.",

    "Nuance Communications, Inc.":         "Microsoft Corp.",

    "Agere Systems, Inc.":                 "Broadcom Corp.",
    "LSI, Corp.":                          "Broadcom Corp.",

    "Life Technologies, Corp.":            "Thermo Fisher Scientific Inc.",

    "Hospira Inc.":                        "Pfizer Inc.",

}

def get_parent_org(name: str) -> str:
    """Return canonical parent org name, or the original if not in map."""
    return PARENT_MAP.get(name, name)

names['standardized_org_name'] = names['standardized_org_name'].apply(get_parent_org)

In [9]:
location_terms = (
    r'Delaware|California|Florida|Washington|New York|Illinois|Minnesota|Texas|Utah|'
    r'Nevada|Colorado|Michigan|New Jersey|Georgia|Oregon|Arizona|Ohio|Indiana|Pennsylvania|'
    r'Massachusetts|Missouri|Wisconsin|Virginia|Maryland|Nebraska|Connecticut|Kansas|'
    r'Oklahoma|Tennessee|Iowa|Idaho|Arkansas|Kentucky|North Carolina|South Carolina|'
    r'Alabama|Mississippi|Louisiana|Hawaii|Alaska|Montana|Wyoming|North Dakota|South Dakota|'
    r'West Virginia|Rhode Island|Vermont|Maine|New Hampshire|New Mexico|'
    r'Canadian|Japanese|German|French|Korean|Australian|Taiwanese|Taiwan|'
    r'Calif|foreign|limited liability'
)

location_mask = names['standardized_org_name'].str.contains(
    rf'^\s*an?\s+(?:{location_terms}),?\s*(?:Corp\.|LLC|Co\.|L\.P\.|corporation|company|corp|limited|ltd\.?)?$',
    case=False, na=False, flags=re.IGNORECASE
)

names.loc[location_mask, 'standardized_org_name'] = 'Non-Specific Entity'

In [10]:
dba_mask = names['standardized_org_name'].str.contains(
    r'^\s*(?:doing business as|also known as)\s*$',
    case=False, na=False, flags=re.IGNORECASE
)

names.loc[dba_mask, 'standardized_org_name'] = 'Non-Specific Entity'

In [11]:
leftover_mask = names['standardized_org_name'].str.match(
    r'^a,?\s*(Corp\.|LLC|Co\.|L\.P\.)?$',
    case=False, na=False
)

names.loc[leftover_mask, 'standardized_org_name'] = 'Non-Specific Entity'

In [12]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 100)
names['standardized_org_name'].value_counts().head(100).reset_index()

,standardized_org_name,count
0,Non-Specific Entity,40420
1,Samsung Electronics,1952
2,Actavis Inc.,1713
3,Sony Corp.,1501
4,Teva Pharmaceutical Industries,1209
5,Mylan Inc.,1146
6,Pfizer Inc.,1119
7,LG Electronics,1100
8,Apple Inc.,1084
9,AT&T Inc.,912


In [13]:
names[names['standardized_org_name'].str.contains('GeoTag', case=False)]['standardized_org_name'].unique()

<StringArray>
['GeoTag Inc.']
Length: 1, dtype: str

In [14]:
# GICS Sector mapping for standardized_org_name values
# GICS Sectors:
#   Information Technology
#   Health Care
#   Communication Services
#   Consumer Discretionary
#   Consumer Staples
#   Financials
#   Industrials
#   Materials
#   Energy
#   Utilities
#   Real Estate
#   Unknown  (patent trolls, LLCs with no clear industry, etc.)

GICS_MAP = {

    "Samsung Electronics":              "Information Technology",
    "Apple Inc.":                       "Information Technology",
    "Microsoft Corp.":                  "Information Technology",
    "LG Electronics":                   "Information Technology",
    "HTC Corp.":                        "Information Technology",
    "Toshiba Corp.":                    "Information Technology",
    "HP Inc.":                          "Information Technology",
    "BlackBerry Ltd.":                  "Information Technology",
    "Dell Inc.":                        "Information Technology",
    "Panasonic Corp.":                  "Information Technology",
    "Melvino Technologies, Ltd.":       "Information Technology",
    "ASUSTeK Computer Inc.":            "Information Technology",
    "Fujitsu Ltd.":                     "Information Technology",
    "Acer Inc.":                        "Information Technology",
    "Lenovo Group Ltd.":                "Information Technology",
    "Cisco Systems Inc.":               "Information Technology",
    "Nintendo Co. Ltd.":                "Information Technology",
    "Intel Corp.":                      "Information Technology",
    "Sharp Corp.":                      "Information Technology",
    "IBM Corp.":                        "Information Technology",
    "Broadcom Corp.":                   "Information Technology",
    "Hitachi Ltd.":                     "Information Technology",
    "Ericsson AB":                      "Information Technology",
    "Qualcomm Inc.":                    "Information Technology",
    "Oracle Corp.":                     "Information Technology",
    "Ricoh Co. Ltd.":                   "Information Technology",
    "Texas Instruments Inc.":           "Information Technology",
    "Eastman Kodak Co.":                "Information Technology",
    "Gateway Inc.":                     "Information Technology",
    "Robert Bosch GmbH":                "Information Technology",
    "Kyocera Corp.":                    "Information Technology",
    "Adaptix, Inc.":                    "Information Technology",
    "Adobe Inc.":                       "Information Technology",
    "Symantec Corp.":                   "Information Technology",
    "Canon Inc.":                       "Information Technology",
    "Seiko Epson Corp.":                "Information Technology",
    "NEC Corp.":                        "Information Technology",
    "Siemens AG":                       "Information Technology",
    "Maxim Integrated Products, Inc.":  "Information Technology",
    "Casio Computer Co. Ltd.":          "Information Technology",
    "Palm Inc.":                        "Information Technology",
    "Freescale Semiconductor Inc.":     "Information Technology",
    "AMD Inc.":                         "Information Technology",
    "SanDisk Corp.":                    "Information Technology",
    "Infineon Technologies AG":         "Information Technology",
    "Micron Technology, Inc.":          "Information Technology",
    "JVC Kenwood Corp.":                "Information Technology",
    "Nikon Corp.":                      "Information Technology",
    "Analog Devices, Inc.":             "Information Technology",
    "Marvell Semiconductor, Inc.":      "Information Technology",
    "STMicroelectronics, Inc.":         "Information Technology",
    "D-Link Systems, Inc.":             "Information Technology",
    "EMC, Corp.":                       "Information Technology",
    "Seagate Technology":               "Information Technology",
    "Atmel, Corp.":                     "Information Technology",
    "Western Digital Corp.":            "Information Technology",
    "National Semiconductor, Corp.":    "Information Technology",
    "Netgear, Inc.":                    "Information Technology",
    "SAP AG":                           "Information Technology",
    "Xerox Corp.":                      "Information Technology",
    "NCR Corp.":                        "Information Technology",
    "Sanyo Electric Co.":               "Information Technology",
    "Pantech Co. Ltd.":                 "Information Technology",
    "Vizio, Inc.":                      "Information Technology",
    "Brother Industries Ltd.":          "Information Technology",
    "Imation, Corp.":                   "Information Technology",
    "Lexmark International, Inc.":      "Information Technology",
    "Pioneer Electronics (USA), Inc.":  "Information Technology",
    "Mitsubishi Electric Corp.":        "Information Technology",
    "Logitech International S.A.":      "Information Technology",
    "Applied Materials, Inc.":          "Information Technology",
    "Thermo Fisher Scientific Inc.":    "Information Technology",
    "Illumina, Inc.":                   "Information Technology",
    "Viewsonic, Corp.":                 "Information Technology",
    "Imation, Corp.":                   "Information Technology",
    "Symbol Technologies, Inc.":        "Information Technology",
    "Huawei Technologies":              "Information Technology",
    "ZTE Corp.":                        "Information Technology",
    "Directed Electronics, Inc.":       "Information Technology",
    "Belkin International, Inc.":       "Information Technology",
    "Novatel Wireless, Inc.":           "Information Technology",
    "Juniper Networks Inc.":            "Information Technology",
    "Netgear, Inc.":                    "Information Technology",
    "Valve, Corp.":                     "Information Technology",
    "McAfee, Inc.":                     "Information Technology",
    "Intuit, Inc.":                     "Information Technology",
    "Avaya, Inc.":                      "Information Technology",
    "Nuance Communications, Inc.":      "Information Technology",
    "Garmin Ltd.":                      "Information Technology",
    "Thomas & Betts, Corp.":            "Information Technology",
    "GeoTag Inc.":                      "Information Technology",
    "Innovative Automation, LLC":       "Information Technology",

    "AT&T Inc.":                        "Communication Services",
    "Verizon Communications":           "Communication Services",
    "Nokia Corp.":                      "Communication Services",
    "Motorola Inc.":                    "Communication Services",
    "Sprint Corp.":                     "Communication Services",
    "Google Inc.":                      "Communication Services",
    "T-Mobile USA Inc.":                "Communication Services",
    "Comcast Corp.":                    "Communication Services",
    "Yahoo Inc.":                       "Communication Services",
    "Meta Platforms Inc.":              "Communication Services",
    "Time Warner Inc.":                 "Communication Services",
    "AOL Inc.":                         "Communication Services",
    "Charter Communications, Inc.":     "Communication Services",
    "Cox Communications, Inc.":         "Communication Services",
    "Zynga, Inc.":                      "Communication Services",
    "Electronic Arts, Inc.":            "Communication Services",
    "Netflix Inc.":                     "Communication Services",
    "Amazon.com Inc.":                  "Communication Services",
    "eBay Inc.":                        "Communication Services",
    "Overstock.com, Inc.":              "Communication Services",
    "Expedia, Inc.":                    "Communication Services",
    "Groupon, Inc.":                    "Communication Services",
    "QVC, Inc.":                        "Communication Services",
    "HSN, Inc.":                        "Communication Services",

    "Actavis Inc.":                     "Health Care",
    "Pfizer Inc.":                      "Health Care",
    "Teva Pharmaceutical Industries":   "Health Care",
    "Mylan Inc.":                       "Health Care",
    "Novartis AG":                      "Health Care",
    "Sanofi S.A.":                      "Health Care",
    "AstraZeneca PLC":                  "Health Care",
    "Apotex Inc.":                      "Health Care",
    "Lupin Ltd.":                       "Health Care",
    "Abbott Laboratories":              "Health Care",
    "GlaxoSmithKline PLC":              "Health Care",
    "Dr. Reddy's Laboratories":         "Health Care",
    "Sandoz Inc.":                      "Health Care",
    "Boston Scientific Corp.":          "Health Care",
    "Merck & Co.":                      "Health Care",
    "Ranbaxy Laboratories":             "Health Care",
    "Par Pharmaceutical Inc.":          "Health Care",
    "Medtronic Inc.":                   "Health Care",
    "Johnson & Johnson":                "Health Care",
    "Sun Pharmaceutical Industries":    "Health Care",
    "Allergan Inc.":                    "Health Care",
    "Roche Holding AG":                 "Health Care",
    "Bayer AG":                         "Health Care",
    "Purdue Pharma L.P.":               "Health Care",
    "Alcon Inc.":                       "Health Care",
    "Barr Laboratories Inc.":           "Health Care",
    "Forest Laboratories Inc.":         "Health Care",
    "Genentech Inc.":                   "Health Care",
    "Eli Lilly and Co.":                "Health Care",
    "Bristol-Myers Squibb Co.":         "Health Care",
    "Impax Laboratories, Inc.":         "Health Care",
    "Hospira Inc.":                     "Health Care",
    "UCB S.A.":                         "Health Care",
    "Baxter International Inc.":        "Health Care",
    "Cordis Corp.":                     "Health Care",
    "Endo Pharmaceuticals Inc.":        "Health Care",
    "Roxane Laboratories, Inc.":        "Health Care",
    "Amneal Pharmaceuticals, LLC":      "Health Care",
    "Stryker Corp.":                    "Health Care",
    "Celgene Corp.":                    "Health Care",
    "Ecolab Inc.":                      "Health Care",
    "Perrigo, Co.":                     "Health Care",
    "Breckenridge Pharmaceutical, Inc.": "Health Care",
    "Aurobindo Pharma, Ltd.":           "Health Care",
    "Wockhardt, Ltd.":                  "Health Care",
    "Mallinckrodt, Inc.":               "Health Care",
    "Otsuka Pharmaceutical Co., Ltd.":  "Health Care",
    "Smith & Nephew, Inc.":             "Health Care",
    "Anchen Pharmaceuticals, Inc.":     "Health Care",
    "Zimmer, Inc.":                     "Health Care",
    "AbbVie, Inc.":                     "Health Care",
    "Gilead Sciences, Inc.":            "Health Care",
    "Amgen, Inc.":                      "Health Care",
    "Becton Dickinson and Co.":         "Health Care",
    "Biomet, Inc.":                     "Health Care",
    "C.R. Bard, Inc.":                  "Health Care",
    "Supernus Pharmaceuticals, Inc.":   "Health Care",
    "Medicis Pharmaceutical, Corp.":    "Health Care",
    "Hetero Labs, Ltd.":                "Health Care",
    "Cadila Healthcare, Ltd.":          "Health Care",
    "Accord Healthcare, Inc.":          "Health Care",
    "Jazz Pharmaceuticals, Inc.":       "Health Care",
    "Advanced Cardiovascular Systems, Inc.": "Health Care",
    "Honeywell International Inc.":     "Health Care",
    "Bausch & Lomb, Inc.":              "Health Care",
    "Lemelson Medical, Education & Research Foundation LP": "Health Care",
    "The P.F. Laboratories, Inc.":      "Health Care",
    "Shire, LLC":                       "Health Care",
    "Koninklijke Philips N.V.":         "Health Care",

    "Sony Corp.":                       "Consumer Discretionary",
    "Walmart Inc.":                     "Consumer Discretionary",
    "Target Corp.":                     "Consumer Discretionary",
    "Best Buy Co.":                     "Consumer Discretionary",
    "Sears Holdings Corp.":             "Consumer Discretionary",
    "Toyota Motor Corp.":               "Consumer Discretionary",
    "Hyundai Motor Co.":                "Consumer Discretionary",
    "Nissan Motor Co.":                 "Consumer Discretionary",
    "Ford Motor Co.":                   "Consumer Discretionary",
    "General Motors Co.":               "Consumer Discretionary",
    "Honda Motor Co. Ltd.":             "Consumer Discretionary",
    "Mercedes-Benz Group":              "Consumer Discretionary",
    "BMW AG":                           "Consumer Discretionary",
    "Volkswagen AG":                    "Consumer Discretionary",
    "Kia Motors Corp.":                 "Consumer Discretionary",
    "Nike Inc.":                        "Consumer Discretionary",
    "Adidas AG":                        "Consumer Discretionary",
    "Oakley, Inc.":                     "Consumer Discretionary",
    "Mattel, Inc.":                     "Consumer Discretionary",
    "Lowe's Companies Inc.":            "Consumer Discretionary",
    "J.C. Penney Co.":                  "Consumer Discretionary",
    "Macy's, Inc.":                     "Consumer Discretionary",
    "Nordstrom, Inc.":                  "Consumer Discretionary",
    "Kohl's Department Stores, Inc.":   "Consumer Discretionary",
    "Toys \"R\" Us, Inc.":              "Consumer Discretionary",
    "Dick's Sporting Goods, Inc.":      "Consumer Discretionary",
    "Cabela's, Inc.":                   "Consumer Discretionary",
    "Barnes & Noble Inc.":              "Consumer Discretionary",
    "Bed Bath & Beyond, Inc.":          "Consumer Discretionary",
    "Newegg, Inc.":                     "Consumer Discretionary",
    "Sharper Image, Corp.":             "Consumer Discretionary",
    "Callaway Golf Co.":                "Consumer Discretionary",
    "Precor, Inc.":                     "Consumer Discretionary",
    "Conair, Corp.":                    "Consumer Discretionary",
    "Sunbeam Products, Inc.":           "Consumer Discretionary",
    "Whirlpool Corp.":                  "Consumer Discretionary",
    "Kmart, Corp.":                     "Consumer Discretionary",
    "Williams-Sonoma, Inc.":            "Consumer Discretionary",
    "Mag Instrument, Inc.":             "Consumer Discretionary",
    "Vizio, Inc.":                      "Consumer Discretionary",
    "Reebok International, Ltd.":       "Consumer Discretionary",

    "Procter & Gamble Co.":             "Consumer Staples",
    "Kimberly-Clark Corp.":             "Consumer Staples",
    "Monsanto Co.":                     "Consumer Staples",
    "Walgreens Co.":                    "Consumer Staples",
    "Costco Wholesale Corp.":           "Consumer Staples",
    "Colgate-Palmolive Co.":            "Consumer Staples",
    "3M Co.":                           "Consumer Staples",
    "Rite Aid Corp.":                   "Consumer Staples",
    "CVS Pharmacy, Inc.":               "Consumer Staples",
    "Staples Inc.":                     "Consumer Staples",
    "Office Depot Inc.":                "Consumer Staples",

    "General Electric Co.":             "Industrials",
    "Honeywell International Inc.":     "Industrials",
    "Illinois Tool Works Inc.":         "Industrials",
    "Emerson Electric Co.":             "Industrials",
    "Stanley Black & Decker Inc.":      "Industrials",
    "Garmin Ltd.":                      "Industrials",
    "Leggett & Platt, Inc.":            "Industrials",
    "Halliburton Co.":                  "Industrials",
    "Baker Hughes, Inc.":               "Industrials",
    "Pall, Corp.":                      "Industrials",
    "Leviton Manufacturing Co., Inc.":  "Industrials",
    "American Airlines, Inc.":          "Industrials",
    "Weatherford International, Inc.":  "Industrials",

    "American Express Co.":             "Financials",
    "JPMorgan Chase & Co.":             "Financials",
    "Bank of America Corp.":            "Financials",
    "Datatreasury Corp.":               "Financials",

    "Eastman Kodak Co.":                "Materials",

    "Non-Specific Entity":              "Unknown",
    "TQP Development, LLC":             "Unknown",
    "ArrivalStar S.A.":                 "Unknown",
    "eDekka, LLC":                      "Unknown",
    "Blue Spike, LLC":                  "Unknown",
    "Ronald A. Katz Technology Licensing L.P.": "Unknown",
    "Parallel Networks, LLC":           "Unknown",
    "Uniloc USA, Inc.":                 "Unknown",
    "Uniloc Luxembourg S.A.":           "Unknown",
    "Eclipse IP, LLC":                  "Unknown",
    "Data Carriers, LLC":               "Unknown",
    "Unified Messaging Solutions, LLC": "Unknown",
    "Landmark Technology, LLC":         "Unknown",
    "Phoenix Licensing, LLC":           "Unknown",
    "LPL Licensing, LLC":               "Unknown",
    "Lodsys Group, LLC":                "Unknown",
    "Hawk Technology Systems, LLC":     "Unknown",
    "Thermolife International, LLC":    "Unknown",
    "Brandywine Communications Technologies, LLC": "Unknown",
    "Rates Technology, Inc.":           "Unknown",
    "WI-LAN Inc.":                      "Unknown",
    "Orion IP, LLC":                    "Unknown",
    "NovelPoint Tracking, LLC":         "Unknown",
    "Patent Group, LLC":                "Unknown",
    "Clear With Computers, LLC":        "Unknown",
    "Walker Digital, LLC":              "Unknown",
    "Acacia Media Technologies, Corp.": "Unknown",
    "CTP Innovations, LLC":             "Unknown",
    "EMG Technology, LLC":              "Unknown",
    "Wyncomm, LLC":                     "Unknown",
    "Webvention, LLC":                  "Unknown",
    "Automated Transactions, LLC":      "Unknown",
    "BSG Tech, LLC":                    "Unknown",
    "Affinity Labs of Texas, LLC":      "Unknown",
    "Beacon Navigation GmbH":           "Unknown",
    "Traffic Information, LLC":         "Unknown",
    "Network Signatures, Inc.":         "Unknown",
    "Penovia, LLC":                     "Unknown",
    "Genaville, LLC":                   "Unknown",
    "Technology Licensing, Corp.":      "Unknown",
    "Financial Systems Innovation, LLC": "Unknown",
    "Golden Bridge Technology, Inc.":   "Unknown",
    "CryptoPeak Solutions, LLC":        "Unknown",
    "LBS Innovations, LLC":             "Unknown",
    "Flashpoint Technology, Inc.":      "Unknown",
    "Chrimar Systems, Inc.":            "Unknown",
    "Chrimar Holding Company, LLC":     "Unknown",
    "Vantage Point Technology, Inc.":   "Unknown",
    "American Vehicular Sciences, LLC": "Unknown",
    "Intellectual Ventures II, LLC":    "Unknown",
    "Intellectual Ventures I, LLC":     "Unknown",
    "Wetro Lan, LLC":                   "Unknown",
    "Rothschild Connected Devices Innovations,LLC": "Unknown",
    "Pi-Net International, Inc.":       "Unknown",
    "Norman IP Holdings, LLC":          "Unknown",
    "Symbology Innovations, LLC":       "Unknown",
    "Protegrity, Corp.":                "Unknown",
    "Marshall Feature Recognition, LLC": "Unknown",
    "Soverain Software, LLC":           "Unknown",
    "Cygnus Telecommunications Technology, LLC": "Unknown",
    "UbiComm, LLC":                     "Unknown",
    "Innovative Wireless Solutions, LLC": "Unknown",
    "Location Services IP, LLC":        "Unknown",
    "Dynamic Hosting Company, LLC":     "Unknown",
    "Select Retrieval, LLC":            "Unknown",
    "Shipping and Transit, LLC":        "Unknown",
    "NovelPoint Security, LLC":         "Unknown",
    "Mosaid Technologies, Inc.":        "Unknown",
    "FastVDO, LLC":                     "Unknown",
    "Technology Properties Limited, LLC": "Unknown",
    "Sockeye Licensing TX, LLC":        "Unknown",
    "Innovative Display Technologies, LLC": "Unknown",
    "Rothschild Location Technologies, LLC": "Unknown",
    "WIAV Networks, LLC":               "Unknown",
    "Internet Machines, LLC":           "Unknown",
    "SmartPhone Technologies, LLC":     "Unknown",
    "SmartFit Solutions, LLC":          "Unknown",
    "IP Innovation, LLC":               "Unknown",
    "Multiplayer Network Innovations, LLC": "Unknown",
    "Cellular Communications Equipment, LLC": "Unknown",
    "InMotion Imagery Technologies, LLC": "Unknown",
    "CYVA Research Holdings, LLC":      "Unknown",
    "Innovatio IP Ventures, LLC":       "Unknown",
    "DietGoal Innovations, LLC":        "Unknown",
    "Lodsys Group, LLC":                "Unknown",
    "EON Corp. IP Holdings, LLC":       "Unknown",
    "Pragmatus Telecom, LLC":           "Unknown",
    "Azure Networks, LLC":              "Unknown",
    "Saxon Innovations, LLC":           "Unknown",
    "Rembrandt Technologies LP":        "Unknown",
    "Trading Technologies International, Inc.": "Unknown",
    "Klausner Technologies, Inc.":      "Unknown",
    "Bandspeed, Inc.":                  "Unknown",
}

def classify_by_keyword(name: str) -> str:
    if not isinstance(name, str):
        return "Unknown"
    n = name.lower()
    if re.search(r'pharma|pharmaceutical|therapeutics|biotech|bioscience|laborator|medical|health|clinic|drug|oncolog|surgical|diagnostics', n):
        return "Health Care"
    if re.search(r'bank|financ|insurance|capital|investment|securit|mortgage|credit|lending', n):
        return "Financials"
    if re.search(r'motor|automotive|vehicle|automob', n):
        return "Consumer Discretionary"
    if re.search(r'telecom|wireless|cellular|mobile|broadband|cable|satellite', n):
        return "Communication Services"
    if re.search(r'semiconductor|software|computer|tech|digital|electronic|network|cyber|data|systems|logic|micro|nano|cloud|ai\b|machine learning', n):
        return "Information Technology"
    if re.search(r'energy|oil|gas|petroleum|mining|chemical', n):
        return "Energy"
    if re.search(r'retail|store|shop|market|supermarket|grocer', n):
        return "Consumer Staples"
    if re.search(r'aerospace|defense|aviation|construction|engineering|manufacturing|industrial', n):
        return "Industrials"
    return "Unknown"

def get_gics_sector(name: str) -> str:
    """Return GICS sector for a standardized org name."""
    if name in GICS_MAP:
        return GICS_MAP[name]
    return classify_by_keyword(name)

names['gics_sector'] = names['standardized_org_name'].apply(get_gics_sector)

In [15]:
llc_desc_mask = names['standardized_org_name'].str.contains(
    r'\blimited liability\b',
    case=False, na=False
)

names.loc[llc_desc_mask, 'standardized_org_name'] = 'Non-Specific Entity'

In [16]:
inc_mask = names['standardized_org_name'].str.match(r'^inc\.?$', case=False, na=False)
names.loc[inc_mask, 'standardized_org_name'] = 'Non-Specific Entity'

In [17]:
tech_mask = (names['gics_sector'] == 'Unknown') & (
    names['standardized_org_name'].str.contains(
        r'\b(?:tech|technology|technologies|computer|computers|software|digital|systems|networks|semiconductor|electronics|microsystems|microelectronics)\b',
        case=False, na=False
    )
)

names.loc[tech_mask, 'gics_sector'] = 'Information Technology'

In [18]:
unknown_mask = names['gics_sector'] == 'Unknown'
print(names[unknown_mask].groupby('party_type')['standardized_org_name'].count())

party_type
Amicus                                82
Appellant                              6
Appellee                              15
Claimant                              13
Consolidated Counter Claimant        662
Consolidated Counter Defendant       473
Consolidated Defendant              3807
Consolidated Intervenor                9
Consolidated Plaintiff              1220
Counter Claimant                   46549
Counter Defendant                  43283
Counter Plaintiff                    593
Cross Claimant                       481
Cross Defendant                      600
Defendant                         103667
Garnishee                             29
In Re                                 21
Interested Party                     225
Intervenor                           551
Mediator                              78
Miscellaneous                        225
Movant                               736
Nominal Defendant                      6
Notice Only                           91
Objec

In [19]:
unknown_defendant_mask = (names['gics_sector'] == 'Unknown') & (names['party_type'] == 'Defendant')
print(names[unknown_defendant_mask]['standardized_org_name'].value_counts())

standardized_org_name
Non-Specific Entity                                                                                                                  18550
Erroneously Sued As                                                                                                                     46
Commissioner of Patents and Trademarks                                                                                                  45
Deputy Under Secretary of Commerce for Intellectual Property and Deputy Director of the United States Patent and Trademark Office       39
Under Secretary of Commerce for Intellectual Property and Director of the United States Patent and Trademark Office                     37
                                                                                                                                     ...  
Trans-Expedite, Inc.                                                                                                                     1
OSMI,

In [20]:
terminated_mask = (
    (names['entity_type'] == 'Unknown') &
    names['name'].str.contains('terminated', case=False, na=False)
)

names.loc[terminated_mask, 'entity_type'] = 'Role in Case Terminated'

names.loc[terminated_mask, 'name'].value_counts().head(50)

Series([], Name: count, dtype: int64)

In [21]:
names['entity_type'].value_counts()

entity_type
Organization               423919
Independent                 75643
Role in Case Terminated     50728
Consolidated                 5484
Unknown                      3958
Name: count, dtype: int64

In [22]:
consolidated_mask = (
    (names['entity_type'] == 'Unknown') &
    names['name'].str.contains(r'consolidated', case=False, na=False)
)

names.loc[consolidated_mask, 'entity_type'] = 'Consolidated'

In [23]:
names['entity_type'].value_counts()

entity_type
Organization               423919
Independent                 75643
Role in Case Terminated     50728
Consolidated                 5484
Unknown                      3958
Name: count, dtype: int64

In [24]:
unknown_names_top50 = names.loc[
    names['entity_type'].str.casefold() == 'unknown',
    'name'
].value_counts().head(50)

unknown_names_top50

name
also known as                                                1520
formerly known as                                             973
agent of                                                      133
ADR Provider                                                   82
Facilitative Mediator                                          71
Special Master                                                 61
trading as                                                     51
Mediator                                                       42
ENE Evaluator                                                  30
a sole proprietorship                                          30
Attorney Settlement Officer                                    29
now known as                                                   25
Technical Advisor                                              24
All Defendants                                                 20
Relator                                                        17
Liais

In [25]:
names.loc[
    (names['entity_type'] == 'Unknown') &
    (
        names['name'].str.contains(
            r'\bresident\b|\bindividually\b|\bperson\b',
            case=False,
            na=False
        )
    ),
    'entity_type'
] = 'Independent'

In [26]:
unknown_names_top50 = names.loc[
    names['entity_type'].str.casefold() == 'unknown',
    'name'
].value_counts().head(50)

unknown_names_top50

name
also known as                                                1520
formerly known as                                             973
agent of                                                      133
ADR Provider                                                   82
Facilitative Mediator                                          71
Special Master                                                 61
trading as                                                     51
Mediator                                                       42
ENE Evaluator                                                  30
a sole proprietorship                                          30
Attorney Settlement Officer                                    29
now known as                                                   25
Technical Advisor                                              24
All Defendants                                                 20
Relator                                                        17
Liais

In [27]:
# Disney (case-insensitive)
names.loc[
    names['standardized_org_name'].str.contains(r'disney', case=False, na=False),
    'standardized_org_name'
] = 'The Walt Disney Company'

names.loc[
    names['standardized_org_name'].str.contains(r"reddy's", case=False, na=False),
    'standardized_org_name'
] = "Dr. Reddy's Laboratories"

# DirecTV (case-insensitive)
names.loc[
    names['standardized_org_name'].str.contains(r'directv', case=False, na=False),
    'standardized_org_name'
] = 'DirecTV LLC.'

names.loc[
    names['standardized_org_name'].str.contains(r'adidas', case=False, na=False),
    'standardized_org_name'
] = 'Adidas'

names.loc[
    names['standardized_org_name'].str.contains(r'daewoo', case=False, na=False),
    'standardized_org_name'
] = 'Daewoo'

names.loc[
    names['standardized_org_name'].str.contains(r'ferrari', case=False, na=False),
    'standardized_org_name'
] = 'Ferrari'

names.loc[
    names['standardized_org_name'].str.contains(r'fleetmatics', case=False, na=False),
    'standardized_org_name'
] = 'Fleetmatics'

names.loc[
    names['standardized_org_name'].str.contains(r'echostar', case=False, na=False),
    'standardized_org_name'
] = 'EchoStar'

# AT&T (EXACT capitalization sequence)
names.loc[
    names['standardized_org_name'].str.contains(r'AT&T', case=True, na=False),
    'standardized_org_name'
] = 'AT&T Inc.'

In [28]:
canonical_map = {
    '3Com, Corp.': '3Com',
    '3Com, Inc.': '3Com',

    '3D Systems, Inc.': '3D Systems',
    '3D Systems, Corp.': '3D Systems',

    '3Dlabs, Inc.': '3Dlabs',
    '3Dlabs, Ltd.': '3Dlabs',

    '3Form, Inc.': '3Form',
    '3Form, LLC': '3Form',

    '3M Co.': '3M',
    '3M, Corp.': '3M',

    '4 Point Lift Systems, Inc.': '4 Point Lift Systems',
    'ABC, Corp.': 'American Broadcast Company Inc.',
    'ABC, Co.': 'American Broadcast Company Inc.',
    'ABC, Inc.': 'American Broadcast Company Inc.',
    'Abc': 'American Broadcast Company Inc.',
    'Abc, Corp.': 'American Broadcast Company Inc.',
    'Abc, Co.': 'American Broadcast Company Inc.',
    'Abc, LLC': 'American Broadcast Company Inc.',
    'A & E Products Group, L.P.': 'A & E Products Group',

    'A M Manufacturing, Co.': 'A M Manufacturing',
    'A M Manufacturing Company, Inc.': 'A M Manufacturing',

    'A&A Manufacturing Company, Inc.': 'A&A Manufacturing',
    'A&A Manufacturing, Co.': 'A&A Manufacturing',

    'AGI Media, Inc.': 'AGI Media',

    'AIM Industries, Inc.': 'AIM Industries',
    'AIM Industries, LLC': 'AIM Industries',

    'A&E Products Group, L.P.': 'A&E Products Group',
    'A&E Products Group, Inc.': 'A&E Products Group',

    'A&J Manufacturing, LLC': 'A&J Manufacturing',
    'A&J Manufacturing, Inc.': 'A&J Manufacturing',
    '3-D Matrix, Inc.': '3-D Matrix',
    '3-D Matrix, Ltd.': '3-D Matrix',

    '3-D Trading, Co.': '3-D Trading',
    '3-D Trading Co., Inc.': '3-D Trading',

    '3CX, Ltd.': '3CX',
    '3CX, Inc.': '3CX',

    'A.C.E. International Company, Inc.': 'A.C.E. International',

    'A.R. Fleming Printing, Co.': 'A.R. Fleming Printing',
    'A.R. Fleming Printing Company, Inc.': 'A.R. Fleming Printing',

    'AAMP of America, Inc.': 'AAMP of America',

    'A.G. Findings & Mfg. Co., Inc.': 'A.G. Findings & Mfg.',
    'A.G. Findings & Mfg., Co.': 'A.G. Findings & Mfg.',

    'A.J. Manufacturing, Inc.': 'A.J. Manufacturing',
    'A.J. Manufacturing Company, Inc.': 'A.J. Manufacturing',

    'ACCO Brands, Inc.': 'ACCO Brands',
    'ACCO Brands, Corp.': 'ACCO Brands',

    'ACTi, Corp.': 'ACTi',
    'ACTi Corporation, Inc.': 'ACTi',

    'AOC International, Corp.': 'AOC International',

    'AOptix Technologies, Inc.': 'AOptix Technologies',

    'APP Pharmaceuticals, LLC': 'APP Pharmaceuticals',
    'APP Pharmaceuticals, Inc.': 'APP Pharmaceuticals',

    'ARM, Ltd.': 'ARM',
    'ARM, Inc.': 'ARM',

    'ATC Leasing, Co.': 'ATC Leasing',
    'ATC Leasing Company, LLC': 'ATC Leasing',

    'ATLeisure, Inc.': 'ATLeisure',
    'ATLeisure, LLC': 'ATLeisure',

    'ATM Systems, Corp.': 'ATM Systems',

    'AWI Acquisition, Inc.': 'AWI Acquisition',
    'AWI Acquisition, Co.': 'AWI Acquisition',

    'Abacon Telecommunications, Corp.': 'Abacon Telecommunications',
    'Abacon Telecommunications, LLC': 'Abacon Telecommunications',

    'Abbyy USA Software House, Inc.': 'Abbyy USA Software House',

    'Absolute Software, Inc.': 'Absolute Software',
    'Absolute Software, Corp.': 'Absolute Software',

    'Ac Global Systems, Inc.': 'Ac Global Systems',
    'Ac Global Systems, Ltd.': 'Ac Global Systems',

    'Accord Healthcare, Inc.': 'Accord Healthcare',
    'Accord Healthcare, Ltd.': 'Accord Healthcare',

    'Acella Pharmaceuticals, LLC': 'Acella Pharmaceuticals',
    'Acella Pharmaceuticals, Inc.': 'Acella Pharmaceuticals',

    'Act Marketing, Ltd.': 'Act Marketing',
    'Act Marketing, Inc.': 'Act Marketing',

    'Actavis Pharma Manufacturing Pvt. Ltd., LLC': 'Actavis Pharma Manufacturing Pvt.',
    'Actavis Pharma Manufacturing Pvt., Ltd.': 'Actavis Pharma Manufacturing Pvt.',

    'Actron Manufacturing Company, Inc.': 'Actron Manufacturing',
    'Actron Manufacturing, Inc.': 'Actron Manufacturing',

    'Addonics Technologies, Inc.': 'Addonics Technologies',

    'Agila Specialties Private, Ltd.': 'Agila Specialties Private',
    'Agila Specialties Private Limited, Inc.': 'Agila Specialties Private',

    'Agtek Development, Co.': 'Agtek Development',
    'Agtek Development Co., Inc.': 'Agtek Development',

    'Air Bak Technologies, LLC': 'Air Bak Technologies',
    'Air Bak Technologies, Corp.': 'Air Bak Technologies',

    'Air Cool Industrial Company, Ltd.': 'Air Cool Industrial',
    'Air Cool Industrial Co., Ltd.': 'Air Cool Industrial',
    'Air Cool Industrial Co, Ltd.': 'Air Cool Industrial',

    'Air Liquide America, Corp.': 'Air Liquide America',
    'Air Liquide America, L.P.': 'Air Liquide America',

    'Air2Water, LLC': 'Air2Water',
    'Air2Water, Inc.': 'Air2Water',

    'AirIQ, Inc.': 'AirIQ',
    'AirIQ, LLC': 'AirIQ',

    'Airlite Plastics, Co.': 'Airlite Plastics',
    'Airlite Plastics, Inc.': 'Airlite Plastics',

    'Airshield, Inc.': 'Airshield',
    'Airshield, Corp.': 'Airshield',

    'Airway Industries, Inc.': 'Airway Industries',

    'Ajinomoto Co., Inc.': 'Ajinomoto',
    'Ajinomoto Company, Inc.': 'Ajinomoto',

    'Akai Electric Company, Ltd.': 'Akai Electric',
    'Akai Electric Co., Ltd.': 'Akai Electric',
    'Brinkmann Instruments Company, Inc.': 'Brinkmann Instruments',
    'Brinkmann Instruments, Inc.': 'Brinkmann Instruments',

    'Bristol-Myers Squibb Co.': 'Bristol-Myers Squibb',
    'Bristol-Myers Squibb, Inc.': 'Bristol-Myers Squibb',
    'Bristol-Myers Squibb Company, Inc.': 'Bristol-Myers Squibb',

    'British Airways PLC, Corp.': 'British Airways PLC',

    'Brivo Systems, LLC': 'Brivo Systems',
    'Brivo Systems, Inc.': 'Brivo Systems',

    'Broadvox, Inc.': 'Broadvox',
    'Broadvox, LLC': 'Broadvox',

    'Brookshire Brothers, Ltd.': 'Brookshire Brothers',
    'Brookshire Brothers, Inc.': 'Brookshire Brothers',

    'Brookstone, Inc.': 'Brookstone',
    'Brookstone, Co.': 'Brookstone',
    'Brookstone Company, Inc.': 'Brookstone',
    'Brookstone Co, Inc.': 'Brookstone',

    'ESC Medical Systems, Ltd.': 'ESC Medical Systems',
    'ESC Medical Systems, Inc.': 'ESC Medical Systems',

    'ESI Enterprises, LLC': 'ESI Enterprises',
    'ESI Enterprises, Inc.': 'ESI Enterprises',

    'ETView, Inc.': 'ETView',
    'ETView, Ltd.': 'ETView',

    'Earth Eco Research, LLC': 'Earth Eco Research',

    'E.W. Scripps Company, Inc.': 'E.W. Scripps',
    'E.W. Scripps, Co.': 'E.W. Scripps',

    'EBI Medical Systems, Inc.': 'EBI Medical Systems',
    'EBI Medical Systems, LLC': 'EBI Medical Systems',

    'ECI Telecom, Ltd.': 'ECI Telecom',
    'ECI Telecom, Inc.': 'ECI Telecom',

    'EDCO Incorporated of Florida, Inc.': 'EDCO Incorporated of Florida',

    'EGS Electrical Group, LLC': 'EGS Electrical Group',

    'EGS International, Inc.': 'EGS International',

    'EIZO, Corp.': 'EIZO',
    'EIZO, Inc.': 'EIZO',

    'ELSCINT, Ltd.': 'ELSCINT',
    'ELSCINT, Inc.': 'ELSCINT',

    'EMD Millipore, Corp.': 'EMD Millipore',
    'EMD Millipore, Inc.': 'EMD Millipore',

    'EMI Tubular Products, Inc.': 'EMI Tubular Products',

    'EMS Technologies, Inc.': 'EMS Technologies',
    'EMS Technologies, LLC': 'EMS Technologies',

    'EON Labs Manufacturing, Inc.': 'EON Labs Manufacturing',
    'EON Labs Manufacturing Corporation, Inc.': 'EON Labs Manufacturing',

    'EPO Science and Technology, Inc.': 'EPO Science and Technology',
    'EPO Science and Technology Co., Inc.': 'EPO Science and Technology',

    'Earthlink, Inc.': 'Earthlink',
    'Earthlink, LLC': 'Earthlink',

    'Eastern Molding International, LLC': 'Eastern Molding International',
    'Eastern Molding International, Inc.': 'Eastern Molding International',

    'Eastman Chemical, Co.': 'Eastman Chemical',
    'Eastman Chemical Company, Inc.': 'Eastman Chemical',

    'Eastman Holding, Co.': 'Eastman Holding',

    'Easton Technical Products, Inc.': 'Easton Technical Products',

    'E-Systems, Inc.': 'E-Systems',
    'E-Systems, LLC': 'E-Systems',

    'Downhole Injection Systems, LLC': 'Downhole Injection Systems',

    'Dr. Fresh, Inc.': 'Dr. Fresh',
    'Dr. Fresh, LLC': 'Dr. Fresh',

    'Dri Mark Products, Inc.': 'Dri Mark Products',

    'Drift Innovation, Inc.': 'Drift Innovation',
    'Drift Innovation, Ltd.': 'Drift Innovation',

    'Dsm Nutritional Products, Ltd.': 'Dsm Nutritional Products',
    'Dsm Nutritional Products, Inc.': 'Dsm Nutritional Products',

    'Dualstar Entertainment Group, Inc.': 'Dualstar Entertainment Group',
    'Dualstar Entertainment Group, LLC': 'Dualstar Entertainment Group',

    "Dunkin' Donuts, Inc.": "Dunkin' Donuts",
    "Dunkin' Donuts, LLC": "Dunkin' Donuts",

    'Dura Automotive Systems, Inc.': 'Dura Automotive Systems',
    'Dura Automotive Systems, LLC': 'Dura Automotive Systems',

    'Dura Global Technologies, Inc.': 'Dura Global Technologies',
    'Dura Global Technologies, LLC': 'Dura Global Technologies',

    'Dura Lube, Inc.': 'Dura Lube',
    'Dura Lube, Corp.': 'Dura Lube',

    'Dura Operating, Corp.': 'Dura Operating',
    'Dura Operating, LLC': 'Dura Operating',

    'Durabag Co, Inc.': 'Durabag',
    'Durabag Company, Inc.': 'Durabag',

    'Duro-Med Industries, Inc.': 'Duro-Med Industries',

    'Dwyer Instruments, Inc.': 'Dwyer Instruments',
    'Dwyer Instruments, Ltd.': 'Dwyer Instruments',
    'Dwyer Instruments Company, Ltd.': 'Dwyer Instruments',

    'Dymatize Enterprises, Inc.': 'Dymatize Enterprises',
    'Dymatize Enterprises, LLC': 'Dymatize Enterprises',

    'Dynacraft Golf Products, Inc.': 'Dynacraft Golf Products',

    'Dynaflex International, Inc.': 'Dynaflex International',

    'Dynamic Classics, Ltd.': 'Dynamic Classics',
    'Dynamic Classics LTD, Inc.': 'Dynamic Classics',

    'Dynamic Imaging, LLC': 'Dynamic Imaging',
    'Dynamic Imaging, Inc.': 'Dynamic Imaging',

    'Dynamic Tire, Corp.': 'Dynamic Tire',
    'Dynamic Tire, Inc.': 'Dynamic Tire',

    'Dynascan Technology, Inc.': 'Dynascan Technology',
    'Dynascan Technology, Corp.': 'Dynascan Technology',

    'Dyson, Inc.': 'Dyson',
    'Dyson, Ltd.': 'Dyson',
    'Electronic Identification Devices, Inc.': 'Electronic Identification Devices',
    'Electronic Identification Devices, Ltd.': 'Electronic Identification Devices',

    'Electronics for Imaging, Inc.': 'Electronics for Imaging',

    'Electrovaya, Inc.': 'Electrovaya',
    'Electrovaya Company, Inc.': 'Electrovaya',

    'Electrovert, Ltd.': 'Electrovert',
    'Electrovert, Inc.': 'Electrovert',

    'Edelbrock, Corp.': 'Edelbrock',
    'Edelbrock, LLC': 'Edelbrock',

    'Edge Systems, Corp.': 'Edge Systems',
    'Edge Systems, LLC': 'Edge Systems',

    'Edge Tech, Corp.': 'Edge Tech',
    'Edge Tech, Inc.': 'Edge Tech',

    'Edible Arrangements International, LLC': 'Edible Arrangements International',

    'Edimax Technology Co., Ltd.': 'Edimax Technology',
    'Edimax Technology, Co.': 'Edimax Technology',

    'Edisync Systems, LLC': 'Edisync Systems',
    'Edisync Systems, Inc.': 'Edisync Systems',

    'Edwards Lifesciences, Corp.': 'Edwards Lifesciences',
    'Edwards Lifesciences, LLC': 'Edwards Lifesciences',

    'Edwards Ski Products, Inc.': 'Edwards Ski Products',

    'Eftchildsupport.com, Inc.': 'Eftchildsupport.com',
    'Eftchildsupport.com, LLC': 'Eftchildsupport.com',

    'Eicon Networks, Corp.': 'Eicon Networks',
    'Eicon Networks, Inc.': 'Eicon Networks',

    'Eisai Co, Ltd.': 'Eisai',
    'Eisai, Inc.': 'Eisai',
    'Eisai Co., Ltd.': 'Eisai',

    'Eizo, Corp.': 'Eizo',
    'Eizo, Inc.': 'Eizo',

    'Ejay International, Inc.': 'Ejay International',

    'Ekr Therapeutics, LLC': 'Ekr Therapeutics',
    'Ekr Therapeutics, Inc.': 'Ekr Therapeutics',

    'Election Systems & Software, Inc.': 'Election Systems & Software',
    'Election Systems & Software, LLC': 'Election Systems & Software',

    'Electric Motion, Corp.': 'Electric Motion',
    'Electric Motion Co, Inc.': 'Electric Motion',

    'Electro-Biology, Inc.': 'Electro-Biology',
    'Electro-Biology, LLC': 'Electro-Biology',

    'Electrocatalytic, Inc.': 'Electrocatalytic',
    'Electrocatalytic, Ltd.': 'Electrocatalytic',

    'Eckerd, Inc.': 'Eckerd',
    'Eckerd, Corp.': 'Eckerd',

    'Ecommerce, LLC': 'Ecommerce',
    'Ecommerce, Inc.': 'Ecommerce',

    'Ecopave California, Corp.': 'Ecopave California',
    'Ecopave California, L.P.': 'Ecopave California',

    'Elekta, Inc.': 'Elekta',
    'Elekta, Ltd.': 'Elekta',

    'Elektrobit, Inc.': 'Elektrobit',
    'Elektrobit, Corp.': 'Elektrobit',

    'Elite Lighting, Corp.': 'Elite Lighting',

    'E.R. Squibb & Sons, Inc.': 'E.R. Squibb & Sons',
    'E.R. Squibb & Sons, LLC': 'E.R. Squibb & Sons',

    'Easton-Bell Sports, Inc.': 'Easton-Bell Sports',
    'Easton-Bell Sports, LLC': 'Easton-Bell Sports',

    'Eastpak, Inc.': 'Eastpak',
    'Eastpak, Corp.': 'Eastpak',

    'Easy Day Manufacturing, Co.': 'Easy Day Manufacturing',
    'Easy Day Manufacturing Co., Inc.': 'Easy Day Manufacturing',

    'Easy Gardner Products, Ltd.': 'Easy Gardner Products',
    'Easy Gardner Products, Inc.': 'Easy Gardner Products',

    'Eaton, Corp.': 'Eaton',
    'Eaton Corporation, Inc.': 'Eaton',
    'Eaton, Inc.': 'Eaton',

    'Eaton Aeroquip, Inc.': 'Eaton Aeroquip',
    'Eaton Aeroquip, LLC': 'Eaton Aeroquip',

    'Eatoni Ergonomics, Inc.': 'Eatoni Ergonomics',
    'Eatoni Ergonomics, LLC': 'Eatoni Ergonomics',

    'Ebi, L.P.': 'Ebi',
    'Ebi, LLC': 'Ebi',

    'Ebonite International, Inc.': 'Ebonite International',

    'Ebsco Industries, Inc.': 'Ebsco Industries',
    'Elite Manufacturing, Co.': 'Elite Manufacturing',
    'Elite Manufacturing, Corp.': 'Elite Manufacturing',

    'Elizabeth Arden, Co.': 'Elizabeth Arden',
    'Elizabeth Arden, Inc.': 'Elizabeth Arden',

    'Elk Corporation Of Dallas, Inc.': 'Elk Corporation Of Dallas',

    'Ellison Company, Inc.': 'Ellison',
    'Ellison, Co.': 'Ellison',

    'Eloqua, Corp.': 'Eloqua',
    'Eloqua, Ltd.': 'Eloqua',

    'Elsa L, Inc.': 'Elsa L',
    'Elsa L, Corp.': 'Elsa L',

    'Elscint, Inc.': 'Elscint',
    'Elscint, Ltd.': 'Elscint',

    'Embarq, Corp.': 'Embarq',
    'Embarq, Inc.': 'Embarq',

    'Emerson Electric Co.': 'Emerson Electric',
    'Emerson Electric Co., Inc.': 'Emerson Electric',
    'Emerson Electric Company, Inc.': 'Emerson Electric',

    'Emerson Industrial Automation USA, Inc.': 'Emerson Industrial Automation USA',
    'Emerson Industrial Automation USA, LLC': 'Emerson Industrial Automation USA',

    'Emhart Teknologies, LLC': 'Emhart Teknologies',
    'Emhart Teknologies, Inc.': 'Emhart Teknologies',

    'Emine Technology Co., Ltd.': 'Emine Technology',
    'Emine Technology Co, Ltd.': 'Emine Technology',

    'Empak, Inc.': 'Empak',
    'Empak, Corp.': 'Empak',

    'Emporium Leather Company, Inc.': 'Emporium Leather',
    'Emporium Leather Co., Inc.': 'Emporium Leather',

    'Emson, Inc.': 'Emson',
    'Emson, Corp.': 'Emson',

    'Encore Medical, Corp.': 'Encore Medical',
    'Encore Medical, L.P.': 'Encore Medical',

    'Enelf, Inc.': 'Enelf',
    'Enelf, LLC': 'Enelf',

    'Enersyst Development Center, Inc.': 'Enersyst Development Center',
    'Enersyst Development Center, LLC': 'Enersyst Development Center',

    'Engineered Plastics, Inc.': 'Engineered Plastics',
    'Engineered Plastics, LLC': 'Engineered Plastics',

    'Engis, Corp.': 'Engis',
    'Engis, Ltd.': 'Engis',

    'Enpac, LLC': 'Enpac',
    'Enpac, Corp.': 'Enpac',

    'Enrich International, Inc.': 'Enrich International',

    'Entek IRD International, Corp.': 'Entek IRD International',

    'Entek Systems, LLC': 'Entek Systems',
    'Entek Systems, Inc.': 'Entek Systems',

    'Entercom Communications, Corp.': 'Entercom Communications',

    'Entertainment Distribution, Co.': 'Entertainment Distribution',
    'Entertainment Distribution Company, LLC': 'Entertainment Distribution',

    'Environ Products, Inc.': 'Environ Products',

    'Environmaster International, Corp.': 'Environmaster International',
    'Environmaster International, LLC': 'Environmaster International',

    'Enzymotec USA, Inc.': 'Enzymotec USA',

    'Equipment Services, Inc.': 'Equipment Services',

    'Esco, Corp.': 'Esco',
    'Esco, LLC': 'Esco',

    'Esna Technologies, Corp.': 'Esna Technologies',
    'Esna Technologies, Inc.': 'Esna Technologies',

    'Esquire Deposition Services, LLC': 'Esquire Deposition Services',

    'Esselte, Corp.': 'Esselte',
    'Esselte, Ltd.': 'Esselte',

    'Etagz, Inc.': 'Etagz',

    'Ethan Group, Ltd.': 'Ethan Group',
    'Ethan Group, Inc.': 'Ethan Group',

    'Ethicon Endo-Surgery, Inc.': 'Ethicon Endo-Surgery',
    'Ethicon Endo-Surgery, LLC': 'Ethicon Endo-Surgery',

    'Etna Products Co., Inc.': 'Etna Products',
    'Etna Products Company, Inc.': 'Etna Products',
    'Etna Products Co, Inc.': 'Etna Products',

    'Etonic Worldwide, Corp.': 'Etonic Worldwide',
    'Etonic Worldwide, LLC': 'Etonic Worldwide',
    'Daimler Chrysler, Corp.': 'DaimlerChrysler',
    'Daimler Chrysler Co, LLC': 'DaimlerChrysler',
    'Daimlerchrysler, Corp.': 'DaimlerChrysler',
    'Daimlerchrysler Company, LLC': 'DaimlerChrysler',

    'Daisy Manufacturing Company, Inc.': 'Daisy Manufacturing',
    'Daisy Manufacturing, Co.': 'Daisy Manufacturing',

    'Dale Chavez, Co.': 'Dale Chavez',
    'Dale Chavez Company, Inc.': 'Dale Chavez',
    'Dale Chavez Co., Inc.': 'Dale Chavez',
    'Dale Chavez Co, Inc.': 'Dale Chavez',

    'Dale Electronics, Inc.': 'Dale Electronics',

    'Dalow Industries, Inc.': 'Dalow Industries',

    'Damark International, Inc.': 'Damark International',
    'Everbrite, Inc.': 'Everbrite',
    'Everbrite, LLC': 'Everbrite',

    'Eveready Battery Company, Inc.': 'Eveready Battery',
    'Eveready Battery Co., Inc.': 'Eveready Battery',
    'Eveready Battery Co, Inc.': 'Eveready Battery',

    'Eveready Industrial Services, Corp.': 'Eveready Industrial Services',
    'Eveready Industrial Services, Inc.': 'Eveready Industrial Services',

    'Everett Charles Technologies, Inc.': 'Everett Charles Technologies',

    'Everpure, LLC': 'Everpure',
    'Everpure, Inc.': 'Everpure',

    'Everything Furniture, Inc.': 'Everything Furniture',

    'Evike.com, Inc.': 'Evike.com',
    'Evike.com, Co.': 'Evike.com',

    'Evite, Inc.': 'Evite',
    'Evite, LLC': 'Evite',

    'Excel Industries, Inc.': 'Excel Industries',
    'Excel Industries, Ltd.': 'Excel Industries',

    'Excel Mining Systems, LLC': 'Excel Mining Systems',
    'Excel Mining Systems, Inc.': 'Excel Mining Systems',

    'Excel Paint Applicator, LLC': 'Excel Paint Applicator',
    'Excel Paint Applicator, Inc.': 'Excel Paint Applicator',

    'Excelstor Technology, Ltd.': 'Excelstor Technology',
    'Excelstor Technology, Inc.': 'Excelstor Technology',

    'Excite, Inc.': 'Excite',
    'Excite, Ltd.': 'Excite',

    'Expedeon, Ltd.': 'Expedeon',
    'Expedeon, Inc.': 'Expedeon',

    'Express, Inc.': 'Express',
    'Express, LLC': 'Express',

    'Extreme Reseach, Inc.': 'Extreme Reseach',
    'Extreme Reseach, Corp.': 'Extreme Reseach',

    'F and G Research, Inc.': 'F and G Research',

    'F-Secure, Inc.': 'F-Secure',
    'F-Secure, Corp.': 'F-Secure',

    'F.W. Woolworth, Co.': 'F.W. Woolworth',
    'F.W. Woolworth, Inc.': 'F.W. Woolworth',

    'FAS Technologies, Ltd.': 'FAS Technologies',
    'FAS Technologies, Inc.': 'FAS Technologies',

    'FEI, Corp.': 'FEI',
    'FEI, Co.': 'FEI',

    'FGX International, Inc.': 'FGX International',

    'FKI Industries, Inc.': 'FKI Industries',

    'FM Industries, Inc.': 'FM Industries',

    'FMT Corporation, Inc.': 'FMT',
    'FMT, Corp.': 'FMT',

    'FSP Group, Inc.': 'FSP Group',

    'FSV Payment Systems, Inc.': 'FSV Payment Systems',
    'FSV Payment Systems, Ltd.': 'FSV Payment Systems',

    'Facility Management Systems, Inc.': 'Facility Management Systems',

    'Fairchild Semiconductor, Corp.': 'Fairchild Semiconductor',
    'Fairchild Semiconductor Corporation, Inc.': 'Fairchild Semiconductor',

    'Falcon Enterprises, Inc.': 'Falcon Enterprises',

    'Famous Music, Corp.': 'Famous Music',
    'Famous Music, LLC': 'Famous Music',

    'Fandango, Inc.': 'Fandango',
    'Fandango, LLC': 'Fandango',

    'Fanuc, Ltd.': 'Fanuc',
    'Fanuc, Corp.': 'Fanuc',

    'Farmland Industries, Inc.': 'Farmland Industries',

    'Faro Technologies, Inc.': 'Faro Technologies',

    'Fascinations Toys & Gifts, Inc.': 'Fascinations Toys & Gifts',

    'Fastenal, Co.': 'Fastenal',
    'Fastenal, Inc.': 'Fastenal',

    'Faus Group, Inc.': 'Faus Group',

    'Fawn Engineering, Corp.': 'Fawn Engineering',

    'FedEx Freight, Inc.': 'FedEx Freight',
    'FedEx Freight, Corp.': 'FedEx Freight',

    'Federal Laboratories, Corp.': 'Federal Laboratories',
    'Federal Laboratories, Inc.': 'Federal Laboratories',

    'Feel Golf Company, Inc.': 'Feel Golf',
    'Feel Golf Co., Inc.': 'Feel Golf',

    'Feelux Lighting, Inc.': 'Feelux Lighting',
    'Feelux Lighting Co., Ltd.': 'Feelux Lighting',

    'Femtosoft Technologies, LLC': 'Femtosoft Technologies',

    'Fergus Concrete Products, Co.': 'Fergus Concrete Products',

    'Ferraz Shawmut, LLC': 'Ferraz Shawmut',
    'Ferraz Shawmut, Inc.': 'Ferraz Shawmut',

    'Fezer North America, Inc.': 'Fezer North America',
    'Fezer North America, LLC': 'Fezer North America',

    'Fibercore, Ltd.': 'Fibercore',
    'Fibercore, Inc.': 'Fibercore',
     'Flents Products Co., Inc.': 'Flents Products',
    'Flents Products, Co.': 'Flents Products',

    'Flex-O-Glass, Co.': 'Flex-O-Glass',
    'Flex-O-Glass, Inc.': 'Flex-O-Glass',

    'Flex-Rest, LLC': 'Flex-Rest',
    'Flex-Rest, Inc.': 'Flex-Rest',

    'Flexible Foam Products, Inc.': 'Flexible Foam Products',
    'Flexible Foam Products, LLC': 'Flexible Foam Products',

    'Flixster, LLC': 'Flixster',
    'Flixster, Inc.': 'Flixster',

    'Floatec, Inc.': 'Floatec',
    'Floatec, Corp.': 'Floatec',

    'Foldfast, Inc.': 'Foldfast',
    'Foldfast, LLC': 'Foldfast',

    'Fibertec, Inc.': 'Fibertec',
    'Fibertec, Corp.': 'Fibertec',

    'Fiddlers Green Golf Co, Inc.': 'Fiddlers Green Golf',
    'Fiddlers Green Golf, Co.': 'Fiddlers Green Golf',

    'Fidelity Information Services, Inc.': 'Fidelity Information Services',
    'Fidelity Information Services, LLC': 'Fidelity Information Services',

    'Fidelity Investments, Inc.': 'Fidelity Investments',
    'Fidelity Investments, LLC': 'Fidelity Investments',

    'Fidelix Co, Ltd.': 'Fidelix',
    'Fidelix Co., Ltd.': 'Fidelix',

    'Fieldturf International, Inc.': 'Fieldturf International',
    'Fieldturf International , Inc.': 'Fieldturf International',

    'Fieldturf USA, Inc.': 'Fieldturf USA',

    'Fin Machine Company, Ltd.': 'Fin Machine',
    'Fin Machine, Co.': 'Fin Machine',

    'Finest Industrial Co., Ltd.': 'Finest Industrial',
    'Finest Industrial, Ltd.': 'Finest Industrial',
    'Finest Industrial Co, Ltd.': 'Finest Industrial',

    'FingerTec USA, Inc.': 'FingerTec USA',
    'FingerTec USA, LLC': 'FingerTec USA',

    'Finjan Software, Ltd.': 'Finjan Software',
    'Finjan Software, Inc.': 'Finjan Software',

    'FireKing International, LLC': 'FireKing International',
    'FireKing International, Inc.': 'FireKing International',

    'First Access, Inc.': 'First Access',
    'First Access, Ltd.': 'First Access',

    'First Citizens Bank and Trust Company, Inc.': 'First Citizens Bank and Trust',
    'First Citizens Bank and Trust, Co.': 'First Citizens Bank and Trust',

    'First Commercial Bank Co., Ltd.': 'First Commercial Bank',

    'First Data, Corp.': 'First Data',
    'First Data, LLC': 'First Data',

    'First Media Group, Inc.': 'First Media Group',
    'First Media Group, LLC': 'First Media Group',

    'First Prominence Co., Ltd.': 'First Prominence',
    'First Prominence Co, Ltd.': 'First Prominence',

    'Firstech, Inc.': 'Firstech',
    'Firstech, LLC': 'Firstech',

    'Firstline, Corp.': 'Firstline',
    'Firstline, LLC': 'Firstline',

    'Fisher & Paykel Appliances, Ltd.': 'Fisher & Paykel Appliances',
    'Fisher & Paykel Appliances, Inc.': 'Fisher & Paykel Appliances',

    'Fisher Controls International, LLC': 'Fisher Controls International',
    'Fisher Controls International, Inc.': 'Fisher Controls International',

    'Fisher Hamilton, LLC': 'Fisher Hamilton',
    'Fisher Hamilton, Inc.': 'Fisher Hamilton',

    'Fisher and Paykel HealthCare, Inc.': 'Fisher and Paykel HealthCare',
    'Fisher and Paykel HealthCare, Ltd.': 'Fisher and Paykel HealthCare',

    'Fiskars, Inc.': 'Fiskars',
    'Fiskars, Corp.': 'Fiskars',

    'Fitbug, Inc.': 'Fitbug',
    'Fitbug, Ltd.': 'Fitbug',

    'Fitness Anywhere, LLC': 'Fitness Anywhere',
    'Fitness Anywhere, Inc.': 'Fitness Anywhere',

    'Fitness Concepts, Inc.': 'Fitness Concepts',

    'Fitness Equipment Services, LLC': 'Fitness Equipment Services',

    'Fitness Master, Inc.': 'Fitness Master',

    'Fitness Quest, Inc.': 'Fitness Quest',

    'Fitness Warehouse, Inc.': 'Fitness Warehouse',

    'Flanders Industries, Inc.': 'Flanders Industries',
    'Ford Motor Co.': 'Ford Motor',
    'Ford Motor Company, Inc.': 'Ford Motor',
    'Ford Global Technologies, Inc.': 'Ford Motor',
    'Ford Global Technologies, LLC': 'Ford Motor'

}

names['standardized_org_name'] = names['standardized_org_name'].replace(canonical_map)

KeyboardInterrupt: 

In [ ]:
import re

# Similar groups map - Batch 1 (A entries)
#
# Strategy notes:
# - REGEX used when variants are clearly the same entity with
#   only suffix/punctuation differences
# - MAP entries used when the base name is specific enough
#   to be a single company
# - SKIPPED entries that are too generic (ABC, AMF, ADI etc.)
#   or are clearly non-specific location descriptors

# These catch suffix-only variants of the same company

REGEX_RULES = [
    # 180S / 180s — same brand, case inconsistency
    (r'^180[Ss],?\s*(Inc\.|LLC|Corp\.)?$', '180S Inc.'),

    # A-Top Technology
    (r'^A-Top Technology,?\s*(Inc\.)?$', 'A-Top Technology Inc.'),

    # A.C. Aukerman
    (r'^A\.?\s*C\.?\s*Aukerman\s*(Company,?\s*)?(Inc\.|Co\.)?$', 'A.C. Aukerman Co.'),

    # ABB — real company (ABB Ltd, Swiss industrial conglomerate)
    (r'^ABB,?\s*(Inc\.|Ltd\.)$', 'ABB Ltd.'),
    (r'^Abb,?\s*(Inc\.|Ltd\.)$', 'ABB Ltd.'),

    # ABC / Abc — too generic, skip (don't map)

    # ABC Distributing
    (r'^ABC Distributing,?\s*(Co\.|LLC)$', 'ABC Distributing Co.'),

    # ADB Airfield Solutions
    (r'^ADB Airfield Solutions,?\s*(LLC)?$', 'ADB Airfield Solutions LLC'),

    # AE Tech
    (r'^AE Tech Co\.?,?\s*Ltd\.$', 'AE Tech Co. Ltd.'),

    # AFC Enterprises
    (r'^AFC Enterprises,?\s*(Inc\.)?$', 'AFC Enterprises Inc.'),

    # AJ Manufacturing
    (r'^AJ Manufacturing\s*(Company,?\s*)?,?\s*(LLC|Inc\.|Co\.)?$', 'AJ Manufacturing Inc.'),

    # AM General
    (r'^AM General,?\s*(Corp\.|LLC)$', 'AM General Corp.'),

    # AS America
    (r'^AS America,?\s*(Inc\.|LLC)$', 'AS America Inc.'),

    # ASSA — could be ASSA ABLOY, keep together
    (r'^ASSA,?\s*(Corp\.|Inc\.)$', 'ASSA Corp.'),

    # AT & T → already handled in main map, but catch this spacing variant
    (r'^AT\s*&\s*T,?\s*(Corp\.|Inc\.)$', 'AT&T Inc.'),

    # Abraxis Bioscience
    (r'^Abraxis Bioscience,?\s*(Inc\.|LLC)$', 'Abraxis Bioscience Inc.'),

    # Acacia Research Group — NPE
    (r'^Acacia Research Group,?\s*(LLC)?$', 'Acacia Research Group LLC'),

    # Acacia Patent Acquisition — NPE
    (r'^Acacia Patent Acquisition,?\s*(LLC|Corp\.)$', 'Acacia Patent Acquisition LLC'),

    # Acacia Technology Services — NPE
    (r'^Acacia Technology Services,?\s*(Corp\.|LLC)$', 'Acacia Technology Services Corp.'),

    # Academy Bus
    (r'^Academy Bus\s*(Company,?\s*)?,?\s*(LLC|Inc\.)?$', 'Academy Bus LLC'),

    # Access Co.
    (r'^Access Co\.?,?\s*Ltd\.$', 'Access Co. Ltd.'),

    # Accessory Network Group
    (r'^Accessory Network Group,?\s*(Inc\.|LLC)$', 'Accessory Network Group Inc.'),

    # Acco Brands
    (r'^Acco Brands,?\s*(Inc\.|Corp\.)$', 'Acco Brands Corp.'),

    # Accor North America
    (r'^Accor North America,?\s*(Corp\.|Inc\.)$', 'Accor North America Corp.'),

    # Accton Technology
    (r'^Accton Technology,?\s*(Corp\.)?$', 'Accton Technology Corp.'),

    # Accuform Golf
    (r'^Accuform Golf,?\s*(Ltd\.|Corp\.)$', 'Accuform Golf Ltd.'),

    # Accurate Tool
    (r'^Accurate Tool,?\s*(Inc\.|Co\.)$', 'Accurate Tool Inc.'),

    # Ace Medical
    (r'^Ace Medical,?\s*(Corp\.|Co\.)$', 'Ace Medical Corp.'),

    # Ace Products
    (r'^Ace Products,?\s*(Inc\.)?$', 'Ace Products Inc.'),

    # Acer American Holdings
    (r'^Acer American Holdings,?\s*(Inc\.|Corp\.)$', 'Acer American Holdings Inc.'),

    # Acer Communications & Multimedia America
    (r'^Acer Communications & Multimedia America,?\s*(Inc\.)?$', 'Acer Communications & Multimedia America Inc.'),

    # Acme International
    (r'^Acme International,?\s*(Inc\.|LLC)?$', 'Acme International Inc.'),

    # Acorn Engineering
    (r'^Acorn Engineering\s*(Company,?\s*|Co,?\s*)?(Inc\.)?$', 'Acorn Engineering Co. Inc.'),

    # Acorn Products
    (r'^Acorn Products\s*Co\.?,?\s*(Inc\.)?$', 'Acorn Products Co. Inc.'),

    # Action Electronics
    (r'^Action Electronics\s*(Co\.?,?\s*Ltd\.)?$', 'Action Electronics Co. Ltd.'),

    # Action Enterprises
    (r'^Action Enterprises,?\s*(Inc\.)?$', 'Action Enterprises Inc.'),

    # Action Star Enterprise
    (r'^Action Star Enterprise\s*Co\.?,?\s*Ltd\.$', 'Action Star Enterprise Co. Ltd.'),

    # Action Technologies
    (r'^Action Technologies,?\s*(LLC|Inc\.)$', 'Action Technologies LLC'),

    # Active Shock
    (r'^Active Shock,?\s*(Inc\.|LLC)$', 'Active Shock Inc.'),

    # Active Voice
    (r'^Active Voice,?\s*(Corp\.|Inc\.)$', 'Active Voice Corp.'),

    # Acumed
    (r'^Acumed,?\s*(Inc\.|LLC)$', 'Acumed Inc.'),

    # Acushnet
    (r'^Acushnet\s*(Company,?\s*)?,?\s*(Inc\.|Co\.)?$', 'Acushnet Co.'),

    # Acuson
    (r'^Acuson,?\s*(Inc\.|Corp\.)$', 'Acuson Corp.'),

    # Adams Golf
    (r'^Adams Golf,?\s*(Inc\.|Ltd\.)$', 'Adams Golf Inc.'),

    # Adams Manufacturing
    (r'^Adams Manufacturing\s*(Company,?\s*)?,?\s*(Co\.|Inc\.)?$', 'Adams Manufacturing Co.'),

    # Adaptive Micro Systems
    (r'^Adaptive Micro Systems,?\s*(LLC|Inc\.)$', 'Adaptive Micro Systems LLC'),

    # Adda USA
    (r'^Adda USA,?\s*(Corp\.)?$', 'Adda USA Corp.'),

    # Adjustable Clamp
    (r'^Adjustable Clamp\s*(Company,?\s*)?,?\s*(Inc\.|Co\.)?$', 'Adjustable Clamp Co.'),

    # Advance Metalworking
    (r'^Advance Metalworking\s*(Co\.?,?\s*)?(Inc\.)?$', 'Advance Metalworking Co. Inc.'),

    # Advance Watch
    (r'^Advance Watch\s*(CO|Co|Company)?,?\s*Ltd\.$', 'Advance Watch Co. Ltd.'),

    # Advanced Bionics
    (r'^Advanced Bionics,?\s*(Corp\.|LLC)$', 'Advanced Bionics Corp.'),

    # Advanced Care Pharmacy
    (r'^Advanced Care Pharmacy,?\s*(Inc\.|LLC)$', 'Advanced Care Pharmacy Inc.'),

    # Advanced Display
    (r'^Advanced Display,?\s*(Inc\.|Co\.)$', 'Advanced Display Inc.'),

    # Advanced Duplication Services
    (r'^Advanced Duplication Services,?\s*(LLC)?$', 'Advanced Duplication Services LLC'),

    # Advanced Geotech Systems
    (r'^Advanced Geotech Systems,?\s*(Inc\.|LLC)$', 'Advanced Geotech Systems Inc.'),

    # Advanced Media Networks
    (r'^Advanced Media Networks,?\s*(LLC)?$', 'Advanced Media Networks LLC'),

    # Advanced Polymer Technology
    (r'^Advanced Polymer Technology,?\s*(Inc\.)?$', 'Advanced Polymer Technology Inc.'),

    # Advanced Technology Laboratories
    (r'^Advanced Technology Laboratories,?\s*(Inc\.)?$', 'Advanced Technology Laboratories Inc.'),

    # Advantica / Advantica Technologies — two separate companies, keep separate
    (r'^Advantica\s*,?\s*(Inc\.)?$', 'Advantica Inc.'),
    (r'^Advantica Technologies,?\s*(Inc\.|Ltd\.)$', 'Advantica Technologies Inc.'),

    # Aearo
    (r'^Aearo,?\s*(Corp\.|Co\.)$', 'Aearo Corp.'),

    # AeroScout
    (r'^AeroScout,?\s*(Ltd\.|Inc\.)$', 'AeroScout Ltd.'),

    # Aerotel U.S.A. / Aerotel USA — same company, normalize
    (r'^Aerotel\s*U\.?S\.?A\.?,?\s*(Inc\.|LLC)$', 'Aerotel USA Inc.'),

    # Affiliated Computer Services
    (r'^Affiliated Computer Services,?\s*(Inc\.)?$', 'Affiliated Computer Services Inc.'),

    # Ag-Chem Equipment
    (r'^Ag-Chem Equipment\s*(Company,?\s*)?,?\s*(Inc\.|Co\.)?$', 'Ag-Chem Equipment Co.'),

    # Akrion
    (r'^Akrion,?\s*(LLC|Inc\.)$', 'Akrion LLC'),

    # Aladdin / Alladin Knowledge Systems — same company, typo variant
    (r'^(?:Aladdin|Alladin) Knowledge Systems,?\s*(Inc\.|Ltd\.)$', 'Aladdin Knowledge Systems Inc.'),

    # Alamy
    (r'^Alamy,?\s*(Ltd\.|Inc\.)$', 'Alamy Ltd.'),

    # Alaris Medical Systems
    (r'^Alaris Medical Systems,?\s*(Inc\.)?$', 'Alaris Medical Systems Inc.'),

    # Albertson's / Albertsons — same company
    (r"^Albertson'?s,?\s*(Inc\.|LLC)$", "Albertsons LLC"),

    # Alcatel USA
    (r'^Alcatel USA,?\s*(Inc\.)?$', 'Alcatel USA Inc.'),

    # Alchemy Worldwide
    (r'^Alchemy Worldwide,?\s*(LLC|Inc\.)$', 'Alchemy Worldwide LLC'),

    # Alco Electronics
    (r'^Alco Electronics,?\s*(Ltd\.|Inc\.)$', 'Alco Electronics Ltd.'),

    # Alembic Pharmaceuticals
    (r'^Alembic Pharmaceuticals,?\s*(Ltd\.|Inc\.)$', 'Alembic Pharmaceuticals Ltd.'),

    # Alere
    (r'^Alere,?\s*(Inc\.|LLC)$', 'Alere Inc.'),

    # Alex and Ani
    (r'^Alex and Ani,?\s*(Inc\.|LLC)$', 'Alex and Ani Inc.'),

    # Alfresco Software
    (r'^Alfresco Software,?\s*(Ltd\.|Inc\.)$', 'Alfresco Software Ltd.'),

    # Alibaba.com
    (r'^Alibaba\.com,?\s*(Inc\.|Ltd\.)$', 'Alibaba.com Ltd.'),

    # Aliphcom
    (r'^Aliphcom,?\s*(Inc\.)?$', 'Aliphcom Inc.'),

    # All-Luminum Products
    (r'^All-Luminum Products,?\s*(Inc\.)?$', 'All-Luminum Products Inc.'),

    # Allegiance Healthcare
    (r'^Allegiance Healthcare,?\s*(Corp\.)?$', 'Allegiance Healthcare Corp.'),

    # Allen Engineering
    (r'^Allen Engineering,?\s*(Corp\.)?$', 'Allen Engineering Corp.'),

    # Allen Telecom
    (r'^Allen Telecom,?\s*(Inc\.)?$', 'Allen Telecom Inc.'),

    # Allen-Bradley — Rockwell Automation subsidiary
    (r'^Allen-Bradley\s*(Company,?\s*)?,?\s*(Inc\.|LLC|Co\.)?$', 'Allen-Bradley Co.'),

    # Alliance Data Systems
    (r'^Alliance Data Systems,?\s*(Corp\.|LLC)$', 'Alliance Data Systems Corp.'),

    # Alliance Entertainment
    (r'^Alliance Entertainment,?\s*(LLC|Corp\.)$', 'Alliance Entertainment LLC'),

    # Alliance Research
    (r'^Alliance Research\s*(Corp,?\s*)?,?\s*(Inc\.)?$', 'Alliance Research Corp.'),

    # Allied Colloids
    (r'^Allied Colloids,?\s*(Inc\.|Ltd\.)$', 'Allied Colloids Inc.'),

    # Allied Communications
    (r'^Allied Communications,?\s*(Inc\.|Corp\.)$', 'Allied Communications Inc.'),

    # Allied International
    (r'^Allied International,?\s*(Inc\.)?$', 'Allied International Inc.'),

    # Allied Signal
    (r'^Allied Signal,?\s*(Inc\.|Corp\.)$', 'Allied Signal Inc.'),

    # Allied Tube & Conduit
    (r'^Allied Tube & Conduit,?\s*(Corp\.|Inc\.)$', 'Allied Tube & Conduit Corp.'),

    # Allstate Insurance
    (r'^Allstate Insurance\s*(Company,?\s*)?,?\s*(Inc\.|Co\.)?$', 'Allstate Insurance Co.'),

    # Alltel Communications
    (r'^Alltel Communications,?\s*(Inc\.|LLC)$', 'Alltel Communications Inc.'),

    # Allwall Technologies
    (r'^Allwall Technologies,?\s*(Inc\.|Co\.)$', 'Allwall Technologies Inc.'),

    # Alma Lasers
    (r'^Alma Lasers,?\s*(Ltd\.|Inc\.)$', 'Alma Lasers Ltd.'),

    # Almar Sales
    (r'^Almar Sales\s*(Company,?\s*|Co\.,?\s*)?(Inc\.)?$', 'Almar Sales Co. Inc.'),

    # Almell Products
    (r'^Almell Products,?\s*(Ltd\.|Inc\.)$', 'Almell Products Ltd.'),

    # Aloe Health Fitness
    (r'^Aloe Health Fitness,?\s*(Inc\.)?$', 'Aloe Health Fitness Inc.'),

    # Aloha Housewares
    (r'^Aloha Housewares\s*(Company,?\s*)?,?\s*(Inc\.|Co\.,?\s*Ltd\.)?$', 'Aloha Housewares Inc.'),

    # Alpha Products
    (r'^Alpha Products,?\s*(Inc\.|Co\.)$', 'Alpha Products Inc.'),

    # Alpha and Omega Semiconductor
    (r'^Alpha and Omega Semiconductor,?\s*(Inc\.|Ltd\.)$', 'Alpha and Omega Semiconductor Inc.'),

    # Alpine Products
    (r'^Alpine Products,?\s*(Inc\.)?$', 'Alpine Products Inc.'),

    # Altaire Pharmaceuticals
    (r'^Altaire Pharmaceuticals,?\s*(Inc\.)?$', 'Altaire Pharmaceuticals Inc.'),

    # Altec Lansing Technologies
    (r'^Altec Lansing Technologies,?\s*(Inc\.)?$', 'Altec Lansing Technologies Inc.'),

    # Alteva
    (r'^Alteva,?\s*(Inc\.|LLC)$', 'Alteva Inc.'),

    # Altman Stage Lighting
    (r'^Altman Stage Lighting\s*(Company,?\s*|Co\.,?\s*)?(Inc\.)?$', 'Altman Stage Lighting Co. Inc.'),

    # Aluminum Company of America (Alcoa)
    (r'^Aluminum Company of America,?\s*(Inc\.)?$', 'Aluminum Company of America'),

    # Alvogen Pine Brook
    (r'^Alvogen Pine Brook,?\s*(Inc\.|LLC)$', 'Alvogen Pine Brook Inc.'),

    # Alwin Manufacturing
    (r'^Alwin Manufacturing\s*(Company,?\s*|Co\.,?\s*)?(Inc\.)?$', 'Alwin Manufacturing Co. Inc.'),

    # Alzheimer's Institute of America
    (r"^Alzheimer's Institute of America,?\s*(Inc\.)?$", "Alzheimer's Institute of America Inc."),

    # Amazon.com — already in main map but catch bare variant
    (r'^Amazon\.com$', 'Amazon.com Inc.'),

    # Ambit Microsystems
    (r'^Ambit Microsystems,?\s*(Inc\.|Corp\.)$', 'Ambit Microsystems Corp.'),

    # Ambry Genetics
    (r'^Ambry Genetics,?\s*(Corp\.)?$', 'Ambry Genetics Corp.'),

    # Amcor Industries
    (r'^Amcor Industries,?\s*(Inc\.)?$', 'Amcor Industries Inc.'),

    # Amdocs
    (r'^Amdocs,?\s*(Inc\.|Ltd\.)$', 'Amdocs Ltd.'),

    # America Greener Technologies
    (r'^America Greener Technologies,?\s*(Inc\.|Corp\.)$', 'America Greener Technologies Inc.'),

    # American Allsafe
    (r'^American Allsafe\s*(Company,?\s*)?,?\s*(Co\.|Inc\.)?$', 'American Allsafe Co.'),

    # American Apparel
    (r'^American Apparel,?\s*(Inc\.|LLC)$', 'American Apparel Inc.'),

    # American Axle and Manufacturing
    (r'^American Axle and Manufacturing,?\s*(Inc\.)?$', 'American Axle and Manufacturing Inc.'),

    # American Bank Note / American Bank Note Holographics — different entities
    (r'^American Bank Note,?\s*(Co\.|Corp\.)$', 'American Bank Note Co.'),
    (r'^American Bank Note Holographics,?\s*(Inc\.|Corp\.)$', 'American Bank Note Holographics Inc.'),

    # American Builders & Contractors Supply
    (r'^American Builders & Contractors Supply\s*(Co\.|Company)?,?\s*(Inc\.)?$', 'American Builders & Contractors Supply Co. Inc.'),

    # American Colloid
    (r'^American Colloid\s*(Company,?\s*)?,?\s*(Inc\.|Co\.)?$', 'American Colloid Co.'),

    # American Covers
    (r'^American Covers,?\s*(Inc\.)?$', 'American Covers Inc.'),

    # American Dental Technologies
    (r'^American Dental Technologies,?\s*(Inc\.)?$', 'American Dental Technologies Inc.'),

    # American Electronic Sign
    (r'^American Electronic Sign\s*Co,?\s*(Inc\.)?$', 'American Electronic Sign Co. Inc.'),

    # American Eurocopter
    (r'^American Eurocopter,?\s*(Corp\.|LLC)$', 'American Eurocopter Corp.'),

    # American Express Travel Related Services
    (r'^American Express Travel Related Services\s*(Company,?\s*)?,?\s*(Inc\.)?$', 'American Express Travel Related Services Inc.'),

    # American Home Products
    (r'^American Home Products,?\s*(Inc\.|Corp\.)?$', 'American Home Products Corp.'),

    # American Honda Motor — already handled in main map
    (r'^American Honda Motor\s*CO?\.?,?\s*(Inc\.)?$', 'Honda Motor Co. Ltd.'),

    # American Household Products
    (r'^American Household Products,?\s*(Inc\.)?$', 'American Household Products Inc.'),

    # American International Industries
    (r'^American International Industries,?\s*(Inc\.)?$', 'American International Industries Inc.'),

    # American Marketing Enterprises
    (r'^American Marketing Enterprises,?\s*(Inc\.)?$', 'American Marketing Enterprises Inc.'),

    # American Metal Products
    (r'^American Metal Products,?\s*(Inc\.)?$', 'American Metal Products Inc.'),

    # American Microsystems
    (r'^American Microsystems,?\s*(Inc\.|Ltd\.,?\s*Inc\.)?$', 'American Microsystems Inc.'),

    # American National Can
    (r'^American National Can,?\s*(Corp\.|Co\.)$', 'American National Can Corp.'),

    # American Power Products
    (r'^American Power Products,?\s*(Inc\.)?$', 'American Power Products Inc.'),

    # American Radionic
    (r'^American Radionic\s*(Co\.|Company)?,?\s*(Inc\.)?$', 'American Radionic Co. Inc.'),

    # American Recreation Products
    (r'^American Recreation Products,?\s*(Inc\.|LLC)$', 'American Recreation Products Inc.'),

    # American Safety Razor
    (r'^American Safety Razor\s*(Company,?\s*)?,?\s*(Co\.|LLC)?$', 'American Safety Razor Co.'),

    # American Seed
    (r'^American Seed\s*(Company,?\s*)?,?\s*(Inc\.|Co\.)?$', 'American Seed Co.'),

    # American Stock Exchange
    (r'^American Stock Exchange,?\s*(Inc\.|LLC)$', 'American Stock Exchange Inc.'),

    # American Tack & Hardware
    (r'^American Tack & Hardware\s*Co\.?,?\s*(Inc\.)?$', 'American Tack & Hardware Co. Inc.'),

    # American Telephone & Telegraph / American Telephone and Telegraph — AT&T
    (r'^American Telephone\s*(&|and)\s*Telegraph,?\s*(Co\.|Inc\.|Corp\.)?$', 'AT&T Inc.'),

    # American Thermal Instruments
    (r'^American Thermal Instruments,?\s*(Co\.|Inc\.)$', 'American Thermal Instruments Inc.'),

    # American Torch Tip
    (r'^American Torch Tip,?\s*(Co\.|Ltd\.)$', 'American Torch Tip Co.'),

    # American Underwater Products
    (r'^American Underwater Products,?\s*(Inc\.)?$', 'American Underwater Products Inc.'),

    # Amerigen Pharmaceuticals
    (r'^Amerigen Pharmaceuticals,?\s*(Inc\.|Ltd\.)$', 'Amerigen Pharmaceuticals Inc.'),

    # Ameristar Fence Products
    (r'^Ameristar Fence Products,?\s*(Inc\.)?$', 'Ameristar Fence Products Inc.'),

    # Ameristep
    (r'^Ameristep,?\s*(Corp\.|Inc\.)$', 'Ameristep Corp.'),

    # Ameritrade / Ameritrade Holding — same parent, keep separate levels
    (r'^Ameritrade,?\s*(Inc\.|LLC)$', 'Ameritrade Inc.'),
    (r'^Ameritrade Holding\s*(Corporation,?\s*)?,?\s*(Corp\.)?$', 'Ameritrade Holding Corp.'),

    # Amersham Pharmacia Biotech
    (r'^Amersham Pharmacia Biotech,?\s*(Inc\.)?$', 'Amersham Pharmacia Biotech Inc.'),

    # Amimon
    (r'^Amimon,?\s*(Inc\.|Ltd\.)$', 'Amimon Inc.'),

    # Amneal Pharmaceuticals
    (r'^Amneal Pharmaceuticals,?\s*(LLC|Co\.)?$', 'Amneal Pharmaceuticals LLC'),

    # Amoisonic Electronics
    (r'^Amoisonic Electronics\s*(Co\.?,?\s*Ltd\.|,?\s*Inc\.)?$', 'Amoisonic Electronics Co. Ltd.'),

    # Amphenol
    (r'^Amphenol\s*(Corporation,?\s*)?,?\s*(Corp\.)?$', 'Amphenol Corp.'),

    # Amtech Systems
    (r'^Amtech Systems,?\s*(LLC|Inc\.|Corp\.)$', 'Amtech Systems Inc.'),

    # Amway
    (r'^Amway,?\s*(Corp\.|Inc\.)$', 'Amway Corp.'),

    # Amylin Pharmaceuticals
    (r'^Amylin Pharmaceuticals,?\s*(LLC|Inc\.)$', 'Amylin Pharmaceuticals LLC'),

    # Anaba Group
    (r'^Anaba Group,?\s*(Inc\.)?$', 'Anaba Group Inc.'),

    # Anchen International Pharmaceuticals / Anchen Pharmaceuticals
    # These are related but distinct entities — keep separate
    (r'^Anchen International Pharmaceuticals\s*(Co|Company)?,?\s*Ltd\.$', 'Anchen International Pharmaceuticals Co. Ltd.'),
    (r'^Anchen Pharmaceuticals,?\s*(Inc\.)?$', 'Anchen Pharmaceuticals Inc.'),

    # Anchor Block
    (r'^Anchor Block\s*(Company,?\s*)?,?\s*(Co\.|Inc\.)?$', 'Anchor Block Co.'),

    # Anchor Wall Systems
    (r'^Anchor Wall Systems,?\s*(Inc\.)?$', 'Anchor Wall Systems Inc.'),

    # Ancra International
    (r'^Ancra International,?\s*(LLC)?$', 'Ancra International LLC'),

    # Anderson Bat
    (r'^Anderson Bat\s*(Company,?\s*)?,?\s*(LLC|Co\.)?$', 'Anderson Bat Co.'),

    # Andrews Bowen
    (r'^Andrews Bowen,?\s*(Inc\.|Ltd\.)$', 'Andrews Bowen Inc.'),

    # Andrx Pharmaceutical (distinct from Andrx Pharmaceuticals already in main map)
    (r'^Andrx Pharmaceutical,?\s*(LLC|Inc\.)$', 'Actavis Inc.'),

    # AngleFix
    (r'^AngleFix,?\s*(Inc\.|LLC)$', 'AngleFix Inc.'),

    # Anheuser-Busch Companies
    (r'^Anheuser-Busch Companies,?\s*(Inc\.|LLC)$', 'Anheuser-Busch Companies Inc.'),

    # Anker Technology
    (r'^Anker Technology\s*(Corp\.|Co,?\s*Ltd\.)?$', 'Anker Technology Corp.'),

    # Anova Food
    (r'^Anova Food,?\s*(Inc\.|LLC)$', 'Anova Food Inc.'),

    # Anritsu
    (r'^Anritsu,?\s*(Corp\.|Co\.)$', 'Anritsu Corp.'),

    # Ansa / Ansa Bottle — KEEP SEPARATE (different products)
    (r'^Ansa\s*(Co\.?,?\s*)?(Inc\.|Company,?\s*Inc\.)?$', 'Ansa Co. Inc.'),
    (r'^Ansa Bottle\s*(Co\.?,?\s*)?(Inc\.)?$', 'Ansa Bottle Co. Inc.'),

    # Ansell Healthcare Products
    (r'^Ansell Healthcare Products,?\s*(Inc\.|LLC)$', 'Ansell Healthcare Products Inc.'),

    # Antec
    (r'^Antec,?\s*(Corp\.|Inc\.)$', 'Antec Corp.'),

    # ApaTech
    (r'^ApaTech,?\s*(Inc\.|Ltd\.)$', 'ApaTech Inc.'),

    # Apogent Technologies
    (r'^Apogent Technologies,?\s*(Inc\.)?$', 'Apogent Technologies Inc.'),

    # Apollo Enterprise Solutions
    (r'^Apollo Enterprise Solutions,?\s*(LLC|Inc\.)?$', 'Apollo Enterprise Solutions LLC'),

    # App Pharmaceuticals
    (r'^App Pharmaceuticals,?\s*(LLC|Inc\.)$', 'App Pharmaceuticals LLC'),

    # Appistry
    (r'^Appistry,?\s*(LLC|Inc\.)$', 'Appistry LLC'),
]

DIRECT_MAP = {
    # AT&T spacing variants
    'AT & T, Corp.':    'AT&T Inc.',
    'AT & T, Inc.':     'AT&T Inc.',

    # Amazon bare variant
    'Amazon.com':       'Amazon.com Inc.',

    # American Telephone variants → AT&T
    'American Telephone & Telegraph, Co.':   'AT&T Inc.',
    'American Telephone & Telegraph, Inc.':  'AT&T Inc.',
    'American Telephone & Telegraph, Corp.': 'AT&T Inc.',
    'American Telephone and Telegraph, Co.': 'AT&T Inc.',
    'American Telephone and Telegraph, Inc.': 'AT&T Inc.',

    # American Honda Motor → Honda
    'American Honda Motor, Co.':     'Honda Motor Co. Ltd.',
    'American Honda Motor CO., Inc.': 'Honda Motor Co. Ltd.',

    # Andrx Pharmaceutical → Actavis
    'Andrx Pharmaceutical, LLC': 'Actavis Inc.',
    'Andrx Pharmaceutical, Inc.': 'Actavis Inc.',

    # Non-specific descriptors
    'A Chinese, Corp.':  'Non-Specific Entity',
    'A Chinese, Co.':    'Non-Specific Entity',
    'An English, Corp.': 'Non-Specific Entity',
    'An English, Co.':   'Non-Specific Entity',
    'An Israeli, Corp.': 'Non-Specific Entity',
    'An Israeli, Co.':   'Non-Specific Entity',
}

def apply_similar_groups_map(name: str) -> str:
    """Apply regex rules then direct map to standardize org names."""
    if not isinstance(name, str):
        return name

    # Check direct map first
    if name in DIRECT_MAP:
        return DIRECT_MAP[name]

    # Try regex rules
    for pattern, replacement in REGEX_RULES:
        if re.match(pattern, name, flags=re.IGNORECASE):
            return replacement

    return name

names['standardized_org_name'] = names['standardized_org_name'].apply(apply_similar_groups_map)

In [ ]:
import re

# Similar groups map - Batch 2 (A-B entries)

REGEX_RULES_2 = [

    # A.C. Aukerman — suffix only variants
    (r'^A\.C\. Aukerman\s*(Company,?\s*)?,?\s*(Co\.|Inc\.)?$', 'A.C. Aukerman Co.'),

    # Allen — too generic alone, but map the suffixed variants together
    (r'^Allen\s*(Company,?\s*|Co\.,?\s*)?(Inc\.)?$', 'Allen Co. Inc.'),

    # Allen-Bradley — already normalized, catch remaining variant
    (r'^Allen-Bradley\s*Co,?\s*(Inc\.)?$', 'Allen-Bradley Co.'),

    # Alliance Research — catch Corp. vs Corp
    (r'^Alliance Research\s*,?\s*Corp\.?$', 'Alliance Research Corp.'),

    # Aloha Housewares — catch Company Ltd variant
    (r'^Aloha Housewares\s*(Company,?\s*)?,?\s*(Inc\.|Ltd\.)?$', 'Aloha Housewares Inc.'),

    # American Electronic Sign — catch remaining variant
    (r'^American Electronic Sign\s*Co\.?,?\s*(Inc\.)?$', 'American Electronic Sign Co. Inc.'),

    # Ameritrade Holding — catch Corporation variant
    (r'^Ameritrade Holding\s*(Corporation,?\s*)?,?\s*(Corp\.)?$', 'Ameritrade Holding Corp.'),

    # Amphenol — catch Corporation variant
    (r'^Amphenol\s*(Corporation,?\s*)?,?\s*(Corp\.)?$', 'Amphenol Corp.'),

    # Ansa Bottle — remaining variant
    (r'^Ansa Bottle,?\s*(Co\.|Co\.\s*Inc\.)?$', 'Ansa Bottle Co. Inc.'),

    # Anker Technology — already normalized variant
    (r'^Anker Technology,?\s*(Corp\.)?$', 'Anker Technology Corp.'),

    # Applera (biotech holding company)
    (r'^Applera,?\s*(Corp\.|Inc\.)$', 'Applera Corp.'),

    # Application Art Laboratories
    (r'^Application Art Laboratories\s*Co\.?,?\s*(Inc\.|Ltd\.)$', 'Application Art Laboratories Co. Ltd.'),
    (r'^Application Art Laboratories,?\s*Co\.$', 'Application Art Laboratories Co. Ltd.'),

    # Applied Ballistics
    (r'^Applied Ballistics,?\s*(LLC|Inc\.)$', 'Applied Ballistics LLC'),

    # Applied Biosystems — Thermo Fisher subsidiary
    (r'^Applied Biosystems,?\s*(Inc\.|LLC)?$', 'Applied Biosystems Inc.'),

    # Applied Industrial Technologies
    (r'^Applied Industrial Technologies,?\s*(Inc\.)?$', 'Applied Industrial Technologies Inc.'),

    # Applied Precision
    (r'^Applied Precision,?\s*(LLC|Inc\.)$', 'Applied Precision LLC'),

    # Applied Vision
    (r'^Applied Vision,?\s*(Ltd\.|Inc\.)$', 'Applied Vision Ltd.'),

    # Aptina Imaging
    (r'^Aptina Imaging,?\s*(Corp\.|Inc\.)$', 'Aptina Imaging Corp.'),

    # Aqua-Leisure Industries
    (r'^Aqua-Leisure Industries,?\s*(Inc\.)?$', 'Aqua-Leisure Industries Inc.'),

    # Aquila
    (r'^Aquila,?\s*(Inc\.|Corp\.)$', 'Aquila Inc.'),

    # ArCzar
    (r'^ArCzar,?\s*(LLC|Inc\.)$', 'ArCzar LLC'),

    # Arch Environmental Equipment
    (r'^Arch Environmental Equipment,?\s*(Inc\.|Co\.)$', 'Arch Environmental Equipment Inc.'),

    # Archos
    (r'^Archos,?\s*(Inc\.|LLC)$', 'Archos Inc.'),

    # Arctic Products
    (r'^Arctic Products,?\s*(Inc\.|LLC)$', 'Arctic Products Inc.'),

    # Arena Industries
    (r'^Arena Industries,?\s*(LLC)?$', 'Arena Industries LLC'),

    # Argus Genetics
    (r'^Argus Genetics,?\s*(LLC|Inc\.)$', 'Argus Genetics LLC'),

    # Arista Records
    (r'^Arista Records,?\s*(LLC|Inc\.)$', 'Arista Records LLC'),

    # ARM Ltd
    (r'^Arm,?\s*(Ltd\.|Inc\.)$', 'ARM Ltd.'),

    # Arminak & Associates
    (r'^Arminak & Associates,?\s*(Inc\.|LLC)$', 'Arminak & Associates Inc.'),

    # Armstrong World Industries
    (r'^Armstrong World Industries,?\s*(Inc\.)?$', 'Armstrong World Industries Inc.'),

    # Aromate Industries
    (r'^Aromate Industries\s*(Company,?\s*)?,?\s*(Ltd\.)?$', 'Aromate Industries Co. Ltd.'),

    # Arris Group
    (r'^Arris Group,?\s*(Inc\.)?$', 'Arris Group Inc.'),

    # Arrow International
    (r'^Arrow International,?\s*(Inc\.|Ltd\.)?$', 'Arrow International Inc.'),

    # Arrow Trading
    (r'^Arrow Trading\s*Co\.?,?\s*(Inc\.)?$', 'Arrow Trading Co. Inc.'),

    # Art Leather Manufacturing
    (r'^Art Leather Manufacturing\s*(Co\.?,?\s*)?(Inc\.)?$', 'Art Leather Manufacturing Co. Inc.'),

    # Art Technology Group
    (r'^Art Technology Group,?\s*(Inc\.|LLC)$', 'Art Technology Group Inc.'),

    # Arterial Vascular Engineering
    (r'^Arterial Vascular Engineering,?\s*(Inc\.)?$', 'Arterial Vascular Engineering Inc.'),

    # Artesyn Technologies
    (r'^Artesyn Technologies,?\s*(Inc\.)?$', 'Artesyn Technologies Inc.'),

    # Artos Technology
    (r'^Artos Technology,?\s*(LLC)?$', 'Artos Technology LLC'),

    # Artronix Technology
    (r'^Artronix Technology,?\s*(Inc\.)?$', 'Artronix Technology Inc.'),

    # Arysta Lifescience North America
    (r'^Arysta Lifescience North America,?\s*(LLC|Corp\.)$', 'Arysta Lifescience North America LLC'),

    # As Seen on TV
    (r'^As Seen on TV,?\s*(LLC|Inc\.)$', 'As Seen on TV LLC'),

    # Asahi Glass
    (r'^Asahi Glass\s*(Company,?\s*)?,?\s*Ltd\.$', 'Asahi Glass Co. Ltd.'),

    # Asahi Optical
    (r'^Asahi Optical\s*(Company,?\s*|Co\.?,?\s*)?Ltd\.$', 'Asahi Optical Co. Ltd.'),

    # Ashland
    (r'^Ashland,?\s*(Inc\.|LLC)$', 'Ashland Inc.'),

    # Asia Optical
    (r'^Asia Optical\s*(Company,?\s*|Co\.?,?\s*)?(Inc\.)?$', 'Asia Optical Co. Inc.'),

    # Asia Pacific Trading
    (r'^Asia Pacific Trading\s*Co\.?,?\s*(Inc\.)?$', 'Asia Pacific Trading Co. Inc.'),

    # Asia Vital Components
    (r'^Asia Vital Components\s*Co\.?,?\s*Ltd\.$', 'Asia Vital Components Co. Ltd.'),

    # Asoka USA
    (r'^Asoka USA,?\s*(Inc\.|Corp\.)$', 'Asoka USA Inc.'),

    # Ason Engineering
    (r'^Ason Engineering,?\s*(Inc\.|Ltd\.)$', 'Ason Engineering Inc.'),

    # Aspen Manufacturing
    (r'^Aspen Manufacturing,?\s*(Co\.|Inc\.)$', 'Aspen Manufacturing Co.'),

    # Aspen Medical Products
    (r'^Aspen Medical Products,?\s*(Inc\.)?$', 'Aspen Medical Products Inc.'),

    # Associated Equipment
    (r'^Associated Equipment,?\s*(Inc\.|Corp\.)$', 'Associated Equipment Inc.'),

    # Astellas Pharma US
    (r'^Astellas Pharma US,?\s*(Inc\.)?$', 'Astellas Pharma US Inc.'),

    # Aster Graphics
    (r'^Aster Graphics\s*(Company,?\s*|Co\.?,?\s*)?(Inc\.|Ltd\.)?$', 'Aster Graphics Co. Ltd.'),

    # AstraZeneca bare variants → already in main map
    (r'^AstraZeneca,?\s*(Inc\.|Ltd\.)$', 'AstraZeneca PLC'),

    # Atari
    (r'^Atari,?\s*(Inc\.|Corp\.)$', 'Atari Inc.'),

    # Atcoflex
    (r'^Atcoflex,?\s*(Inc\.|LLC)$', 'Atcoflex Inc.'),

    # Aten International
    (r'^Aten International\s*Co\.?,?\s*Ltd\.$', 'Aten International Co. Ltd.'),

    # Athena Cosmetics
    (r'^Athena Cosmetics,?\s*(Corp\.|Inc\.)$', 'Athena Cosmetics Corp.'),

    # Atico International / Atico International USA — keep separate
    (r'^Atico International,?\s*(Inc\.)?$', 'Atico International Inc.'),
    (r'^Atico International USA,?\s*(Inc\.)?$', 'Atico International USA Inc.'),

    # Atlanta Attachment
    (r'^Atlanta Attachment\s*(Company,?\s*)?,?\s*(Co\.|Inc\.)?$', 'Atlanta Attachment Co.'),

    # Atlanta Network Technologies
    (r'^Atlanta Network Technologies,?\s*(Inc\.)?$', 'Atlanta Network Technologies Inc.'),

    # Atlantic Richfield (ARCO)
    (r'^Atlantic Richfield\s*Co,?\s*(Inc\.)?$', 'Atlantic Richfield Co.'),
    (r'^Atlantic Richfield,?\s*Co\.$', 'Atlantic Richfield Co.'),

    # Atlantic Thermoplastics
    (r'^Atlantic Thermoplastics\s*(CO\.|Company)?,?\s*(Inc\.)?$', 'Atlantic Thermoplastics Co. Inc.'),

    # Atlantic Trading
    (r'^Atlantic Trading\s*Co\.?,?\s*(Inc\.)?$', 'Atlantic Trading Co. Inc.'),
    (r'^Atlantic Trading,?\s*Co\.$', 'Atlantic Trading Co. Inc.'),

    # Atlantis Enterprises
    (r'^Atlantis Enterprises,?\s*(Inc\.)?$', 'Atlantis Enterprises Inc.'),

    # Atsco Footwear
    (r'^Atsco Footwear,?\s*(Inc\.|LLC)$', 'Atsco Footwear Inc.'),

    # Attachment Technologies
    (r'^Attachment Technologies,?\s*(Inc\.)?$', 'Attachment Technologies Inc.'),

    # Audi of America — VW subsidiary
    (r'^Audi of America,?\s*(Inc\.|LLC)?$', 'Audi of America Inc.'),

    # Audience
    (r'^Audience,?\s*(Corp\.|Inc\.)$', 'Audience Corp.'),

    # Audio Communications
    (r'^Audio Communications,?\s*(Inc\.)?$', 'Audio Communications Inc.'),

    # Audiocodes
    (r'^Audiocodes,?\s*(Ltd\.|Inc\.)$', 'Audiocodes Ltd.'),

    # Auditory Licensing
    (r'^Auditory Licensing\s*(Company,?\s*|Co,?\s*)LLC$', 'Auditory Licensing Co. LLC'),

    # Augusta Bag
    (r'^Augusta Bag\s*(Company,?\s*)?,?\s*(Inc\.|Co\.)?$', 'Augusta Bag Co.'),

    # Aurobindo Pharma — catch Limited Inc variant
    (r'^Aurobindo Pharma\s*(Limited,?\s*)?,?\s*(Ltd\.|Inc\.)?$', 'Aurobindo Pharma Ltd.'),

    # Auspex Systems
    (r'^Auspex Systems,?\s*(Inc\.)?$', 'Auspex Systems Inc.'),

    # Austin Powder
    (r'^Austin Powder\s*(Company,?\s*)?,?\s*(Co\.|Inc\.)?$', 'Austin Powder Co.'),

    # Authentify Patent
    (r'^Authentify Patent\s*Co\.?,?\s*LLC$', 'Authentify Patent Co. LLC'),

    # Authorize.net
    (r'^Authorize\.net,?\s*(LLC|Corp\.)$', 'Authorize.net LLC'),

    # Auto Wax
    (r'^Auto Wax\s*(Company,?\s*|Co,?\s*|Co\.,?\s*)?(Inc\.)?$', 'Auto Wax Co. Inc.'),

    # Auto-Chlor System
    (r'^Auto-Chlor System,?\s*(Inc\.|LLC)$', 'Auto-Chlor System Inc.'),

    # AutoData Solutions
    (r'^AutoData Solutions,?\s*(Inc\.|Co\.)$', 'AutoData Solutions Inc.'),

    # Automated Business Machines
    (r'^Automated Business Machines,?\s*(LLC|Inc\.)$', 'Automated Business Machines LLC'),

    # Automatic Data Processing (ADP)
    (r'^Automatic Data Processing,?\s*(Inc\.|LLC)$', 'Automatic Data Processing Inc.'),

    # Automatic Equipment Manufacturing
    (r'^Automatic Equipment Manufacturing,?\s*(Co\.)?$', 'Automatic Equipment Manufacturing Co.'),

    # Automotive Carrier Services
    (r'^Automotive Carrier Services\s*Co\.?,?\s*LLC$', 'Automotive Carrier Services Co. LLC'),

    # Automotive Digital Systems
    (r'^Automotive Digital Systems,?\s*(Inc\.)?$', 'Automotive Digital Systems Inc.'),

    # Automotive Technologies International
    (r'^Automotive Technologies International,?\s*(Inc\.)?$', 'Automotive Technologies International Inc.'),

    # Availity — trailing space variant
    (r'^Availity\s*,?\s*LLC$', 'Availity LLC'),

    # Avanir Pharmaceuticals
    (r'^Avanir Pharmaceuticals,?\s*(Inc\.)?$', 'Avanir Pharmaceuticals Inc.'),

    # Avante International Technology
    (r'^Avante International Technology,?\s*(Inc\.|Corp\.)?$', 'Avante International Technology Inc.'),

    # Aventis CropScience USA Holding → Sanofi
    (r'^Aventis CropScience USA Holding,?\s*(Inc\.)?$', 'Sanofi S.A.'),

    # Avigilon USA
    (r'^Avigilon USA\s*(Corporation,?\s*)?,?\s*(Corp\.)?$', 'Avigilon USA Corp.'),

    # Avis Rent A Car System
    (r'^Avis Rent A Car System,?\s*(Inc\.|LLC)$', 'Avis Rent A Car System Inc.'),

    # Avista
    (r'^Avista,?\s*(Inc\.|Corp\.)$', 'Avista Corp.'),

    # Aviva Sports
    (r'^Aviva Sports,?\s*(Inc\.|LLC)$', 'Aviva Sports Inc.'),

    # Avtech
    (r'^Avtech,?\s*(Corp\.|Inc\.)$', 'Avtech Corp.'),

    # Axiom Memory Solutions
    (r'^Axiom Memory Solutions,?\s*(LLC)?$', 'Axiom Memory Solutions LLC'),

    # Axiom Process
    (r'^Axiom Process,?\s*(LLC|Ltd\.)$', 'Axiom Process LLC'),

    # Axis International Marketing
    (r'^Axis International Marketing\s*(Ltd\.?,?\s*)?(Inc\.)?$', 'Axis International Marketing Ltd.'),

    # Axis Systems
    (r'^Axis Systems,?\s*(Inc\.)?$', 'Axis Systems Inc.'),

    # Axonn
    (r'^Axonn,?\s*(LLC|Corp\.)$', 'Axonn LLC'),

    # B & B Homes
    (r'^B & B Homes,?\s*(Corp\.|Inc\.)$', 'B & B Homes Corp.'),

    # B & H Manufacturing
    (r'^B & H Manufacturing\s*(Company,?\s*)?,?\s*(Inc\.)?$', 'B & H Manufacturing Inc.'),

    # B and E Sales
    (r'^B and E Sales\s*(Company,?\s*|Co\.,?\s*)?(Inc\.)?$', 'B and E Sales Co. Inc.'),

    # B and S Plastics
    (r'^B and S Plastics,?\s*(Inc\.)?$', 'B and S Plastics Inc.'),

    # BA Merchant Services
    (r'^BA Merchant Services,?\s*(Inc\.|LLC)$', 'BA Merchant Services Inc.'),

    # BAK Industries
    (r'^BAK Industries,?\s*(Inc\.)?$', 'BAK Industries Inc.'),

    # BBC Distribution
    (r'^BBC Distribution,?\s*(LLC|Corp\.)$', 'BBC Distribution LLC'),

    # BBC International
    (r'^BBC International,?\s*(Ltd\.|LLC)$', 'BBC International Ltd.'),

    # BCBG Max Azria Group
    (r'^BCBG Max Azria Group,?\s*(Inc\.|LLC)$', 'BCBG Max Azria Group Inc.'),

    # BCE Emergis Technologies
    (r'^BCE Emergis Technologies,?\s*(Inc\.)?$', 'BCE Emergis Technologies Inc.'),

    # BCI Burke
    (r'^BCI Burke\s*(Company,?\s*)?,?\s*(Inc\.|LLC)$', 'BCI Burke Co.'),

    # BCNY International
    (r'^BCNY International,?\s*(Inc\.)?$', 'BCNY International Inc.'),

    # BEC Technologies
    (r'^BEC Technologies,?\s*(Inc\.)?$', 'BEC Technologies Inc.'),

    # BG Medical Products
    (r'^BG Medical Products,?\s*(LLC)?$', 'BG Medical Products LLC'),

    # BG Tech
    (r'^BG Tech\s*(Co\.?,?\s*Ltd\.|,?\s*Inc\.)?$', 'BG Tech Co. Ltd.'),

    # BGC Partners
    (r'^BGC Partners,?\s*(Inc\.|L\.P\.)$', 'BGC Partners Inc.'),

    # BHK of America
    (r'^BHK of America,?\s*(Inc\.)?$', 'BHK of America Inc.'),

    # BMW Manufacturing — BMW AG subsidiary
    (r'^BMW Manufacturing\s*(Company,?\s*|Co\.,?\s*)?,?\s*(Corp\.|LLC)$', 'BMW AG'),

    # BMW of North America — BMW AG subsidiary
    (r'^BMW of North America,?\s*(Inc\.)?$', 'BMW AG'),

    # BNC Nutrition
    (r'^BNC Nutrition,?\s*(LLC)?$', 'BNC Nutrition LLC'),

    # BOC Group
    (r'^BOC Group,?\s*(Inc\.)?$', 'BOC Group Inc.'),

    # BRK Electronics
    (r'^BRK Electronics,?\s*(Inc\.)?$', 'BRK Electronics Inc.'),

    # BS&B Safety Systems
    (r'^BS&B Safety Systems,?\s*(Inc\.|LLC)$', 'BS&B Safety Systems Inc.'),

    # BSN Sports
    (r'^BSN Sports,?\s*(Inc\.|LLC)$', 'BSN Sports Inc.'),

    # BTG International
    (r'^BTG International,?\s*(Ltd\.|Inc\.)$', 'BTG International Ltd.'),

    # Baby Dayz
    (r'^Baby Dayz\s*Co\.?,?\s*(Inc\.)?$', 'Baby Dayz Co. Inc.'),

    # Baby Jogger
    (r'^Baby Jogger,?\s*(Co\.|LLC)$', 'Baby Jogger Co.'),

    # Back to Basics Products
    (r'^Back to Basics Products,?\s*(Inc\.|LLC)?$', 'Back to Basics Products Inc.'),

    # Backflip Studios
    (r'^Backflip Studios,?\s*(LLC|Inc\.)$', 'Backflip Studios LLC'),

    # Bafo Technologies
    (r'^Bafo Technologies,?\s*(Corp\.)?$', 'Bafo Technologies Corp.'),

    # Baker & Taylor
    (r'^Baker & Taylor,?\s*(Inc\.|LLC)$', 'Baker & Taylor Inc.'),

    # Bal Seal Engineering
    (r'^Bal Seal Engineering\s*(Company,?\s*)?,?\s*(Inc\.)?$', 'Bal Seal Engineering Inc.'),

    # Balboa Manufacturing
    (r'^Balboa Manufacturing\s*(Company,?\s*)?,?\s*(LLC|Co\.)?$', 'Balboa Manufacturing Co.'),

    # Balboa Water Group
    (r'^Balboa Water Group,?\s*(Inc\.|LLC)$', 'Balboa Water Group Inc.'),

    # Baldor Electric
    (r'^Baldor Electric\s*(Company,?\s*|Co\.?,?\s*)?(Inc\.)?$', 'Baldor Electric Co.'),

    # Baldwin Technology
    (r'^Baldwin Technology\s*(Company,?\s*)?,?\s*(Corp\.)?$', 'Baldwin Technology Corp.'),

    # Bali
    (r'^Bali\s*(Company,?\s*)?,?\s*(Inc\.|Co\.)?$', 'Bali Co.'),

    # Ballard Medical Products
    (r'^Ballard Medical Products,?\s*(Inc\.)?$', 'Ballard Medical Products Inc.'),

    # Ballweg Chevrolet
    (r'^Ballweg Chevrolet,?\s*(Inc\.)?$', 'Ballweg Chevrolet Inc.'),

    # Bally Gaming International
    (r'^Bally Gaming International,?\s*(Inc\.)?$', 'Bally Gaming International Inc.'),

    # Bally Technologies
    (r'^Bally Technologies,?\s*(Inc\.)?$', 'Bally Technologies Inc.'),

    # Baltimore Technologies
    (r'^Baltimore Technologies,?\s*(Ltd\.|Inc\.)$', 'Baltimore Technologies Ltd.'),

    # Banana Republic — Gap subsidiary
    (r'^Banana Republic,?\s*(Inc\.|LLC)$', 'Banana Republic LLC'),

    # Banco Popular North America
    (r'^Banco Popular North America,?\s*(Inc\.)?$', 'Banco Popular North America Inc.'),

    # Bank Of America / Bank of America — same entity, normalize case
    (r'^Bank [Oo]f America,?\s*(Corp\.)?$', 'Bank of America Corp.'),

    # Bank One — acquired by JPMorgan Chase
    (r'^Bank One,?\s*(Corp\.)?$', 'JPMorgan Chase & Co.'),

    # Bank Sinopac
    (r'^Bank Sinopac\s*(Company,?\s*)?,?\s*(Ltd\.)?$', 'Bank Sinopac Co. Ltd.'),

    # Bank and Estate Liquidators
    (r'^Bank and Estate Liquidators,?\s*(Inc\.)?$', 'Bank and Estate Liquidators Inc.'),

    # Bank of the Ozarks
    (r'^Bank of the Ozarks,?\s*(Inc\.)?$', 'Bank of the Ozarks Inc.'),

    # Bank of the West
    (r'^Bank of the West,?\s*(Corp\.)?$', 'Bank of the West Corp.'),

    # Bard Access Systems — C.R. Bard subsidiary
    (r'^Bard Access Systems,?\s*(Inc\.)?$', 'Bard Access Systems Inc.'),

    # Barlow Marine USA
    (r'^Barlow Marine USA,?\s*(Inc\.)?$', 'Barlow Marine USA Inc.'),

    # Barnes & Noble College Booksellers
    (r'^Barnes & Noble College Booksellers,?\s*(Inc\.|LLC)$', 'Barnes & Noble College Booksellers Inc.'),

    # BarnesandNoble.com / Barnesandnoble.Com / Barnesandnoble.com — all same
    (r'^[Bb]arnesandnoble\.[Cc]om,?\s*(Inc\.|LLC)$', 'BarnesandNoble.com LLC'),
    (r'^BarnesandNoble\.com,?\s*(Inc\.|LLC)$', 'BarnesandNoble.com LLC'),

    # Barnett Int'l / Barnett International — same company
    (r"^Barnett Int'?l?,?\s*(Ltd\.|Inc\.)$", 'Barnett International Ltd.'),
    (r'^Barnett International,?\s*(Ltd\.|Inc\.)$', 'Barnett International Ltd.'),

    # Baroid
    (r'^Baroid,?\s*(Ltd\.|Corp\.)$', 'Baroid Corp.'),

    # Barrday
    (r'^Barrday,?\s*(Inc\.|Corp\.)$', 'Barrday Inc.'),

    # Barrie Archery
    (r'^Barrie Archery,?\s*(LLC)?$', 'Barrie Archery LLC'),

    # Bass Pro / Bass Pro Outdoors Online — keep separate
    (r'^Bass Pro,?\s*(Inc\.|LLC)$', 'Bass Pro Inc.'),
    (r'^Bass Pro Outdoors Online,?\s*(LLC)?$', 'Bass Pro Outdoors Online LLC'),

    # Bath & Body Works — L Brands subsidiary
    (r'^Bath & Body Works,?\s*(Inc\.|LLC)$', 'Bath & Body Works LLC'),

    # Baxter International — catch bare variant
    (r'^Baxter International,?\s*(Inc\.)?$', 'Baxter International Inc.'),

    # Bay Medical
    (r'^Bay Medical\s*(Company,?\s*|Co\.?,?\s*)?(Inc\.)?$', 'Bay Medical Co. Inc.'),

    # Bayco Products
    (r'^Bayco Products,?\s*(Inc\.|Ltd\.)$', 'Bayco Products Inc.'),

    # Bayer Cropscience — Bayer AG subsidiary
    (r'^Bayer Cropscience,?\s*(Inc\.|LLC)$', 'Bayer AG'),

    # Beacon Power
    (r'^Beacon Power,?\s*(Corp\.|Inc\.|LLC)$', 'Beacon Power Corp.'),

    # Bear Archery
    (r'^Bear Archery,?\s*(Inc\.|Co\.)$', 'Bear Archery Inc.'),

    # Bear Stearns — acquired by JPMorgan Chase
    (r'^Bear Stearns\s*&,?\s*Co\.?,?\s*(Inc\.)?$', 'JPMorgan Chase & Co.'),

    # Beauty Maid Products
    (r'^Beauty Maid Products,?\s*(Ltd\.|Inc\.)$', 'Beauty Maid Products Ltd.'),

    # Becton Dickinson Infusion Therapy Systems — BD subsidiary
    (r'^Becton Dickinson Infusion Therapy Systems,?\s*(Inc\.)?$', 'Becton Dickinson and Co.'),

    # Bedford Laboratories — Boehringer Ingelheim subsidiary
    (r'^Bedford Laboratories,?\s*(Inc\.)?$', 'Bedford Laboratories Inc.'),

    # Bel Air Lighting
    (r'^Bel Air Lighting,?\s*(Inc\.)?$', 'Bel Air Lighting Inc.'),

    # Bel Fuse
    (r'^Bel Fuse,?\s*(Inc\.|Ltd\.)$', 'Bel Fuse Inc.'),

    # Belair Networks
    (r'^Belair Networks,?\s*(Inc\.|Corp\.)$', 'Belair Networks Inc.'),

    # Belcher Pharmaceuticals
    (r'^Belcher Pharmaceuticals,?\s*(Inc\.|LLC)$', 'Belcher Pharmaceuticals Inc.'),

    # Belden Technologies
    (r'^Belden Technologies,?\s*(Inc\.|LLC)$', 'Belden Technologies Inc.'),

    # Belkin / Belkin International — same company
    (r'^Belkin,?\s*(Corp\.|Inc\.|Ltd\.)$', 'Belkin International Inc.'),
    (r'^Belkin International\s*,?\s*(Inc\.)?$', 'Belkin International Inc.'),

    # Bell & Howell Mail Processing Systems
    (r'^Bell & Howell Mail Processing Systems,?\s*(Co\.)?$', 'Bell & Howell Mail Processing Systems Co.'),

    # Bell Atlantic Mobile — became Verizon Wireless
    (r'^Bell Atlantic Mobile,?\s*(Inc\.)?$', 'Verizon Communications'),

    # BellSouth Telecommunications — now AT&T
    (r'^BellSouth Telecommunications,?\s*(Inc\.|LLC)$', 'AT&T Inc.'),

    # Bemis Manufacturing
    (r'^Bemis Manufacturing\s*(Company,?\s*)?,?\s*(Co\.|Inc\.)?$', 'Bemis Manufacturing Co.'),

    # Ben Venue Laboratories
    (r'^Ben Venue Laboratories,?\s*(Inc\.)?$', 'Ben Venue Laboratories Inc.'),

    # Benchmade Knife
    (r'^Benchmade Knife\s*(Company,?\s*|Co\.?,?\s*)?(Inc\.)?$', 'Benchmade Knife Co. Inc.'),
]

DIRECT_MAP_2 = {
    # AstraZeneca bare variants
    'AstraZeneca, Inc.':    'AstraZeneca PLC',
    'AstraZeneca, Ltd.':    'AstraZeneca PLC',

    # Bank One → JPMorgan Chase
    'Bank One, Corp.':      'JPMorgan Chase & Co.',
    'Bank One':             'JPMorgan Chase & Co.',

    # Bear Stearns → JPMorgan Chase
    'Bear Stearns & Co., Inc.': 'JPMorgan Chase & Co.',
    'Bear Stearns &, Co.':      'JPMorgan Chase & Co.',

    # Bell Atlantic Mobile → Verizon
    'Bell Atlantic Mobile':      'Verizon Communications',
    'Bell Atlantic Mobile, Inc.': 'Verizon Communications',

    # BellSouth → AT&T
    'BellSouth Telecommunications, Inc.': 'AT&T Inc.',
    'BellSouth Telecommunications, LLC':  'AT&T Inc.',

    # BMW Manufacturing/North America → BMW AG
    'BMW Manufacturing, Corp.':        'BMW AG',
    'BMW Manufacturing Co., LLC':       'BMW AG',
    'BMW Manufacturing Company, LLC':   'BMW AG',
    'BMW of North America, Inc.':       'BMW AG',
    'BMW of North America':             'BMW AG',

    # Aventis CropScience → Sanofi
    'Aventis CropScience USA Holding':       'Sanofi S.A.',
    'Aventis CropScience USA Holding, Inc.': 'Sanofi S.A.',

    # Bayer Cropscience → Bayer AG
    'Bayer Cropscience, Inc.': 'Bayer AG',
    'Bayer Cropscience, LLC':  'Bayer AG',

    # Becton Dickinson subsidiaries
    'Becton Dickinson Infusion Therapy Systems':       'Becton Dickinson and Co.',
    'Becton Dickinson Infusion Therapy Systems, Inc.': 'Becton Dickinson and Co.',

    # BarnesandNoble.com variants
    'BarnesandNoble.com, Inc.':    'BarnesandNoble.com LLC',
    'BarnesandNoble.com, LLC':     'BarnesandNoble.com LLC',
    'Barnesandnoble.Com, Inc.':    'BarnesandNoble.com LLC',
    'Barnesandnoble.Com, LLC':     'BarnesandNoble.com LLC',
    'Barnesandnoble.com, Inc.':    'BarnesandNoble.com LLC',
    'Barnesandnoble.com, LLC':     'BarnesandNoble.com LLC',

    # Barnett Int'l variants
    "Barnett Int'l, Ltd.": 'Barnett International Ltd.',
    "Barnett Int'l, Inc.": 'Barnett International Ltd.',

    'CBS, Inc.':'CBS Corporation',
    'CBS, Corp.':'CBS Corporation',
}

def apply_similar_groups_map_2(name: str) -> str:
    """Apply batch 2 regex rules then direct map."""
    if not isinstance(name, str):
        return name

    if name in DIRECT_MAP_2:
        return DIRECT_MAP_2[name]

    for pattern, replacement in REGEX_RULES_2:
        if re.match(pattern, name, flags=re.IGNORECASE):
            return replacement

    return name

names['standardized_org_name'] = names['standardized_org_name'].apply(apply_similar_groups_map_2)
# names['standardized_org_name'] = names['standardized_org_name'].apply(apply_similar_groups_map_2)

In [ ]:
import re

# Similar groups map - Batch 3 (B-C entries)

REGEX_RULES_3 = [

    # ARM Ltd — catch bare variant
    (r'^ARM,?\s*(Ltd\.)?$', 'ARM Ltd.'),

    # Advance Metalworking — already normalized, catch remaining
    (r'^Advance Metalworking\s*(Co\.?\s*Inc\.|,?\s*Inc\.)$', 'Advance Metalworking Co. Inc.'),

    # Advanced Technology — too generic, skip mapping

    # Allen — remaining variant after batch 2
    (r'^Allen,?\s*Co\.?\s*(Inc\.)?$', 'Allen Co. Inc.'),

    # American Electronic Sign — remaining
    (r'^American Electronic Sign,?\s*Co\.$', 'American Electronic Sign Co. Inc.'),

    # Application Art Laboratories — catch remaining
    (r'^Application Art Laboratories,?\s*Co\.,?\s*Ltd\.$', 'Application Art Laboratories Co. Ltd.'),

    # Art Leather Manufacturing — remaining
    (r'^Art Leather Manufacturing,?\s*Inc\.$', 'Art Leather Manufacturing Co. Inc.'),

    # Asahi Glass — remaining
    (r'^Asahi Glass\s*Co,?\s*Ltd\.$', 'Asahi Glass Co. Ltd.'),

    # Aster Graphics — catch Inc vs Ltd
    (r'^Aster Graphics,?\s*(Inc\.|Co\.\s*Ltd\.)$', 'Aster Graphics Co. Ltd.'),

    # Auto Wax — remaining variants
    (r'^Auto Wax,?\s*(Inc\.|Co\.,?\s*Inc\.)$', 'Auto Wax Co. Inc.'),

    # Avigilon USA — catch Corporation variant
    (r'^Avigilon USA\s*(Corporation,?\s*)?,?\s*(Corp\.)?$', 'Avigilon USA Corp.'),

    # Axis International Marketing — remaining
    (r'^Axis International Marketing,?\s*Ltd\.$', 'Axis International Marketing Ltd.'),

    # Baldor Electric — remaining
    (r'^Baldor Electric,?\s*Co\.$', 'Baldor Electric Co.'),

    # Baldwin Technology — remaining
    (r'^Baldwin Technology\s*(Company,?\s*)?,?\s*(Corp\.)?$', 'Baldwin Technology Corp.'),

    # Bendix Commercial Vehicle Systems
    (r'^Bendix Commercial Vehicle Systems,?\s*(LLC)?$', 'Bendix Commercial Vehicle Systems LLC'),

    # Bennington Golf
    (r'^Bennington Golf\s*(Company,?\s*)?,?\s*(Inc\.|Co\.)?$', 'Bennington Golf Co.'),

    # BenQ / BenQ America — keep separate
    (r'^Benq,?\s*(Corp\.|Inc\.)$', 'BenQ Corp.'),
    (r'^Benq America,?\s*(Corp\.)?$', 'BenQ America Corp.'),

    # Bentley Motors
    (r'^Bentley Motors,?\s*(Inc\.|Ltd\.)$', 'Bentley Motors Ltd.'),

    # Bentz Metal Products
    (r'^Bentz Metal Products\s*(Co,?\s*|Company,?\s*)?(Inc\.)?$', 'Bentz Metal Products Co. Inc.'),

    # Berlex Laboratories — Bayer subsidiary
    (r'^Berlex Laboratories,?\s*(Inc\.)?$', 'Berlex Laboratories Inc.'),

    # Berlin Industries
    (r'^Berlin Industries,?\s*(Inc\.)?$', 'Berlin Industries Inc.'),

    # Bernafon
    (r'^Bernafon,?\s*(Inc\.|LLC)$', 'Bernafon Inc.'),

    # Bernhardt Furniture
    (r'^Bernhardt Furniture\s*(Company,?\s*)?,?\s*(Co\.)?$', 'Bernhardt Furniture Co.'),

    # Best Buy — already in main map, catch remaining variant
    (r'^Best Buy,?\s*(Inc\.)$', 'Best Buy Co.'),

    # Best Manufacturing
    (r'^Best Manufacturing,?\s*(Corp\.|Co\.)$', 'Best Manufacturing Corp.'),

    # Best Products
    (r'^Best Products\s*(Co,?\s*|Company,?\s*)?(Inc\.)?$', 'Best Products Co. Inc.'),

    # Big Baby
    (r'^Big Baby,?\s*(Co\.|Inc\.)$', 'Big Baby Co.'),

    # Binks Manufacturing
    (r'^Binks Manufacturing,?\s*(Co\.|Corp\.)$', 'Binks Manufacturing Co.'),

    # Bio-Rad Laboratories
    (r'^Bio-Rad Laboratories,?\s*(Inc\.)?$', 'Bio-Rad Laboratories Inc.'),

    # Biocell Technology
    (r'^Biocell Technology,?\s*(LLC)?$', 'Biocell Technology LLC'),

    # Biodex Medical Systems
    (r'^Biodex Medical Systems,?\s*(Inc\.)?$', 'Biodex Medical Systems Inc.'),

    # Biogenex Laboratories
    (r'^Biogenex Laboratories,?\s*(Inc\.)?$', 'Biogenex Laboratories Inc.'),

    # Biolase Technology
    (r'^Biolase Technology,?\s*(Inc\.)?$', 'Biolase Technology Inc.'),

    # Biomet Biologics / Biomet Orthopedics — Biomet subsidiaries, keep separate
    (r'^Biomet Biologics,?\s*(LLC|Inc\.)$', 'Biomet Biologics LLC'),
    (r'^Biomet Orthopedics,?\s*(Inc\.|LLC)$', 'Biomet Orthopedics Inc.'),

    # Bionaire
    (r'^Bionaire,?\s*(Inc\.|Corp\.)$', 'Bionaire Inc.'),

    # Bionix Medical Technologies
    (r'^Bionix Medical Technologies,?\s*(LLC)?$', 'Bionix Medical Technologies LLC'),

    # Biorobotics
    (r'^Biorobotics,?\s*(Ltd\.|Inc\.)$', 'Biorobotics Ltd.'),

    # Biosuccess Biotech
    (r'^Biosuccess Biotech\s*Co\.?,?\s*Ltd\.$', 'Biosuccess Biotech Co. Ltd.'),

    # Biotage
    (r'^Biotage,?\s*(LLC|Inc\.)$', 'Biotage LLC'),

    # Biotal
    (r'^Biotal,?\s*(Ltd\.|Inc\.)$', 'Biotal Ltd.'),

    # Bird Machine
    (r'^Bird Machine\s*(Co\.?,?\s*)?(Inc\.)?$', 'Bird Machine Co. Inc.'),

    # Birtcher Medical Systems
    (r'^Birtcher Medical Systems,?\s*(Inc\.)?$', 'Birtcher Medical Systems Inc.'),

    # Black Diamond Equipment
    (r'^Black Diamond Equipment\s*(Ltd\.?,?\s*)?(Inc\.)?$', 'Black Diamond Equipment Ltd.'),

    # Blackbird Technologies
    (r'^Blackbird Technologies,?\s*(Inc\.)?$', 'Blackbird Technologies Inc.'),

    # Blackhawk Molding
    (r'^Blackhawk Molding\s*(Company,?\s*|Co\.?,?\s*)?(Inc\.)?$', 'Blackhawk Molding Co. Inc.'),

    # Blank Acquisition
    (r'^Blank Acquisition,?\s*(LLC|Corp\.)$', 'Blank Acquisition LLC'),

    # Blastrac Global
    (r'^Blastrac Global,?\s*(LLC|Inc\.)$', 'Blastrac Global LLC'),

    # Blaze Construction
    (r'^Blaze Construction\s*(Company,?\s*)?,?\s*(Inc\.)?$', 'Blaze Construction Inc.'),

    # Blizzard Industrial Supply
    (r'^Blizzard Industrial Supply,?\s*(Inc\.|Co\.)$', 'Blizzard Industrial Supply Inc.'),

    # Block Drug
    (r'^Block Drug\s*(Company,?\s*)?,?\s*(Inc\.|Corp\.)?$', 'Block Drug Co.'),

    # Block Financial
    (r'^Block Financial,?\s*(Corp\.|LLC)$', 'Block Financial Corp.'),

    # Blockbuster
    (r'^Blockbuster,?\s*(Inc\.|LLC)$', 'Blockbuster Inc.'),

    # Bloomberg
    (r'^Bloomberg,?\s*(L\.P\.|Inc\.)$', 'Bloomberg L.P.'),

    # Bloxx
    (r'^Bloxx,?\s*(Inc\.|Ltd\.)$', 'Bloxx Inc.'),

    # Blue Calypso
    (r'^Blue Calypso,?\s*(Inc\.|LLC)$', 'Blue Calypso Inc.'),

    # Blue Pumpkin Software
    (r'^Blue Pumpkin Software,?\s*(Inc\.|LLC)$', 'Blue Pumpkin Software Inc.'),

    # Blue Rhino Global Sourcing
    (r'^Blue Rhino Global Sourcing,?\s*(LLC|Inc\.)$', 'Blue Rhino Global Sourcing LLC'),

    # Blue Sky Network
    (r'^Blue Sky Network,?\s*(LLC|Inc\.)$', 'Blue Sky Network LLC'),

    # Blue Spike — NPE, keep as-is but unify
    (r'^Blue Spike,?\s*(Co\.|Inc\.|LLC)$', 'Blue Spike LLC'),

    # Bluelight.com
    (r'^Bluelight\.com,?\s*(Inc\.|LLC)$', 'Bluelight.com Inc.'),

    # BMG Music
    (r'^Bmg Music,?\s*(Inc\.)?$', 'BMG Music Inc.'),

    # Board of Trade of the City of Chicago
    (r'^Board of Trade of the City of Chicago,?\s*(Inc\.)?$', 'Board of Trade of the City of Chicago Inc.'),

    # Bodyguard Fitness
    (r'^Bodyguard Fitness,?\s*(Corp\.)?$', 'Bodyguard Fitness Corp.'),

    # Bombardier
    (r'^Bombardier,?\s*(Corp\.|Inc\.)$', 'Bombardier Inc.'),

    # Bon Jour International
    (r'^Bon Jour International,?\s*Ltd\.?,?\s*(Inc\.)?$', 'Bon Jour International Ltd.'),

    # Bonneville Artemia International
    (r'^Bonneville Artemia International,?\s*(Inc\.)?$', 'Bonneville Artemia International Inc.'),

    # Boone Supply
    (r'^Boone Supply\s*(Company,?\s*)?,?\s*(Co\.|Inc\.)?$', 'Boone Supply Co.'),

    # Boost Mobile
    (r'^Boost Mobile,?\s*(LLC)?$', 'Boost Mobile LLC'),

    # Boral Material Technologies
    (r'^Boral Material Technologies,?\s*(Inc\.|LLC)$', 'Boral Material Technologies Inc.'),

    # Borders Direct
    (r'^Borders Direct,?\s*(LLC|Inc\.)$', 'Borders Direct LLC'),

    # Borg-Warner — now BorgWarner, already handled in main map
    (r'^Borg-Warner,?\s*(Corp\.|Inc\.)$', 'Stanley Black & Decker Inc.'),

    # Bosch Rexroth — Robert Bosch subsidiary
    (r'^Bosch Rexroth,?\s*(Inc\.|Corp\.)$', 'Robert Bosch GmbH'),

    # Boss Pet Products / Boss Products — different companies, keep separate
    (r'^Boss Pet Products,?\s*(Inc\.)?$', 'Boss Pet Products Inc.'),
    (r'^Boss Products,?\s*(Corp\.)?$', 'Boss Products Corp.'),

    # Boston Scientific — catch Ltd variant
    (r'^Boston Scientific,?\s*(Corp\.|Ltd\.)$', 'Boston Scientific Corp.'),

    # Bouldin & Lawson
    (r'^Bouldin & Lawson,?\s*(Inc\.|LLC)$', 'Bouldin & Lawson Inc.'),

    # Brachysciences
    (r'^Brachysciences,?\s*(Inc\.)?$', 'Brachysciences Inc.'),

    # Bradshaw International — trailing space variant
    (r'^Bradshaw International\s*,?\s*(Inc\.)?$', 'Bradshaw International Inc.'),

    # Brady Environmental
    (r'^Brady Environmental,?\s*(LLC|Inc\.)$', 'Brady Environmental LLC'),

    # Bragel International
    (r'^Bragel International,?\s*(Inc\.)?$', 'Bragel International Inc.'),

    # Branch Banking and Trust
    (r'^Branch Banking and Trust,?\s*(Co\.|Corp\.)$', 'Branch Banking and Trust Co.'),

    # Brandsmart U.S.A.
    (r'^Brandsmart U\.S\.A\.,?\s*(Inc\.)?$', 'Brandsmart U.S.A. Inc.'),

    # Brandywine Communications Technologies — NPE
    (r'^Brandywine Communications Technologies,?\s*(LLC)?$', 'Brandywine Communications Technologies LLC'),

    # Brass Eagle
    (r'^Brass Eagle,?\s*(Inc\.|LLC)$', 'Brass Eagle Inc.'),

    # Brass Smith
    (r'^Brass Smith,?\s*(Inc\.|LLC)$', 'Brass Smith Inc.'),

    # Brassica Protection Products
    (r'^Brassica Protection Products,?\s*(LLC)?$', 'Brassica Protection Products LLC'),

    # Braun
    (r'^Braun,?\s*(Inc\.|Corp\.)$', 'Braun Inc.'),

    # Bray Group
    (r'^Bray Group,?\s*(Inc\.)?$', 'Bray Group Inc.'),

    # Breckenridge Pharmaceutical — already in main map
    (r'^Breckenridge Pharmaceutical,?\s*(Inc\.)?$', 'Breckenridge Pharmaceutical Inc.'),

    # Bridgestone Sports
    (r'^Bridgestone Sports\s*Co\.?,?\s*Ltd\.$', 'Bridgestone Sports Co. Ltd.'),

    # Bridgewater
    (r'^Bridgewater,?\s*(Inc\.|LLC)$', 'Bridgewater Inc.'),

    # Bright Solutions
    (r'^Bright Solutions,?\s*(Inc\.|Corp\.)$', 'Bright Solutions Inc.'),

    # Brilliant Optical Solutions
    (r'^Brilliant Optical Solutions,?\s*(LLC)?$', 'Brilliant Optical Solutions LLC'),

    # Brine
    (r'^Brine,?\s*(Inc\.|Corp\.)$', 'Brine Inc.'),

    # Brother Industries / Brother International — same parent
    (r'^Brother Industries,?\s*(Ltd\.)?$', 'Brother Industries Ltd.'),
    (r'^Brother International,?\s*(Inc\.)?$', 'Brother Industries Ltd.'),

    # Brown Group Retail
    (r'^Brown Group Retail,?\s*(Inc\.)?$', 'Brown Group Retail Inc.'),

    # Brown Shoe
    (r'^Brown Shoe\s*(Company,?\s*|Co\.?,?\s*)?(Inc\.)?$', 'Brown Shoe Co. Inc.'),

    # Browne & — keep as-is
    (r'^Browne\s*&,?\s*Co\.,?\s*(Ltd\.)?$', 'Browne & Co. Ltd.'),

    # Browning
    (r'^Browning,?\s*(Corp\.)?$', 'Browning Corp.'),

    # Brunswick
    (r'^Brunswick\s*,?\s*(Corp\.?,?\s*)?(Inc\.)?$', 'Brunswick Corp.'),

    # Brunton
    (r'^Brunton\s*(Company,?\s*)?,?\s*(Co\.|Inc\.)?$', 'Brunton Co.'),

    # Buck Knives / Buck Tool — different products, keep separate
    (r'^Buck Knives,?\s*(Inc\.)?$', 'Buck Knives Inc.'),
    (r'^Buck Tool,?\s*(Co\.|Inc\.)$', 'Buck Tool Co.'),

    # Buffalo Brake Beam
    (r'^Buffalo Brake Beam\s*(Co\.?,?\s*)?(Inc\.)?$', 'Buffalo Brake Beam Co. Inc.'),

    # Bula America
    (r'^Bula America,?\s*(Inc\.)?$', 'Bula America Inc.'),

    # Bulk Handling Systems
    (r'^Bulk Handling Systems,?\s*(Inc\.)?$', 'Bulk Handling Systems Inc.'),

    # Bullet Line
    (r'^Bullet Line,?\s*(Inc\.|LLC)$', 'Bullet Line Inc.'),

    # Bulova Watch
    (r'^Bulova Watch\s*(CO\.|Company)?,?\s*(Inc\.)?$', 'Bulova Watch Co. Inc.'),

    # Burlington Factory Warehouse
    (r'^Burlington Factory Warehouse,?\s*(Corp\.|Co\.)$', 'Burlington Factory Warehouse Corp.'),

    # Burlington Industries
    (r'^Burlington Industries,?\s*(Inc\.|LLC)?$', 'Burlington Industries Inc.'),

    # Burroughs
    (r'^Burroughs,?\s*(Corp\.|Inc\.)$', 'Burroughs Corp.'),

    # Bush Hog
    (r'^Bush Hog,?\s*(LLC|Inc\.)$', 'Bush Hog LLC'),

    # Bushnell
    (r'^Bushnell,?\s*(Corp\.|Inc\.)$', 'Bushnell Corp.'),

    # Butler Manufacturing
    (r'^Butler Manufacturing,?\s*(Co\.)?$', 'Butler Manufacturing Co.'),

    # Buyers Products
    (r'^Buyers Products\s*(Co,?\s*|Company,?\s*)?,?\s*(Inc\.)?$', 'Buyers Products Co. Inc.'),

    # C & F Packing / C&F Packing — same company
    (r'^C\s*&\s*F Packing\s*(Company,?\s*|Co,?\s*)?(Inc\.)?$', 'C&F Packing Co. Inc.'),

    # C&H Distributors
    (r'^C&H Distributors,?\s*(Inc\.|LLC)$', 'C&H Distributors Inc.'),

    # C&S Manufacturing
    (r'^C&S Manufacturing,?\s*(Corp\.|Co\.)?$', 'C&S Manufacturing Corp.'),

    # C-nario
    (r'^C-nario,?\s*(Ltd\.|Inc\.)$', 'C-nario Ltd.'),

    # C.C. Forbes
    (r'^C\.C\. Forbes,?\s*(LLC|Ltd\.)$', 'C.C. Forbes LLC'),

    # C.R. Laurence
    (r'^C\.R\. Laurence\s*(Co\.?,?\s*)?(Inc\.)?$', 'C.R. Laurence Co. Inc.'),

    # C.V. Starr
    (r'^C\.V\. Starr\s*&,?\s*Co\.,?\s*(Inc\.)?$', 'C.V. Starr & Co. Inc.'),

    # CA Technologies
    (r'^CA Technologies,?\s*(Inc\.)?$', 'CA Technologies Inc.'),

    # CAE Newnes
    (r'^CAE Newnes,?\s*(Inc\.|Ltd\.)$', 'CAE Newnes Inc.'),

    # CAL AM / Cal AM Manufacturing — same company, case variant
    (r'^CAL? AM Manufacturing\s*(Co\.?,?\s*)?(Inc\.)?$', 'Cal AM Manufacturing Co. Inc.'),

    # CAO Group
    (r'^CAO Group,?\s*(Inc\.)?$', 'CAO Group Inc.'),

    # CBK
    (r'^CBK,?\s*Ltd\.?,?\s*(Inc\.)?$', 'CBK Ltd.'),

    # CCC International
    (r'^CCC International\s*(Co,?\s*Ltd\.)?$', 'CCC International Co. Ltd.'),

    # CCL Industries
    (r'^CCL Industries,?\s*(Corp\.|Inc\.)$', 'CCL Industries Corp.'),

    # CCL Products Enterprises
    (r'^CCL Products Enterprises,?\s*(Inc\.)?$', 'CCL Products Enterprises Inc.'),

    # CDI International
    (r'^CDI International,?\s*(Inc\.)?$', 'CDI International Inc.'),

    # CDW
    (r'^CDW,?\s*(Corp\.|Inc\.)$', 'CDW Corp.'),

    # CEG Holdings
    (r'^CEG Holdings,?\s*(LLC|Inc\.)$', 'CEG Holdings LLC'),

    # CIA Wheel Group
    (r'^CIA Wheel Group,?\s*(Inc\.)?$', 'CIA Wheel Group Inc.'),

    # CIBA Specialty Chemicals
    (r'^CIBA Specialty Chemicals\s*(Corporation,?\s*)?,?\s*(Corp\.)?$', 'CIBA Specialty Chemicals Corp.'),

    # CKC International
    (r'^CKC International,?\s*(Inc\.|LLC)$', 'CKC International Inc.'),

    # CMS Technologies
    (r'^CMS Technologies,?\s*(Inc\.)?$', 'CMS Technologies Inc.'),

    # CNET Technology
    (r'^CNET Technology,?\s*(Corp\.|Inc\.)$', 'CNET Technology Corp.'),

    # CNI Manufacturing
    (r'^CNI Manufacturing,?\s*(Inc\.)?$', 'CNI Manufacturing Inc.'),

    # CPI Card Group / CPI Products — different companies, keep separate
    (r'^CPI Card Group,?\s*(Inc\.)?$', 'CPI Card Group Inc.'),
    (r'^CPI Products,?\s*(Inc\.)?$', 'CPI Products Inc.'),

    # CPN International
    (r'^CPN International,?\s*Ltd\.?,?\s*(Inc\.)?$', 'CPN International Ltd.'),

    # CSC Holdings
    (r'^CSC Holdings,?\s*(Inc\.|LLC)$', 'CSC Holdings Inc.'),

    # CSK Auto
    (r'^CSK Auto,?\s*(Inc\.|Corp\.)$', 'CSK Auto Inc.'),

    # CSN Stores
    (r'^CSN Stores,?\s*(LLC|Inc\.)$', 'CSN Stores LLC'),

    # CSX
    (r'^CSX\s*(?:Corporation,?\s*Inc\.|,?\s*Corp\.)?$', 'CSX Corp.'),

    # CVS
    (r'^CVS,?\s*(Inc\.|Corp\.)$', 'CVS Pharmacy Inc.'),

    # CY Technology Group
    (r'^CY Technology Group,?\s*(LLC)?$', 'CY Technology Group LLC'),

    # Cable Design Technologies
    (r'^Cable Design Technologies,?\s*(Corp\.|Inc\.)$', 'Cable Design Technologies Corp.'),

    # Cajun Bag & Supply
    (r'^Cajun Bag & Supply\s*(Co,?\s*|,?\s*)?(Inc\.|Corp\.)?$', 'Cajun Bag & Supply Co. Inc.'),

    # Caledonian Tree
    (r'^Caledonian Tree\s*(Company,?\s*)?,?\s*(Co\.|Ltd\.)?$', 'Caledonian Tree Co. Ltd.'),

    # Calgene
    (r'^Calgene,?\s*(Inc\.|LLC)$', 'Calgene Inc.'),

    # Calibre International
    (r'^Calibre International,?\s*(LLC)?$', 'Calibre International LLC'),

    # California Exotic Novelties
    (r'^California Exotic Novelties,?\s*(Inc\.|LLC)$', 'California Exotic Novelties Inc.'),

    # California Lighting Sales
    (r'^California Lighting Sales,?\s*(Inc\.)?$', 'California Lighting Sales Inc.'),

    # California Micro Devices
    (r'^California Micro Devices,?\s*(Corp\.|Inc\.)$', 'California Micro Devices Corp.'),

    # CallManage
    (r'^CallManage,?\s*(Inc\.|Ltd\.)$', 'CallManage Inc.'),

    # Callaway Golf — catch Company variant
    (r'^Callaway Golf\s*(Company,?\s*Inc\.|,?\s*Co\.)?$', 'Callaway Golf Co.'),

    # Cambridge Associates
    (r'^Cambridge Associates,?\s*(Inc\.|LLC)$', 'Cambridge Associates LLC'),

    # Cambridge Silversmiths
    (r'^Cambridge Silversmiths\s*Ltd\.?,?\s*(Inc\.)?$', 'Cambridge Silversmiths Ltd. Inc.'),

    # Cameco Industries
    (r'^Cameco Industries,?\s*(Inc\.)?$', 'Cameco Industries Inc.'),

    # Cameron
    (r'^Cameron,?\s*(Inc\.|Corp\.)$', 'Cameron Inc.'),

    # Camp Kazoo
    (r'^Camp Kazoo,?\s*(Ltd\.|Inc\.)$', 'Camp Kazoo Ltd.'),

    # Campbell Manufacturing
    (r'^Campbell Manufacturing,?\s*(Co\.|Inc\.)$', 'Campbell Manufacturing Co.'),

    # Canadian Pacific Railway
    (r'^Canadian Pacific Railway,?\s*(Co\.|Ltd\.)$', 'Canadian Pacific Railway Ltd.'),

    # Cancer Treatment Centers of America
    (r'^Cancer Treatment Centers of America,?\s*(Inc\.)?$', 'Cancer Treatment Centers of America Inc.'),

    # Candle Corporation of America
    (r'^Candle Corporation of America,?\s*(Inc\.)?$', 'Candle Corporation of America Inc.'),

    # Cannon U.S.A.
    (r'^Cannon U\.S\.A\.,?\s*(Inc\.)?$', 'Cannon U.S.A. Inc.'),

    # Cannondale
    (r'^Cannondale\s*(Corp\.?,?\s*)?(Inc\.)?$', 'Cannondale Corp.'),

    # Capewell Components
    (r'^Capewell Components\s*(Company,?\s*|Co,?\s*)?,?\s*(L\.P\.|LLC)?$', 'Capewell Components Co. LLC'),

    # Capital One Bank / Capital One Services — keep separate
    (r'^Capital One Bank,?\s*(Inc\.)?$', 'Capital One Bank Inc.'),
    (r'^Capital One Services,?\s*(LLC|Inc\.)$', 'Capital One Services LLC'),

    # Capitol Records
    (r'^Capitol Records,?\s*(Inc\.|LLC)$', 'Capitol Records LLC'),

    # Capro
    (r'^Capro,?\s*(Inc\.|Ltd\.)$', 'Capro Inc.'),

    # Car-Mic Enterprises
    (r'^Car-Mic Enterprises,?\s*(Inc\.)?$', 'Car-Mic Enterprises Inc.'),

    # Caraco Pharmaceutical Laboratories
    (r'^Caraco Pharmaceutical Laboratories,?\s*(Ltd\.|Inc\.)$', 'Caraco Pharmaceutical Laboratories Ltd.'),

    # Carborundum
    (r'^Carborundum,?\s*(Co\.|Corp\.)$', 'Carborundum Co.'),

    # Card-Monroe — keep separate from Card (too generic)
    (r'^Card-Monroe,?\s*(Corp\.|Co\.)$', 'Card-Monroe Corp.'),

    # Cardinal Cartridge
    (r'^Cardinal Cartridge,?\s*(Inc\.|LLC)$', 'Cardinal Cartridge Inc.'),

    # Cardinal Health 200
    (r'^Cardinal Health 200,?\s*(Inc\.|LLC)$', 'Cardinal Health 200 Inc.'),

    # Cardinal Home Products
    (r'^Cardinal Home Products,?\s*(Inc\.)?$', 'Cardinal Home Products Inc.'),

    # Cardinal IG
    (r'^Cardinal IG\s*(Company,?\s*)?,?\s*(Co\.)?$', 'Cardinal IG Co.'),

    # Cardionet
    (r'^Cardionet,?\s*(LLC|Inc\.)$', 'Cardionet LLC'),

    # CardsDirect
    (r'^CardsDirect,?\s*(Inc\.|LLC)$', 'CardsDirect Inc.'),

    # Caremark
    (r'^Caremark,?\s*(Inc\.|LLC)$', 'Caremark Inc.'),
]

DIRECT_MAP_3 = {
    # ARM variants
    'ARM':      'ARM Ltd.',
    'ARM Ltd.': 'ARM Ltd.',

    # Borg-Warner → Stanley Black & Decker (BorgWarner was spun off,
    # but these legacy variants likely refer to the old entity)
    'Borg-Warner, Corp.': 'Borg-Warner Corp.',
    'Borg-Warner, Inc.':  'Borg-Warner Corp.',

    # Bosch Rexroth → Robert Bosch GmbH
    'Bosch Rexroth, Inc.':  'Robert Bosch GmbH',
    'Bosch Rexroth, Corp.': 'Robert Bosch GmbH',

    # Boston Scientific Ltd variant
    'Boston Scientific, Ltd.': 'Boston Scientific Corp.',

    # Bloomberg
    'Bloomberg, L.P.': 'Bloomberg L.P.',
    'Bloomberg, Inc.': 'Bloomberg L.P.',

    # BMG Music case fix
    'Bmg Music, Inc.': 'BMG Music Inc.',
    'Bmg Music':       'BMG Music Inc.',

    # CVS
    'CVS, Inc.':  'CVS Pharmacy Inc.',
    'CVS, Corp.': 'CVS Pharmacy Inc.',

    # C & F Packing / C&F Packing unification
    'C & F Packing Company, Inc.': 'C&F Packing Co. Inc.',
    'C & F Packing, Inc.':         'C&F Packing Co. Inc.',
    'C&F Packing Co, Inc.':        'C&F Packing Co. Inc.',
    'C&F Packing Company, Inc.':   'C&F Packing Co. Inc.',

    # CAL AM / Cal AM case
    'CAL AM Manufacturing':        'Cal AM Manufacturing Co. Inc.',
    'CAL AM Manufacturing Co., Inc.': 'Cal AM Manufacturing Co. Inc.',
    'Cal AM Manufacturing':        'Cal AM Manufacturing Co. Inc.',
    'Cal AM Manufacturing Co, Inc.': 'Cal AM Manufacturing Co. Inc.',
}

def apply_similar_groups_map_3(name: str) -> str:
    """Apply batch 3 regex rules then direct map."""
    if not isinstance(name, str):
        return name

    if name in DIRECT_MAP_3:
        return DIRECT_MAP_3[name]

    for pattern, replacement in REGEX_RULES_3:
        if re.match(pattern, name, flags=re.IGNORECASE):
            return replacement

    return name

names['standardized_org_name'] = names['standardized_org_name'].apply(apply_similar_groups_map_3)
# names['standardized_org_name'] = names['standardized_org_name'].apply(apply_similar_groups_map_3)

In [ ]:
names=pd.read_csv('/Users/matt/Desktop/Patent Litigation Analysis/data/processed/names_cleaned_v3.csv')

In [ ]:
import re

# Similar groups map - Batch 4 (C entries)
#
# Key principle: only consolidate when variants are purely
# punctuation/suffix/spacing differences of the SAME entity.
# Subsidiaries, brands, and historically distinct companies
# kept separate even if under the same parent.

REGEX_RULES_4 = [

    # ADI, AMF, Academy, Advanced Technology, Ams, Andrew,
    # Axel J., Card, Co, Co., Company — all skipped

    # Avigilon USA
    (r'^Avigilon USA\s*(Corporation,?\s*Inc\.|,?\s*Corp\.)$', 'Avigilon USA Corp.'),

    # Baldwin Technology
    (r'^Baldwin Technology\s*(Company,?\s*Inc\.|,?\s*Corp\.)$', 'Baldwin Technology Corp.'),

    # Bernhardt Furniture
    (r'^Bernhardt Furniture\s*(Company,?\s*Inc\.|,?\s*Co\.)$', 'Bernhardt Furniture Co.'),

    # Bird Machine — remaining
    (r'^Bird Machine,?\s*Co\.$', 'Bird Machine Co. Inc.'),

    # Black Diamond Equipment — remaining
    (r'^Black Diamond Equipment,?\s*Ltd\.$', 'Black Diamond Equipment Ltd.'),

    # Buffalo Brake Beam — remaining
    (r'^Buffalo Brake Beam,?\s*Co\.$', 'Buffalo Brake Beam Co. Inc.'),

    # Buyers Products — remaining
    (r'^Buyers Products,?\s*Co\.$', 'Buyers Products Co. Inc.'),

    # C.R. Laurence — remaining
    (r'^C\.R\. Laurence,?\s*Co\.,?\s*Inc\.$', 'C.R. Laurence Co. Inc.'),

    # CIBA Specialty Chemicals
    (r'^CIBA Specialty Chemicals\s*(Corporation,?\s*Inc\.|,?\s*Corp\.)$', 'CIBA Specialty Chemicals Corp.'),

    # CVS Pharmacy — remaining
    (r'^CVS Pharmacy,?\s*(Inc\.)?$', 'CVS Pharmacy Inc.'),

    # Cajun Bag & Supply — remaining
    (r'^Cajun Bag & Supply,?\s*Co\.$', 'Cajun Bag & Supply Co. Inc.'),

    # Cambridge Silversmiths — remaining
    (r'^Cambridge Silversmiths,?\s*Ltd\.,?\s*Inc\.$', 'Cambridge Silversmiths Ltd. Inc.'),

    # Cannondale — remaining
    (r'^Cannondale,?\s*Corp\.$', 'Cannondale Corp.'),

    # Cardinal IG — remaining
    (r'^Cardinal IG\s*(Company,?\s*Inc\.|,?\s*Co\.)$', 'Cardinal IG Co.'),

    # Caremark RX — keep separate from Caremark (different entity)
    (r'^Caremark RX,?\s*(LLC|Inc\.)$', 'Caremark RX Inc.'),

    # Cari-All Products
    (r'^Cari-All Products,?\s*(Inc\.)?$', 'Cari-All Products Inc.'),

    # Caribou Coffee
    (r'^Caribou Coffee\s*(Company,?\s*|Co\.?,?\s*)?(Inc\.)?$', 'Caribou Coffee Co. Inc.'),

    # Carlisle
    (r'^Carlisle\s*(Company,?\s*Inc\.|,?\s*Corp\.)$', 'Carlisle Corp.'),

    # Carolina Pad & Paper
    (r'^Carolina Pad & Paper\s*(Co,?\s*|Company,?\s*)?(Inc\.)?$', 'Carolina Pad & Paper Co. Inc.'),

    # Cartier — keep as-is, luxury brand
    (r'^Cartier,?\s*(Inc\.)?$', 'Cartier Inc.'),

    # Carttronics
    (r'^Carttronics,?\s*(LLC|Inc\.)$', 'Carttronics LLC'),

    # Carver
    (r'^Carver,?\s*(Inc\.|Corp\.)$', 'Carver Inc.'),

    # Casana Furniture
    (r'^Casana Furniture\s*(Company,?\s*)?,?\s*Ltd\.$', 'Casana Furniture Ltd.'),

    # Case-Mate International
    (r'^Case-Mate International,?\s*(LLC|Co\.)$', 'Case-Mate International LLC'),

    # Casino Data Systems
    (r'^Casino Data Systems,?\s*(Inc\.)?$', 'Casino Data Systems Inc.'),

    # Casio Computer — already in main map, catch variants
    (r'^Casio Computer\s*(Company,?\s*|,?\s*Co\.?,?\s*)?\s*Ltd\.$', 'Casio Computer Co. Ltd.'),
    (r'^Casio Computer,?\s*Co\.$', 'Casio Computer Co. Ltd.'),

    # Casio Hitachi Mobile Communications — keep separate from Casio
    (r'^Casio Hitachi Mobile Communications\s*(Company,?\s*|Co\.?,?\s*)?Ltd\.$', 'Casio Hitachi Mobile Communications Co. Ltd.'),

    # CastleNet Technology
    (r'^CastleNet Technology,?\s*(Inc\.|Corp\.)$', 'CastleNet Technology Inc.'),

    # Cat Eye
    (r'^Cat Eye\s*Co\.?,?\s*Ltd\.$', 'Cat Eye Co. Ltd.'),

    # Catheter Connections
    (r'^Catheter Connections,?\s*(Inc\.)?$', 'Catheter Connections Inc.'),

    # Cave Consulting Group
    (r'^Cave Consulting Group,?\s*(LLC|Inc\.)$', 'Cave Consulting Group LLC'),

    # Cayman Chemical
    (r'^Cayman Chemical\s*(Company,?\s*|Co\.?,?\s*)?(Inc\.)?$', 'Cayman Chemical Co. Inc.'),

    # Cayman Golf
    (r'^Cayman Golf\s*(Company,?\s*)?,?\s*(Inc\.)?$', 'Cayman Golf Inc.'),

    # Cayman Islands — non-specific descriptor
    (r'^Cayman Islands,?\s*(Corp\.|Co\.)$', 'Non-Specific Entity'),

    # Ce Soir Lingerie
    (r'^Ce Soir Lingerie\s*Co\.?,?\s*(Inc\.)?$', 'Ce Soir Lingerie Co. Inc.'),

    # Celestica
    (r'^Celestica,?\s*(Inc\.|Corp\.)$', 'Celestica Inc.'),

    # Cellectis Bioresearch — case variant, same company
    (r'^Cellectis [Bb]ioresearch,?\s*(Inc\.)?$', 'Cellectis Bioresearch Inc.'),

    # Cellstar / Cellstar Fulfillment — keep separate (different entities)
    (r'^Cellstar,?\s*(Corp\.|Ltd\.)$', 'Cellstar Corp.'),
    (r'^Cellstar Fulfillment,?\s*(Ltd\.|Inc\.)$', 'Cellstar Fulfillment Ltd.'),

    # Cellular Technical Services
    (r'^Cellular Technical Services\s*(Company,?\s*|Co,?\s*)?(Inc\.)?$', 'Cellular Technical Services Co. Inc.'),

    # Centaur Technology
    (r'^Centaur Technology,?\s*(Inc\.)?$', 'Centaur Technology Inc.'),

    # Centennial Machine
    (r'^Centennial Machine\s*(Company,?\s*Inc\.|Co\.?,?\s*Corp\.|,?\s*Co\.)?$', 'Centennial Machine Co.'),

    # Center Ethanol
    (r'^Center Ethanol\s*(Company,?\s*)?,?\s*LLC$', 'Center Ethanol LLC'),

    # Centillion Data Systems
    (r'^Centillion Data Systems,?\s*(LLC|Inc\.)$', 'Centillion Data Systems LLC'),

    # Centra Software
    (r'^Centra Software,?\s*(Inc\.|LLC)$', 'Centra Software Inc.'),

    # Central Purchasing
    (r'^Central Purchasing,?\s*(Inc\.|LLC)$', 'Central Purchasing Inc.'),

    # Central Sprinkler
    (r'^Central Sprinkler,?\s*(Corp\.|Co\.)$', 'Central Sprinkler Corp.'),

    # Centrex Plastics
    (r'^Centrex Plastics,?\s*(Inc\.|LLC)$', 'Centrex Plastics Inc.'),

    # Centurion Medical Products
    (r'^Centurion Medical Products,?\s*(Corp\.)?$', 'Centurion Medical Products Corp.'),

    # Century Link — keep separate from Century (different company)
    (r'^Century Link,?\s*(Inc\.|LLC)$', 'CenturyLink Inc.'),

    # Century Manufacturing
    (r'^Century Manufacturing\s*(Co\.?,?\s*)?(Inc\.)?$', 'Century Manufacturing Co. Inc.'),

    # Century Products
    (r'^Century Products\s*(Company,?\s*Inc\.|,?\s*Co\.|,?\s*Inc\.)?$', 'Century Products Inc.'),

    # Century Wrecker
    (r'^Century Wrecker\s*(Corporation,?\s*Inc\.|,?\s*Corp\.)$', 'Century Wrecker Corp.'),

    # Cenveo
    (r'^Cenveo,?\s*(Inc\.|Corp\.)$', 'Cenveo Inc.'),

    # Cepheid
    (r'^Cepheid,?\s*(Inc\.)?$', 'Cepheid Inc.'),

    # Cequent Performance Products
    (r'^Cequent Performance Products,?\s*(Inc\.)?$', 'Cequent Performance Products Inc.'),

    # Champion Bow
    (r'^Champion Bow\s*(Company,?\s*|Co\.?,?\s*)?Ltd\.$', 'Champion Bow Co. Ltd.'),

    # Champion International — keep separate from Champion Products
    (r'^Champion International,?\s*(Inc\.|Corp\.)$', 'Champion International Corp.'),

    # Champion Products
    (r'^Champion Products,?\s*(Inc\.)?$', 'Champion Products Inc.'),

    # Changer & Dresser
    (r'^Changer & Dresser,?\s*(Corp\.|Inc\.)$', 'Changer & Dresser Corp.'),

    # Chapman Products
    (r'^Chapman Products,?\s*(Corp\.|Inc\.)$', 'Chapman Products Corp.'),

    # Chard International
    (r'^Chard International,?\s*(LLC|Inc\.)$', 'Chard International LLC'),

    # Charles Schwab
    (r'^Charles Schwab & (?:Co|Company)\.?,?\s*(Inc\.)?$', 'Charles Schwab & Co. Inc.'),

    # Charming Shoppes Outlet Stores
    (r'^Charming Shoppes Outlet Stores,?\s*(LLC|Inc\.)$', 'Charming Shoppes Outlet Stores LLC'),

    # Charter Communications — catch bare and LLC variants
    (r'^Charter Communications,?\s*(Inc\.|LLC)?$', 'Charter Communications Inc.'),

    # Charter Furniture — keep separate from Charter Communications
    (r'^Charter Furniture,?\s*(Inc\.|Corp\.)$', 'Charter Furniture Inc.'),

    # Chase Tool & Die
    (r'^Chase Tool & Die\s*Co\.?,?\s*(Inc\.)?$', 'Chase Tool & Die Co. Inc.'),

    # Check Point Software Technologies
    (r'^Check Point Software Technologies,?\s*(Ltd\.|Inc\.)$', 'Check Point Software Technologies Ltd.'),

    # Chem Pruf Door
    (r'^Chem Pruf Door\s*(Co,?\s*Ltd\.|,?\s*Company,?\s*Ltd\.)$', 'Chem Pruf Door Co. Ltd.'),

    # Chem-Nuclear Systems
    (r'^Chem-Nuclear Systems,?\s*(Inc\.|LLC)$', 'Chem-Nuclear Systems Inc.'),

    # Chemence
    (r'^Chemence,?\s*(Inc\.|Ltd\.)$', 'Chemence Inc.'),

    # Chemical Lime
    (r'^Chemical Lime,?\s*(Co\.|Ltd\.)$', 'Chemical Lime Co.'),

    # Chevron — keep separate from Chevron Chemical (different operations)
    (r'^Chevron,?\s*(Inc\.|Corp\.)$', 'Chevron Corp.'),
    (r'^Chevron Chemical\s*(Company,?\s*)?,?\s*(Co\.|LLC)?$', 'Chevron Chemical Co.'),

    # Chicago Tribune
    (r'^Chicago Tribune,?\s*(Co\.|Corp\.)$', 'Chicago Tribune Co.'),

    # Chicony America / Chicony Electronics — keep separate
    (r'^Chicony America,?\s*(Inc\.)?$', 'Chicony America Inc.'),
    (r'^Chicony Electronics\s*(Company,?\s*|Co\.?,?\s*)?Ltd\.$', 'Chicony Electronics Co. Ltd.'),

    # Chien Luen Industries
    (r'^Chien Luen Industries\s*Co\.?\s*Ltd\.?,?\s*(Inc\.)?$', 'Chien Luen Industries Co. Ltd. Inc.'),

    # Choice Hotels International
    (r'^Choice Hotels International,?\s*(Inc\.)?$', 'Choice Hotels International Inc.'),

    # Choon's Design — case variant, same company
    (r"^Choon'?[Ss] Design,?\s*(LLC|Inc\.)$", "Choon's Design LLC"),

    # Christian Dior Perfumes — keep separate from Dior parent
    (r'^Christian Dior Perfumes,?\s*(LLC|Inc\.)$', 'Christian Dior Perfumes LLC'),

    # Christiana Industries
    (r'^Christiana Industries,?\s*(Corp\.|Inc\.)$', 'Christiana Industries Corp.'),

    # Christopher & Banks
    (r'^Christopher & Banks,?\s*(Inc\.|Corp\.)$', 'Christopher & Banks Inc.'),

    # Chrysler / Chrysler Group — keep separate (distinct legal entities)
    (r'^Chrysler,?\s*(Corp\.|LLC)$', 'Chrysler Corp.'),
    (r'^Chrysler Group,?\s*(LLC|Corp\.)$', 'Chrysler Group LLC'),

    # Chugai Pharmaceutical
    (r'^Chugai Pharmaceutical\s*CO?\.?,?\s*Ltd\.$', 'Chugai Pharmaceutical Co. Ltd.'),

    # Church & Dwight
    (r'^Church & Dwight\s*(Company,?\s*|Co\.?,?\s*)?(Inc\.)?$', 'Church & Dwight Co. Inc.'),

    # Ciba-Geigy — now Novartis, but keep as distinct historical entity
    (r'^Ciba-Geigy,?\s*(Corp\.|Ltd\.)$', 'Ciba-Geigy Corp.'),

    # Cingular Wireless / Cingular Wireless II — keep separate (II is a different legal entity)
    (r'^Cingular Wireless,?\s*(LLC|Corp\.)$', 'Cingular Wireless LLC'),
    (r'^Cingular Wireless II,?\s*(LLC|Inc\.)$', 'Cingular Wireless II LLC'),

    # Cinmar
    (r'^Cinmar,?\s*(Inc\.|LLC|L\.P\.)$', 'Cinmar Inc.'),

    # Cinpres Gas Injection
    (r'^Cinpres Gas Injection,?\s*(Inc\.|Ltd\.)$', 'Cinpres Gas Injection Inc.'),

    # Cinram International
    (r'^Cinram International,?\s*(Inc\.)?$', 'Cinram International Inc.'),

    # Ciphergen Biosystems
    (r'^Ciphergen Biosystems,?\s*(Inc\.)?$', 'Ciphergen Biosystems Inc.'),

    # Cisco — catch bare variants (already in main map)
    (r'^Cisco,?\s*(Inc\.|Corp\.)$', 'Cisco Systems Inc.'),

    # Citizen Electronics / Citizen Watch — keep separate
    (r'^Citizen Electronics\s*Co\.?,?\s*Ltd\.$', 'Citizen Electronics Co. Ltd.'),
    (r'^Citizen Watch\s*Co\.?,?\s*Ltd\.$', 'Citizen Watch Co. Ltd.'),

    # Citizens Communications
    (r'^Citizens Communications,?\s*(Co\.)?$', 'Citizens Communications Co.'),

    # Clad Metals
    (r'^Clad Metals,?\s*(LLC|Inc\.)$', 'Clad Metals LLC'),

    # Clairol
    (r'^Clairol,?\s*(Inc\.|Corp\.)$', 'Clairol Inc.'),

    # Clarion
    (r'^Clarion\s*Co\.?,?\s*Ltd\.$', 'Clarion Co. Ltd.'),

    # Claris Lifesciences
    (r'^Claris Lifesciences,?\s*(Ltd\.|Inc\.)$', 'Claris Lifesciences Ltd.'),

    # Classified Cosmetics
    (r'^Classified Cosmetics,?\s*(LLC|Inc\.)$', 'Classified Cosmetics LLC'),

    # Cleaner's Supply
    (r"^Cleaner's Supply,?\s*(Inc\.|Co\.)$", "Cleaner's Supply Inc."),

    # Clegg Industries
    (r'^Clegg Industries,?\s*(Inc\.)?$', 'Clegg Industries Inc.'),

    # Cleveland Golf
    (r'^Cleveland Golf,?\s*(Inc\.|Co\.)$', 'Cleveland Golf Inc.'),

    # ClientConnect
    (r'^ClientConnect,?\s*(Ltd\.|Inc\.)$', 'ClientConnect Ltd.'),

    # Clinical Innovations
    (r'^Clinical Innovations,?\s*(LLC|Inc\.)$', 'Clinical Innovations LLC'),

    # Cliplight Manufacturing
    (r'^Cliplight Manufacturing,?\s*(Co\.|Inc\.)$', 'Cliplight Manufacturing Co.'),

    # Clover Sports
    (r'^Clover Sports\s*Co\.?,?\s*Ltd\.$', 'Clover Sports Co. Ltd.'),

    # CNBC — keep as distinct brand
    (r'^Cnbc,?\s*(Inc\.|LLC)$', 'CNBC Inc.'),

    # CNX Gas
    (r'^Cnx Gas\s*(Company,?\s*LLC|,?\s*Corp\.)$', 'CNX Gas Corp.'),

    # Coastal Pharmaceuticals
    (r'^Coastal Pharmaceuticals,?\s*(Inc\.)?$', 'Coastal Pharmaceuticals Inc.'),

    # Coaster Company of America
    (r'^Coaster Company of America,?\s*(Inc\.)?$', 'Coaster Company of America Inc.'),

    # Cobalt Pharmaceuticals
    (r'^Cobalt Pharmaceuticals,?\s*(Inc\.)?$', 'Cobalt Pharmaceuticals Inc.'),

    # Cobe Laboratories
    (r'^Cobe Laboratories,?\s*(Inc\.)?$', 'Cobe Laboratories Inc.'),

    # Cobra Electronics / Cobra Golf / Cobra Manufacturing — keep separate
    (r'^Cobra Electronics,?\s*(Corp\.)?$', 'Cobra Electronics Corp.'),
    (r'^Cobra Golf,?\s*(Inc\.|Co\.)$', 'Cobra Golf Inc.'),
    (r'^Cobra Manufacturing\s*(Company,?\s*|Co\.?,?\s*)?(Inc\.)?$', 'Cobra Manufacturing Co. Inc.'),

    # Cobraco Manufacturing
    (r'^Cobraco Manufacturing\s*(Co,?\s*Inc\.|,?\s*Inc\.)$', 'Cobraco Manufacturing Inc.'),

    # Coby Electronics
    (r'^Coby Electronics,?\s*(Inc\.|Corp\.)$', 'Coby Electronics Inc.'),

    # Cochlear
    (r'^Cochlear,?\s*(Corp\.|Ltd\.)$', 'Cochlear Ltd.'),

    # Codemasters Software
    (r'^Codemasters Software\s*(Company,?\s*Ltd\.|,?\s*Inc\.)$', 'Codemasters Software Ltd.'),

    # Cogent Solutions
    (r'^Cogent Solutions,?\s*(Inc\.|LLC)$', 'Cogent Solutions Inc.'),

    # Cognex Technology & Investment
    (r'^Cognex Technology & Investment,?\s*(Corp\.|LLC)$', 'Cognex Technology & Investment Corp.'),

    # Cognizant Technology Solutions
    (r'^Cognizant Technology Solutions,?\s*(Corp\.)?$', 'Cognizant Technology Solutions Corp.'),

    # Cognos
    (r'^Cognos,?\s*(Inc\.|Corp\.)$', 'Cognos Inc.'),

    # Cohesion Products
    (r'^Cohesion Products,?\s*(Inc\.|LLC)$', 'Cohesion Products Inc.'),

    # Cold Chain Technologies
    (r'^Cold Chain Technologies,?\s*(Inc\.)?$', 'Cold Chain Technologies Inc.'),

    # Coldwater Creek — trailing space variant
    (r'^Coldwater Creek\s*,?\s*(Inc\.)?$', 'Coldwater Creek Inc.'),

    # Coleman Cable
    (r'^Coleman Cable,?\s*(Inc\.|LLC)$', 'Coleman Cable Inc.'),

    # Collezione Europa U.S.A.
    (r'^Collezione Europa U\.S\.A\.,?\s*(Inc\.)?$', 'Collezione Europa U.S.A. Inc.'),

    # Collins and Aikman
    (r'^Collins and Aikman,?\s*(Inc\.|Corp\.)$', 'Collins and Aikman Corp.'),

    # Coloplast
    (r'^Coloplast,?\s*(Corp\.|Inc\.)$', 'Coloplast Corp.'),

    # Colorado Marketing Group
    (r'^Colorado Marketing Group,?\s*(LLC|Inc\.)$', 'Colorado Marketing Group LLC'),

    # Colorado Time Systems
    (r'^Colorado Time Systems,?\s*(Inc\.|LLC)$', 'Colorado Time Systems Inc.'),

    # Colt's Manufacturing
    (r"^Colt's Manufacturing\s*Company,?\s*(Inc\.|LLC)$", "Colt's Manufacturing Co."),

    # Columbia & Okanogan Nursery
    (r'^Columbia & Okanogan Nursery\s*(Company,?\s*)?,?\s*(Inc\.)?$', 'Columbia & Okanogan Nursery Inc.'),

    # Comfort Products
    (r'^Comfort Products,?\s*(Inc\.)?$', 'Comfort Products Inc.'),

    # Comfortex
    (r'^Comfortex,?\s*(Corp\.|Inc\.)$', 'Comfortex Corp.'),

    # Command Arms Accessories
    (r'^Command Arms Accessories,?\s*(Ltd\.|LLC)$', 'Command Arms Accessories Ltd.'),

    # Commercial Turf Products
    (r'^Commercial Turf Products,?\s*(Ltd\.)?$', 'Commercial Turf Products Ltd.'),

    # Commins Manufacturing
    (r'^Commins Manufacturing,?\s*(Inc\.)?$', 'Commins Manufacturing Inc.'),

    # Common Time
    (r'^Common Time,?\s*(Inc\.|Ltd\.)$', 'Common Time Inc.'),

    # Commonwealth Edison
    (r'^Commonwealth Edison,?\s*(Corp\.|Co\.)$', 'Commonwealth Edison Co.'),

    # Commtest Instruments
    (r'^Commtest Instruments,?\s*(Ltd\.|Inc\.)$', 'Commtest Instruments Ltd.'),

    # Comodo Group
    (r'^Comodo Group,?\s*(Inc\.|Ltd\.)$', 'Comodo Group Inc.'),

    # Comp USA
    (r'^Comp USA,?\s*(Inc\.)?$', 'Comp USA Inc.'),

    # Compact International
    (r'^Compact International,?\s*(Inc\.)?$', 'Compact International Inc.'),

    # Compass Worldwide
    (r'^Compass Worldwide,?\s*(Inc\.|LLC)$', 'Compass Worldwide Inc.'),

    # Component Hardware Group
    (r'^Component Hardware Group,?\s*(Inc\.)?$', 'Component Hardware Group Inc.'),

    # Composite Concepts
    (r'^Composite Concepts\s*(Company,?\s*Inc\.|,?\s*Co\.|,?\s*Inc\.)?$', 'Composite Concepts Inc.'),

    # Composite Technologies
    (r'^Composite Technologies\s*(Co\.?,?\s*LLC|,?\s*Corp\.)$', 'Composite Technologies Corp.'),

    # Compulink Business Systems
    (r'^Compulink Business Systems,?\s*(Inc\.)?$', 'Compulink Business Systems Inc.'),

    # Computer Nerds International
    (r'^Computer Nerds International,?\s*(Inc\.)?$', 'Computer Nerds International Inc.'),

    # Computime
    (r'^Computime,?\s*(Corp\.|Ltd\.)$', 'Computime Corp.'),

    # Compuware
    (r'^Compuware\s*(Corporation,?\s*Inc\.|,?\s*Corp\.)$', 'Compuware Corp.'),

    # Concept Enterprises
    (r'^Concept Enterprises,?\s*(Inc\.)?$', 'Concept Enterprises Inc.'),

    # Concrete Washout Systems
    (r'^Concrete Washout Systems,?\s*(Inc\.)?$', 'Concrete Washout Systems Inc.'),

    # Concurrent Technologies
    (r'^Concurrent Technologies,?\s*(Inc\.|Corp\.)$', 'Concurrent Technologies Inc.'),

    # Conduit
    (r'^Conduit,?\s*(Ltd\.|Inc\.)$', 'Conduit Ltd.'),

    # Condux International
    (r'^Condux International,?\s*(Inc\.)?$', 'Condux International Inc.'),

    # Confab / Confab Holding — keep separate
    (r'^Confab,?\s*(Inc\.|Corp\.)$', 'Confab Corp.'),
    (r'^Confab Holding\s*(Corp\.?,?\s*Inc\.|,?\s*Corp\.)$', 'Confab Holding Corp.'),

    # Connection Technology
    (r'^Connection Technology,?\s*(Ltd\.)?$', 'Connection Technology Ltd.'),

    # ConocoPhillips / Conocophillips — case variant, same company
    (r'^Conocophillips,?\s*(Inc\.|Co\.)$', 'ConocoPhillips Co.'),
    (r'^ConocoPhillips,?\s*(Co\.|Inc\.)?$', 'ConocoPhillips Co.'),

    # Conrad Hospitality
    (r'^Conrad Hospitality,?\s*(Inc\.|LLC)$', 'Conrad Hospitality Inc.'),

    # Consolidated Stores
    (r'^Consolidated Stores,?\s*(Corp\.|Inc\.)$', 'Consolidated Stores Corp.'),

    # Construction Technology
    (r'^Construction Technology,?\s*(Inc\.)?$', 'Construction Technology Inc.'),

    # Consultronics
    (r'^Consultronics,?\s*(Ltd\.|Inc\.)$', 'Consultronics Ltd.'),

    # Container Store
    (r'^Container Store,?\s*(Co\.|Inc\.)$', 'Container Store Co.'),

    # Containment Technologies
    (r'^Containment Technologies,?\s*(Corp\.)?$', 'Containment Technologies Corp.'),

    # Contico International
    (r'^Contico International,?\s*(Inc\.)?$', 'Contico International Inc.'),

    # Control & Metering
    (r'^Control & Metering,?\s*(Ltd\.|Inc\.)$', 'Control & Metering Ltd.'),

    # Control Screening
    (r'^Control Screening,?\s*(LLC|L\.P\.)$', 'Control Screening LLC'),

    # Cook Biotech / Cook Medical — keep separate
    (r'^Cook Biotech,?\s*(Inc\.)?$', 'Cook Biotech Inc.'),
    (r'^Cook Medical,?\s*(Inc\.|LLC)$', 'Cook Medical Inc.'),

    # Cool Gear International
    (r'^Cool Gear International,?\s*(LLC|Inc\.)$', 'Cool Gear International LLC'),

    # Cooler Concepts
    (r'^Cooler Concepts,?\s*(Inc\.|LLC)$', 'Cooler Concepts Inc.'),

    # Cooler Master
    (r'^Cooler Master\s*Co\.?,?\s*Ltd\.$', 'Cooler Master Co. Ltd.'),
]

DIRECT_MAP_4 = {
    # Charter Communications bare variant
    'Charter Communications':     'Charter Communications Inc.',
    'Charter Communications, LLC': 'Charter Communications Inc.',

    # ConocoPhillips bare variant
    'ConocoPhillips': 'ConocoPhillips Co.',

    # Cayman Islands — non-specific
    'Cayman Islands, Corp.': 'Non-Specific Entity',
    'Cayman Islands, Co.':   'Non-Specific Entity',

    # Choon's Design case variants
    "Choon'S Design, LLC": "Choon's Design LLC",
    "Choon'S Design, Inc.": "Choon's Design LLC",
    "Choon's Design, LLC":  "Choon's Design LLC",
    "Choon's Design, Inc.": "Choon's Design LLC",

    # Cisco bare variants
    'Cisco, Inc.':  'Cisco Systems Inc.',
    'Cisco, Corp.': 'Cisco Systems Inc.',

    # CNBC case fix
    'Cnbc, Inc.': 'CNBC Inc.',
    'Cnbc, LLC':  'CNBC Inc.',

    # CNX Gas
    'Cnx Gas, Corp.':        'CNX Gas Corp.',
    'Cnx Gas Company, LLC':  'CNX Gas Corp.',

    # CenturyLink
    'Century Link, Inc.': 'CenturyLink Inc.',
    'Century Link, LLC':  'CenturyLink Inc.',
    "Samsung Telecommunications America, L.P.":             "Samsung Electronics",
    "Samsung Electronic America, Inc.,":                    "Samsung Electronics",
    "Samsung Semiconductor, Inc.,":                         "Samsung Electronics",
    "Samsung SDI America, Inc.":                            "Samsung Electronics",
    "Samsung Austin Semiconductor, L.P.":                   "Samsung Electronics",
    "Samsung Electronics Co. LTD.,":                        "Samsung Electronics",
    "Samsung Telecommunications America LP":                "Samsung Electronics",
    "Samsung Telecommunications America, LLP":              "Samsung Electronics",
    "Samsung Electrionics America, Inc.":                   "Samsung Electronics",
    "Toshiba Samsung Storage Technology Korea, Corp.":      "Samsung Electronics",
    "Samsung Electronics Co, Ltd;":                         "Samsung Electronics",
    "Samsung Information Systems America, Inc.,":           "Samsung Electronics",
    "Samsung Information Systems America, Inc.":            "Samsung Electronics",
    "Samsung Electronics Research Institute":               "Samsung Electronics",
    "Samsung Semiconductor Europe GMBH":                    "Samsung Electronics",
    "Samsung Electronic Co., Ltd.":                         "Samsung Electronics",
    "Samsung Group":                                        "Samsung Electronics",
    "Samsung Telecommunications America, LP":               "Samsung Electronics",
    "Samsung Electronics America, LLC":                     "Samsung Electronics",
    "Samsung Techwin America":                              "Samsung Electronics",
    "Samsung Electronics Company, LTD.,":                   "Samsung Electronics",
    "Samsung Telecommunications America":                   "Samsung Electronics",
    "Samsung Electronics, Co.":                             "Samsung Electronics",
    "Samsung Telecommunications America General, LLC":      "Samsung Electronics",
    "Samsung Techwin, Co.":                                 "Samsung Electronics",
    "Samsung Electronics Corp., Ltd.":                      "Samsung Electronics",
    "Samsung Telecommunications LLP":                       "Samsung Electronics",
    "Samsung International, Inc.":                          "Samsung Electronics",
    "Samsung Aerospace Industries, Ltd.":                   "Samsung Electronics",
    "Toshiba Samsung Storage Technology, Corp.":            "Samsung Electronics",
    "Samsung Electro-Mechanics America, Inc.":              "Samsung Electronics",
    "Samsung Co., Ltd.":                                    "Samsung Electronics",
    "Samsung Telecommunications, Inc.":                     "Samsung Electronics",
    "Samsung Techwin Co.,":                                 "Samsung Electronics",
    "Samsung Display Devices Co., Ltd.":                    "Samsung Electronics",
    "Samsung Display Devices Co., Inc.":                    "Samsung Electronics",
    "Samsung Elec. Co., Ltd.":                              "Samsung Electronics",
    "Samsung Telecom America LLP":                          "Samsung Electronics",
    "formerly known asSamsung Telecommunications America, LP": "Samsung Electronics",
    "Samsung American, Inc.":                               "Samsung Electronics",
    "Samsung Austin General, LLC":                          "Samsung Electronics",
    "Samsung Sdi Co., Ltd.":                                "Samsung Electronics",
    "Samsung Networks, Inc.":                               "Samsung Electronics",
    "Samsung Electronics, Co., Inc.":                       "Samsung Electronics",
    "Samsung Technology, Inc.":                             "Samsung Electronics",
    "Samsung Electronics America, L.P.":                    "Samsung Electronics",
    "Samsung Electronics, Company, Ltd.":                   "Samsung Electronics",
    "Samsung LED Co, Ltd.":                                 "Samsung Electronics",
    "Samsung LED Company, Ltd.":                            "Samsung Electronics",
    "Samsung Electro-Mechanics Company, Ltd.":              "Samsung Electronics",
    "Samsung LED America, Inc.":                            "Samsung Electronics",
    "Samsung Techwin Co, Ltd.":                             "Samsung Electronics",
    "Samsung Electronics America, Ltd;":                    "Samsung Electronics",
    "Samsung Semiconductor, Inc;":                          "Samsung Electronics",
    "Samsung Electronic USA, Inc.":                         "Samsung Electronics",
    "Samsung Electronics, USA, Inc.":                       "Samsung Electronics",
    "samsung electronics co., Ltd.":                        "Samsung Electronics",
    "Tianjin Samsung Electronics Co., Ltd.":                "Samsung Electronics",
    "Samsung Mexicana S.A. De C.V.":                        "Samsung Electronics",
    "Samsung Electronics Usa":                              "Samsung Electronics",
    "Samsung Digital Imaging Co., Ltd.":                    "Samsung Electronics",
    "Samsung Electronics Co., Ltd. et al.":                 "Samsung Electronics",
    "Samsung Telecommunication America, Inc.":              "Samsung Electronics",
    "Samsung Display, Co.":                                 "Samsung Electronics",
    "Samsung C&T America, Inc.":                            "Samsung Electronics",
    "Samsung Electronics Co.,LTD.":                         "Samsung Electronics",
    "Samsung Electro Mechanics Co, Ltd.":                   "Samsung Electronics",
    "Amazon.Com International, Inc.":           "Amazon.com Inc.",
    "Amazon.com Holdings, LLC":                 "Amazon.com Inc.",
    "Amazon Environmental, Inc.":               "Amazon.com Inc.",
    "Amazon.Com, LLC":                          "Amazon.com Inc.",
    "Amazone, Inc.":                            "Amazon.com Inc.",
    "Amazon European Union S.a.r.l.":           "Amazon.com Inc.",
    "Amazon Services Canada, Inc.":             "Amazon.com Inc.",
    "Amazon Services Europe S.a.r.l.":          "Amazon.com Inc.",
    "Amazon.com Int'l Sales, Inc.":             "Amazon.com Inc.",
    "Amazon.com.ca, Inc.":                      "Amazon.com Inc.",
    "doing business asAmazon.dedoing business asAmazon.frdoing business asAmazon.co.uk": "Amazon.com Inc.",
    "Amazon.com Auctions, Inc.":                "Amazon.com Inc.",
    "Amazon.com, Inc. d/b/a Amazon.com":        "Amazon.com Inc.",
    "Amazon.com.dedc, LLC":                     "Amazon.com Inc.",
    "Amazon Mechanical Turk, Inc.":             "Amazon.com Inc.",
    "formerly known asAmazon Services Incdoing business asAmazon Enterprise Solutionsdoing business asAmazon Services Business Solutions": "Amazon.com Inc.",
    "doing business asAmazon.co.jp":            "Amazon.com Inc.",
    "Apple Sales International":                "Apple Inc.",
    "Apple Medical, Corp.":                     "Apple Inc.",
    "Apple Retail Germany GMBH":                "Apple Inc.",
    "Apple Benelux B.V.":                       "Apple Inc.",
    "Apple Retail Netherlands B.V.":            "Apple Inc.",
    "Apple Retail UK, Ltd.":                    "Apple Inc.",
    "Apple GMBH":                               "Apple Inc.",
    "Apple Japan, Inc.":                        "Apple Inc.",
    "Apple Holding B.V.":                       "Apple Inc.",
    "Apple Netherlands B.V.":                   "Apple Inc.",
    "Apple PTY, Ltd.":                          "Apple Inc.",
    "Apple Korea, Ltd.":                        "Apple Inc.",
    "Apple Distribution International":         "Apple Inc.",
    "Pfizer Manufacturing Holdings, LLC":                   "Pfizer Inc.",
    "Pfizer Health Ab":                                     "Pfizer Inc.",
    "PFizer Inc.,":                                         "Pfizer Inc.",
    "Pfizer Asia Pacific PTE, Ltd.":                        "Pfizer Inc.",
    "Pfizer Health AB":                                     "Pfizer Inc.",
    "Pfizer Technologies, Ltd.":                            "Pfizer Inc.",
    "Pfizer Ireland Pharmaceuticals (a partnership)":       "Pfizer Inc.",
    "Coty Division of Pfizer Inc.,":                        "Pfizer Inc.",
    "Pfizer Consumer Health Care Group":                    "Pfizer Inc.",
    "Pfizer pharmaceuticals, LLC":                          "Pfizer Inc.",
    "Pfizer Asia-Pacific Pte., Ltd.":                       "Pfizer Inc.",
    "Pfizer Research":                                      "Pfizer Inc.",
    "Pfizer Health Solutions":                              "Pfizer Inc.",
    "Pfizer Products, Inc.":                                "Pfizer Inc.",
    "Pfizer Ireland Pharmaceuticals (an unlimited liability company)": "Pfizer Inc.",
    "Pfizer Hospital Prod":                                 "Pfizer Inc.",
    "a Pfizer, Co.":                                        "Pfizer Inc.",

    # Cingular Wireless → AT&T (acquired 2007)
    # NOTE: keeping as Cingular since these are historical litigation records
    # where the entity named was Cingular at time of filing
}

def apply_similar_groups_map_4(name: str) -> str:
    """Apply batch 4 regex rules then direct map."""
    if not isinstance(name, str):
        return name

    if name in DIRECT_MAP_4:
        return DIRECT_MAP_4[name]

    for pattern, replacement in REGEX_RULES_4:
        if re.match(pattern, name, flags=re.IGNORECASE):
            return replacement

    return name

names['standardized_org_name'] = names['standardized_org_name'].apply(apply_similar_groups_map_4)
# names['standardized_org_name'] = names['standardized_org_name'].apply(apply_similar_groups_map_4)

In [ ]:
import re

# Function to normalize names by removing suffixes
def normalize_name(name):
    if not isinstance(name, str):
        return name
    for pattern, _ in SUFFIX_MAP:
        name = re.sub(pattern, '', name, flags=re.IGNORECASE)
    return name.strip()

# Filter out Non-Specific Entity
filtered = names[names['standardized_org_name'] != 'Non-Specific Entity']

# Apply normalization
filtered['normalized'] = filtered['standardized_org_name'].apply(normalize_name)

# Group by normalized name and get unique standardized_org_names
groups = filtered.groupby('normalized')['standardized_org_name'].unique()

# Filter groups with more than one unique standardized_org_name
similar_groups = groups[groups.apply(len) > 1]

# Display the similar groups
for base, variants in similar_groups.head(200).items():  # Show first 20 for brevity
    print(f"Base: {base}")
    print(f"Variants: {list(variants)}")
    print("---")


Base: ADI
Variants: ['ADI, Ltd.', 'ADI, Corp.']
---
Base: AMF
Variants: ['AMF, Corp.', 'AMF, Inc.']
---
Base: Academy
Variants: ['Academy, Corp.', 'Academy, Ltd.']
---
Base: Advanced Technology
Variants: ['Advanced Technology, Ltd.', 'Advanced Technology']
---
Base: Ameritrade Holding
Variants: ['Ameritrade Holding Corp.', 'Ameritrade Holding Corporation, Inc.']
---
Base: Amphenol
Variants: ['Amphenol Corp.', 'Amphenol Corporation, Inc.']
---
Base: Ams
Variants: ['Ams, Inc.', 'Ams, LLC']
---
Base: Andrew
Variants: ['Andrew, Corp.', 'Andrew, LLC']
---
Base: Axel J.
Variants: ['Axel J., LLC', 'Axel J., Corp.']
---
Base: CFM
Variants: ['CFM, Inc.', 'CFM, Corp.']
---
Base: Card
Variants: ['Card, Ltd.', 'Card, LLC']
---
Base: Cayman Chemical
Variants: ['Cayman Chemical Co. Inc.', 'Cayman Chemical, Co.']
---
Base: Century
Variants: ['Century, Inc.', 'Century, LLC']
---
Base: Century Manufacturing
Variants: ['Century Manufacturing, Co.', 'Century Manufacturing Co. Inc.']
---
Base: CenturyLink

In [ ]:
from rapidfuzz import process, fuzz

unique_names = names[
    names['standardized_org_name'] != 'Non-Specific Entity'
]['standardized_org_name'].value_counts()

rare_names = unique_names[unique_names < 5].index.tolist()
common_names = unique_names[unique_names >= 5].index.tolist()

matches = []
for name in rare_names:
    result = process.extractOne(
        name,
        common_names,
        scorer=fuzz.token_sort_ratio,
        score_cutoff=88
    )
    if result:
        matches.append({
            'original': name,
            'matched_to': result[0],
            'score': result[1]
        })

match_df = pd.DataFrame(matches).sort_values('score', ascending=False)

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 200)
print(match_df.head(200).to_string())

                                                                                                                                             original                                                                                                                                       matched_to       score
676                                                                                                                   Charles Machine Works, Inc. The                                                                                                                  The Charles Machine Works, Inc.  100.000000
1230                       Under Secretary of Commerce for Intellectual Property and Acting Director of the United States Patent and Trademark Office                       Acting Under Secretary of Commerce for Intellectual Property and Director of the United States Patent and Trademark Office  100.000000
2935                                                                           

In [ ]:
# Safe to auto-apply — score of 100 only
auto_map = match_df[match_df['score'] == 100].set_index('original')['matched_to'].to_dict()
print(f"Auto-applying {len(auto_map)} mappings")

# Everything else — review manually before applying
review_df = match_df[match_df['score'] < 100]
print(f"{len(review_df)} entries need manual review")

Auto-applying 23 mappings
8531 entries need manual review


In [ ]:
names['standardized_org_name'] = names['standardized_org_name'].map(auto_map).fillna(names['standardized_org_name'])

In [ ]:
tier1_map = match_df[match_df['score'] >= 99].set_index('original')['matched_to'].to_dict()
print(f"Applying {len(tier1_map)} mappings")
names['standardized_org_name'] = names['standardized_org_name'].map(tier1_map).fillna(names['standardized_org_name'])

Applying 43 mappings


In [ ]:
SKIP = {
    'Charter Communications Entertainment II, LLC',
    'Charter Communications V, LLC',
    'Kroger Limited Partnership II',
    'Kroger Limited Partnership Ii',
    'in her official capacity as Under Secretary of Commerce for Intellectual Property and Director of the United States Patent and Trademark Office',
    'Time Warner Entertainment-Advanced/Newhouse Partnership',
}

tier2_clean = tier2[~tier2['original'].isin(SKIP)]
tier2_map = tier2_clean.set_index('original')['matched_to'].to_dict()

print(f"Applying {len(tier2_map)} tier 2 mappings")
names['standardized_org_name'] = names['standardized_org_name'].map(tier2_map).fillna(names['standardized_org_name'])

Applying 1881 tier 2 mappings


In [ ]:
print(names['standardized_org_name'].nunique())

95085


In [ ]:
# Tier 3: 88-95 — higher risk, review carefully
tier3 = match_df[match_df['score'] < 95]
print(f"Tier 3 (88-95): {len(tier3)} entries")
print(tier3.to_string())

Tier 3 (88-95): 6625 entries
                                                                                                                                                                    original                                                                                                                                       matched_to      score
5450                                                                                                                       Board of Regentsof the University of Texas System                                                                                               Board of Regents of the University of Texas System  94.949495
522                                                                                                                                          Samsung Electronics Corp., Ltd.                                                                                                                     Samsung Electronic Co., 

In [ ]:
tmobile_mask = names['standardized_org_name'].str.contains(
    r'T-Mobile', case=False, na=False
)
names.loc[tmobile_mask, 'standardized_org_name'] = 'T-Mobile Inc.'

In [ ]:
microsoft_mask = names['standardized_org_name'].str.contains(
    r'Microsoft', case=False, na=False
)
names.loc[microsoft_mask, 'standardized_org_name'] = 'Microsoft Corporation'

In [ ]:
samsung_mask = names['standardized_org_name'].str.contains(
    r'Samsung', case=False, na=False
)
names.loc[samsung_mask, 'standardized_org_name'] = 'Samsung Electronics Co.'

In [ ]:
names=pd.read_csv('/Users/matt/Desktop/Patent Litigation Analysis/data/processed/names_cleaned_v3.csv')

In [ ]:
import re

REGEX_RULES_3b = []

DIRECT_MAP_3b = {
    "Hewlett-Packard Development Company, LP":                          "HP Inc.",
    "- Hewlett-Packard, Co.":                                           "HP Inc.",
    "Hewlett Packard Company, Inc.":                                    "HP Inc.",
    "The Hewlett-Packard, Co.":                                         "HP Inc.",
    "Hewlett-Packard Company, Inc.":                                    "HP Inc.",
    "Hewlett-Packard Far East Pte. LTD.,":                              "HP Inc.",
    "Hewlett-Packard Company d/b/a Hewlett Packard d/b/a HP d/b/a HP Security Voltage": "HP Inc.",
    "Hewlett-packard, Co.":                                             "HP Inc.",
    "Hewlett-Packard Development, Co.":                                 "HP Inc.",
    "Hewlett-Parkard, Co.":                                             "HP Inc.",
    "Hewlett-Packard Co., Inc.":                                        "HP Inc.",
    "Hewlett-Packard, Inc.":                                            "HP Inc.",
    "Shangai Hewlett-Packard Co., Ltd.":                                "HP Inc.",
    "Hewlett-Packard Development Company Enterprise, LP":               "HP Inc.",
    "HEWLETT-PACKARD COMPANY, a Delaware, Corp.":                       "HP Inc.",
    "Hewlett-packard Development Company, Lp":                          "HP Inc.",
    "Hewlett-Packard Development Company, Limited Partnership":         "HP Inc.",
    "Hewlett-Packard Company, .":                                       "HP Inc.",
    "Hewlett-Packard GmbH":                                             "HP Inc.",
    "Hewlett Packard GmbH":                                             "HP Inc.",
    "Hewlett-Packard Colorspan, Inc.":                                  "HP Inc.",
    "HP Enterprise Services, LLC":      "HP Inc.",
    "HP, Inc.":                         "HP Inc.",
    "HP Development Company, LLC":      "HP Inc.",
    "HP Intellectual, Corp.":           "HP Inc.",
    "Hp, Inc.":                         "HP Inc.",
    "HP Spartacote, LLC":               "HP Inc.",
    "HP Verifone, Inc.":                "HP Inc.",
    "Lg Electronics, Inc.":                                     "LG Electronics",
    "Lg Electronics U.S.A., Inc.":                              "LG Electronics",
    "LG Philips LCD Co, Ltd.":                                  "LG Electronics",
    "LG.Philips LCD Co., Ltd.":                                 "LG Electronics",
    "LG Electronics Mobilcomm USA, Inc.":                       "LG Electronics",
    "LG, Corp.":                                                "LG Electronics",
    "LG Electronics U.S.A, Inc.":                               "LG Electronics",
    "LG Semicon Co., Ltd.":                                     "LG Electronics",
    "Lg Electronics, U.S.A., Inc.":                             "LG Electronics",
    "LG-Ericsson USA, Inc.":                                    "LG Electronics",
    "LG International (America), Inc.":                         "LG Electronics",
    "LG Electronics Alabama, Inc.":                             "LG Electronics",
    "LG Electronics Monterrey Mexico S.A., DE, CV":             "LG Electronics",
    "LG Electronics Mobilecomm USA., Inc.,":                    "LG Electronics",
    "LG Electronics, USA, Inc.,":                               "LG Electronics",
    "LG Electronics Mobile Communications, Co.":                "LG Electronics",
    "LG Electronic, Inc.":                                      "LG Electronics",
    "Lg, Corp.":                                                "LG Electronics",
    "LG Semicon America":                                       "LG Electronics",
    "LG Infocomm USA, Inc.":                                    "LG Electronics",
    "LG Innotek USA, Inc.":                                     "LG Electronics",
    "LG Electronics Inc.,":                                     "LG Electronics",
    "LG Mobilecomm USA, Inc.":                                  "LG Electronics",
    "LG-Ericsson Co., Ltd.":                                    "LG Electronics",
    "LG Electronics MobileCOMM U.S.A., Inc.":                   "LG Electronics",
    "LG Electronics U S A, Inc.":                               "LG Electronics",
    "LG. Philips LCD America, Inc.":                            "LG Electronics",
    "LG Electronics MobileComm, U.S.A., Inc.":                  "LG Electronics",
    "LG Household & Healthcare, Ltd.":                          "LG Electronics",
    "LG Innotek Co., Ltd.":                                     "LG Electronics",
    "Lg Display Co., Ltd.":                                     "LG Electronics",
    "LG Electronics, Co.":                                      "LG Electronics",
    "LG Group":                                                 "LG Electronics",
    "LG Semicon America, Inc.":                                 "LG Electronics",
    "LG Sourcing, Inc.":                                        "LG Electronics",
    "LG.Philips LCD America, Inc.":                             "LG Electronics",
    "LG Display Co, Ltd.":                                      "LG Electronics",
    "Lg Household & Health Care, Ltd.":                         "LG Electronics",
    "Lg Electronics Usa, Inc.":                                 "LG Electronics",
    "Ericsson-LG Co., Ltd.":                                    "LG Electronics",
    "LG Electronic Mobilecomm U.S.A., Inc.":                    "LG Electronics",
    "LG Electronics Mobilecomm U.S.A., Inc. d/b/a LG Mobile Phones": "LG Electronics",
    "LG Electronics Mobile Research U.S.A., LLC":               "LG Electronics",
    "LG Electronics MobileComm, U.S.A.":                        "LG Electronics",
    "LG Innotek Co, Ltd.":                                      "LG Electronics",
    "LG Mobile Phones":                                         "LG Electronics",
    "LG Electroncis, Inc.":                                     "LG Electronics",
    "LG Electroncis U.S.A., Inc.":                              "LG Electronics",
    "Hitachi-LG Data Storage Korea, Inc.":                      "LG Electronics",
    "Hitachi-LG Data Storage, Inc.":                            "LG Electronics",
    "LG Electronics, Inc.,":                                    "LG Electronics",
    "also known asLG Electronics Mobilecomm, a California, Corp.": "LG Electronics",
    "Lg Household & Healthcare America, Inc.":                  "LG Electronics",
    "Lg Electronics Mobilecomm U.S.A., Inc.":                   "LG Electronics",
    "Lg Electronics Indonesia":                                  "LG Electronics",
    "LG Electronics Mobile Research USA, LLC":                  "LG Electronics",
    "Lg.philips Lcd Co., Ltd.":                                  "LG Electronics",
    "Lg.philips Lcd America":                                    "LG Electronics",
    "LG Chem, Ltd.":                                            "LG Electronics",
    "LG Chem America, Inc.":                                    "LG Electronics",
    "LG Electronics Mobilcomm, U.S.A., Inc.":                   "LG Electronics",
    "LG Electronics U.S.A.,Inc.":                               "LG Electronics",
    "LG Electronics Mobilecom U.S.A., Inc.":                    "LG Electronics",
    "LG Electronics Mobilecomm":                                "LG Electronics",
    "LG Interiors, LLC":                                        "LG Electronics",
    "LG Innotek Huizhou, Inc.":                                  "LG Electronics",
    "LG Innotek U.S.A., Inc.":                                   "LG Electronics",
    "LG Electronics Co., Ltd.":                                  "LG Electronics",
    "LG Innotek Company, Ltd.":                                  "LG Electronics",
    "LG Electronics, Ltd.":                                      "LG Electronics",
    "Goldstar Co., Ltd.":                           "LG Electronics",
    "Goldstar Instrument & Electric Co., Ltd.":     "LG Electronics",
    "Goldstar, Co., Ltd.":                          "LG Electronics",
    "Goldstar U.S.A., Inc.":                        "LG Electronics",
    "Goldstar Electronics International, Inc.":     "LG Electronics",
    "Dell Products, L.P.":                                      "Dell Inc.",
    "Dell Gen P, Corp.":                                        "Dell Inc.",
    "Dell Marketing, L.P.":                                     "Dell Inc.",
    "Dell USA LP":                                              "Dell Inc.",
    "Dell Products LP":                                         "Dell Inc.",
    "Dell USA, Corp.":                                          "Dell Inc.",
    "Dell, Inc. D/B/A Dell Services":                           "Dell Inc.",
    "SecureWorks, Inc. D/B/A Dell SecureWorks":                 "Dell Inc.",
    "DELL, Inc.":                                               "Dell Inc.",
    "d/b/a Dell Computer Inc, d/b/a Dell Computer f/k/a Dell Computer, Corp.": "Dell Inc.",
    "Dell Laboratories Inc.,":                                  "Dell Inc.",
    "Dell, Corp.":                                              "Dell Inc.",
    "Dell SecureWorks, Inc.":                                   "Dell Inc.",
    "formerly known asDell Computer, Corp.":                    "Dell Inc.",
    "Dell, USA, LP":                                            "Dell Inc.",
    "DELL INC., a Delaware, Corp.":                             "Dell Inc.",
    "Dell Inc. dba Dell Services":                              "Dell Inc.",
    "Motorola Inc.,":                                           "Motorola Inc.",
    "Motorola, Inc. Motorola, Inc.":                            "Motorola Inc.",
    "also known asMotorola Mobility, Inc.":                     "Motorola Inc.",
    "Motorola Lighting, Inc.":                                  "Motorola Inc.",
    "formerly known asMotorola Mobility, Inc.":                 "Motorola Inc.",
    "Motorola , Incorporated,":                                 "Motorola Inc.",
    "Motorola Trademark Holdings, LLC":                         "Motorola Inc.",
    "Verizon Communications Inc. d/b/a Verizon Wireless":       "Verizon Communications",
    "Verizon Enterprise Delivery, LLC":                         "Verizon Communications",
    "Verizon Delaware, Inc.":                                   "Verizon Communications",
    "Verizon Corporate Resources Group, LLC":                   "Verizon Communications",
    "Verizon North, Inc.":                                      "Verizon Communications",
    "Verizon Business Global, LLC":                             "Verizon Communications",
    "Verizon New York, Inc.":                                   "Verizon Communications",
    "Verizon Delaware, LLC":                                     "Verizon Communications",
    "Verizon Clinton Center Drive, Corp.":                      "Verizon Communications",
    "Verizon New Jersey, Inc.":                                 "Verizon Communications",
    "Verizon Internet Services, Inc.":                          "Verizon Communications",
    "Verizon Corporate Services Group, Inc.":                   "Verizon Communications",
    "doing business asVerizon Wireless":                        "Verizon Communications",
    "Verizon Pennsylvania, Inc.":                               "Verizon Communications",
    "Verizon Select Services, Inc.":                            "Verizon Communications",
    "Verizon New England, Inc.":                                "Verizon Communications",
    "Verizon Maryland, Inc.":                                   "Verizon Communications",
    "Verizon Advanced Data, Inc.":                              "Verizon Communications",
    "Verizon Avenue, Corp.":                                    "Verizon Communications",
    "Verizon Northwest, Inc.":                                  "Verizon Communications",
    "Verizon and Redbox Digital Entertainment Services, LLC":   "Verizon Communications",
    "Verizon Wireless Services, LLC":                           "Verizon Communications",
    "Verizon Florida, LLC":                                     "Verizon Communications",
    "Verizon Wireless (VAM), LLC":                              "Verizon Communications",
    "Verizon Data Services, Inc.":                              "Verizon Communications",
    "Verizon Information Services, Inc.":                       "Verizon Communications",
    "Verizon Laboratories, Inc.":                               "Verizon Communications",
    "Verizon Virginia, LLC":                                    "Verizon Communications",
    "Verizon Federal, Inc.":                                    "Verizon Communications",
    "Verizon Communications, Inc., .":                          "Verizon Communications",
    "Verizon TeleProducts, Corp.":                              "Verizon Communications",
    "Verizon Long Distance, LLC":                               "Verizon Communications",
    "GTE Southwest Incorporated DBA Verizon Southwest":         "Verizon Communications",
    "Verizon Corporate Services, Corp.":                        "Verizon Communications",
    "doing business asVerizon Internet Solutions":              "Verizon Communications",
    "doing business asVerizon Southwest":                       "Verizon Communications",
    "doing business asVerizon Services Group":                  "Verizon Communications",
    "Verizon Info. Technologies, LLC":                          "Verizon Communications",
    "Verizon Services Organization, Inc.":                      "Verizon Communications",
    "Verizon Online - Maryland, LLC":                           "Verizon Communications",
    "Verizon Online Pennsylvania Partnership":                  "Verizon Communications",
    "Verizon Information Technologies, LLC":                    "Verizon Communications",
    "Verizon Florida, Inc.":                                    "Verizon Communications",
    "Verizon Hawaii, Inc.":                                     "Verizon Communications",
    "Verizon West Virginia, Inc.":                              "Verizon Communications",
    "Verizon Global Networks, Inc.":                            "Verizon Communications",
    "doing business asVERIZON WIRELESS":                        "Verizon Communications",
    "d/b/a Verizon Wireless":                                   "Verizon Communications",
    "CELLCO PARTNERSHIP d/b/a VERIZON WIRELESS":                "Verizon Communications",
    "Verizon Comm, Inc.":                                       "Verizon Communications",
    "GTE Southwest Inc. d/b/a Verizon Southwest":               "Verizon Communications",
    "Verizon Wireless (VAW), LLC":                              "Verizon Communications",
    "Verizon Data Services India PVT., Ltd.":                   "Verizon Communications",
    "Verizon Washington, DC, Inc.":                             "Verizon Communications",
    "Verizon North, LLC":                                       "Verizon Communications",
    "Verizon, Inc.":                                            "Verizon Communications",
    ". Verizon Communications, Inc.":                           "Verizon Communications",
    "Verizon Communications , Inc.":                            "Verizon Communications",
    "Verizon Washington Dc, Inc.":                              "Verizon Communications",
    "Verizon West Coast, Inc.":                                 "Verizon Communications",
    "doing business asVerizon Mid-States":                      "Verizon Communications",
    "Verizon Washington, D.C., Inc.":                           "Verizon Communications",
    "Bell Atlantic Communications, Inc. d/b/a Verizon Long Distance": "Verizon Communications",
    "Verizon Trademark Services, LLC":                          "Verizon Communications",
    "Verizon Communications Inc. D/B/A Verizon Wireless":       "Verizon Communications",
    "Doing business asVerizon Wireless":                        "Verizon Communications",
    "Verizon Communications, Inc.,":                            "Verizon Communications",
    "n/k/a Cellco Partnership, d/b/a/ Verizon Wireless":        "Verizon Communications",
    "Cellco Partnership (DBA Verizon Wireless)":                "Verizon Communications",
    "Verizon Wireless Telecom, Inc.":                           "Verizon Communications",
    "Verizon Media, LLC":                                       "Verizon Communications",
    "Teva Parenteral Medicines, Inc.":                          "Teva Pharmaceutical Industries",
    "Teva Women'S Health, Inc.":                                "Teva Pharmaceutical Industries",
    "Teva Pharmaceuticals Usa":                                 "Teva Pharmaceutical Industries",
    "Teva Branded Pharmaceutical Products R&D, Inc.":           "Teva Pharmaceutical Industries",
    "Teva Pharmaceuticals USA":                                 "Teva Pharmaceutical Industries",
    "Teva Women's Health, Inc.":                                "Teva Pharmaceutical Industries",
    "TEVA Pharmaceuticals":                                     "Teva Pharmaceutical Industries",
    "Teva Pharmaceuticals, Ltd.":                               "Teva Pharmaceutical Industries",
    "Teva Sante SAS":                                           "Teva Pharmaceutical Industries",
    "Teva Pharmaceuticals USA , Inc.":                          "Teva Pharmaceutical Industries",
    "Teva Pharmaceutical Industries Ltd (Israel)":              "Teva Pharmaceutical Industries",
    "Teva Respiratory, LLC":                                    "Teva Pharmaceutical Industries",
    "TEVA Pharmaceuticals Europe B.V.":                         "Teva Pharmaceutical Industries",
    "Teva Pharmaceutical Works, Ltd.":                          "Teva Pharmaceutical Industries",
    "TEVA Pharmaceuticals Industries, Ltd.":                    "Teva Pharmaceutical Industries",
    "Teva Industries, Ltd.":                                    "Teva Pharmaceutical Industries",
    "Teva Pharamceuticals USA, Inc.":                           "Teva Pharmaceutical Industries",
    "TEVA Neuroscience, Inc.":                                  "Teva Pharmaceutical Industries",
    "Teva API, Inc.":                                           "Teva Pharmaceutical Industries",
    "Teva Branded Pharmaceutical Products R & D, Inc.":         "Teva Pharmaceutical Industries",
    "Teva Sante S.A.S.":                                        "Teva Pharmaceutical Industries",
    "Teva Pharmaceuticals":                                     "Teva Pharmaceutical Industries",
    "Mylan Institutional, LLC":             "Mylan Inc.",
    "Mylan Institutional, Inc.":            "Mylan Inc.",
    "Mylan Pharamaceuticals, Inc.":         "Mylan Inc.",
    "Mylan Phamaceuticals, Inc.":           "Mylan Inc.",
    "Mylan Pharmaceuticals":               "Mylan Inc.",
    "Mylan, LLC":                           "Mylan Inc.",
    "Mylan pharmaceuticals, Inc.":          "Mylan Inc.",
    "Mylan Pharmceuticals, Inc.":           "Mylan Inc.",
    "Mylan Pharmacueticals, Inc.":          "Mylan Inc.",
    "MYLAN Pharmaceuticals, Inc.":          "Mylan Inc.",
    "MYLAN, Inc.":                          "Mylan Inc.",
    "Mylan Pharma Acquisition, Ltd.":       "Mylan Inc.",
    "Mylan Pharmaceticals, Inc.":           "Mylan Inc.",
    "Mylan Laboratories, Inc.,":            "Mylan Inc.",
    "American Telephone And, Telegraph Co. (At&T)":     "AT&T Inc.",
    "At&T Ipm, Corp.":                                  "AT&T Inc.",
    "At&T Universal Card Services, Corp.":              "AT&T Inc.",
    "At&T American Transtech, Inc.":                    "AT&T Inc.",
    "AT and T Mobility, LLC":       "AT&T Inc.",
    "AT and T Mobility II, LLC":    "AT&T Inc.",
    "AT and T Services, Inc.":      "AT&T Inc.",
    "Sony Electronics":                                         "Sony Corp.",
    "Sony Network Entertainment International, LLC":            "Sony Corp.",
    "Sony Computer Entertainment America":                      "Sony Corp.",
    "Sony Pictures Imageworks, Inc.":                           "Sony Corp.",
    "Sony Mobile Communications (U.S.A), Inc.":                 "Sony Corp.",
    "Sony Ericssson Mobile Communications (USA), Inc.":         "Sony Corp.",
    "Sony Mobile Communications, Inc.":                         "Sony Corp.",
    "SONY, Corp.":                                              "Sony Corp.",
    "Sony Computer Entertainments America, Inc.":               "Sony Corp.",
    "Sony Music Entertainment":                                 "Sony Corp.",
    "Sony Computer Entertainment of America, LLC":              "Sony Corp.",
    "Sony Corporation of America, Inc.":                        "Sony Corp.",
    "Sony Connect, Inc.":                                       "Sony Corp.",
    "Sony Kabushiki Kaisha":                                    "Sony Corp.",
    "Sony EMCS, Corp.":                                         "Sony Corp.",
    "Sony Disc and Digital Solutions":                          "Sony Corp.",
    "Sony-Ericsson Mobile Communications (USA), Inc.":          "Sony Corp.",
    "Sony Corporation of Amercia":                              "Sony Corp.",
    "Sony Corp. of America":                                    "Sony Corp.",
    "Sony Computer Entertainment Inc.,":                        "Sony Corp.",
    "Sony Creative Software, Inc.":                             "Sony Corp.",
    "Sony Online Entertainment, Inc.":                          "Sony Corp.",
    "Sony Network Entertainment America, Inc.":                 "Sony Corp.",
    "Sony Mobile Communications (Usa), Inc.":                   "Sony Corp.",
    "Sony Cinema Products, Corp.":                              "Sony Corp.",
    "Sony-Ericsson Mobile Communication (USA), Inc.":           "Sony Corp.",
    "Sony Software, Corp.":                                     "Sony Corp.",
    "Sony Ericsson Communication (USA) Inc.,":                  "Sony Corp.",
    "Sony Ericsson Mobile Communications AB,":                  "Sony Corp.",
    "Sony Optiarc America, Inc.":                               "Sony Corp.",
    "Sony Ericsson Mobile Communications":                      "Sony Corp.",
    "Sony Bmg Music Entertainment":                             "Sony Corp.",
    "Sony Electrionics, Inc.":                                  "Sony Corp.",
    "Sony Mobile Communications, AB":                           "Sony Corp.",
    "Sony Electronics, Inc. d/b/a Sony Corporation of America": "Sony Corp.",
    "Sony Semiconductor, Corp.":                                "Sony Corp.",
    "Sony Ericsson Mobile Communications USA, Inc.":            "Sony Corp.",
    "Sony Semiconductor Kyushu Corporation, Ltd.":              "Sony Corp.",
    "Sony Chemicals Corporation of America":                    "Sony Corp.",
    "Sony Chemicals, Corp.":                                    "Sony Corp.",
    "SONY Corporation of America":                              "Sony Corp.",
    "Sony BMG Music Entertainment, Inc.":                       "Sony Corp.",
    "Sony/ATV Songs, LLC":                                      "Sony Corp.",
    "Sony Ericsson Mobile Communications U.S.A., Inc.":         "Sony Corp.",
    "formerly known asSony Ericsson Mobile Communications (USA), Inc.": "Sony Corp.",
    "Sony Puerto Rico, Inc.":                                   "Sony Corp.",
    "Sony Tunes, Inc.":                                         "Sony Corp.",
    "Sony USA, Inc.":                                           "Sony Corp.",
    "Sony Pic Studios, Inc.":                                   "Sony Corp.",
    "Sony Chemicals Corporation Of America":                    "Sony Corp.",
    "Sony Corporation of America,":                             "Sony Corp.",
    "Sony BMG Music Entertainment GP, Inc.":                    "Sony Corp.",
    "Sony Pictures Home Entertainment, Inc.":                   "Sony Corp.",
    "SONY Computer Entertainment America, LLC":                 "Sony Corp.",
    "Sony Ericcson Mobile Communications AB":                   "Sony Corp.",
    "Sony Corp. Of America":                                    "Sony Corp.",
    "Sony Mobile Communications USA, Inc.":                     "Sony Corp.",
    "Sony Entertainment America, LLC":                          "Sony Corp.",
    "for Sony Corporation of America":                          "Sony Corp.",
    "Sony Corp Of America":                                     "Sony Corp.",
    "Sony Media Software":                                      "Sony Corp.",
    "Sony Mobile Communications (USA)":                         "Sony Corp.",
    "Sony Electronic Publishing, Co.":                          "Sony Corp.",
    "Sony Corporation of Japan":                                "Sony Corp.",
    "Third Party Sony Mobile Communications USA, Inc.":         "Sony Corp.",
    "also known asSony Ericsson Mobile Communications AB":      "Sony Corp.",
    "also known asSony Ericsson Mobile Communications (USA), Inc.": "Sony Corp.",
    "Sony Corporate Services, Inc.":                            "Sony Corp.",
    "Apotex USA, Inc.":                         "Apotex Inc.",
    "Apotex,Inc.":                              "Apotex Inc.",
    "Apotex Technologies, Inc.":                "Apotex Inc.",
    "Apotex Pharmaceutical Holdings, Inc.":     "Apotex Inc.",
    "Apotex Pharmachem India PVT, Ltd.":        "Apotex Inc.",
    "Apotex Pharmachem, Inc.":                  "Apotex Inc.",
    "a division of APOTEX, Inc.":               "Apotex Inc.",
    "Apotex Pharmaceutical Holding, Inc.":      "Apotex Inc.",
    "a Division of Apotex, Inc.":               "Apotex Inc.",
    "Nokia Inc.,":                                      "Nokia Corp.",
    "Nokia Corporation d/b/a Nokia, Inc.":              "Nokia Corp.",
    "Nokia Mobile Phones Manufacturing (USA), Inc.":    "Nokia Corp.",
    "Oy Nokia AB":                                      "Nokia Corp.",
    "Nokia America, Corp.":                             "Nokia Corp.",
    "FI-00045 Nokia Group":                             "Nokia Corp.",
    "Nokia Solutions And Networks Us, LLC":             "Nokia Corp.",
    "Nokia Capital, Inc.":                              "Nokia Corp.",
    "Nokia Apps, LLC":                                  "Nokia Corp.",
    "Nokia Solutions and Networks US, LLC":             "Nokia Corp.",
    "SUBSTITUTED for Defendant Nokia, Corp.":           "Nokia Corp.",
    "Novartis Ag":                                          "Novartis AG",
    "Novartis Vaccines and Diagnostics, Inc.":              "Novartis AG",
    "Novartis International Ag":                            "Novartis AG",
    "Novartis Seeds, Inc.":                                 "Novartis AG",
    "Novartis Ophthalmics, Inc.":                           "Novartis AG",
    "Novartis International Pharmaceuticals, Ltd.":         "Novartis AG",
    "Novartis Vaccines And Diagnostics, Inc.":              "Novartis AG",
    "Novartis Pharmacueticals, Corp.":                      "Novartis AG",
    "Novartis Animal Health US, Inc.":                      "Novartis AG",
    "Novartis Pharmaceuitcals, Corp.":                      "Novartis AG",
    "Novartis Pharmaceutical, Corp.":                       "Novartis AG",
    "Novartis Crop Protection, Inc.":                       "Novartis AG",
    "Novartis Nutrition, Corp.":                            "Novartis AG",
    "Novartis Pharmceuticals, Corp.":                       "Novartis AG",
    "Novartis Crop Protection AG":                          "Novartis AG",
    "Novartis Institutes for Biomedical Research":          "Novartis AG",
    "Novartis Nutrition, Inc.":                             "Novartis AG",
    "Novartis Seeds AG":                                    "Novartis AG",
    "Novartis Pharmaceuticals":                             "Novartis AG",
    "Novartis Vaccines And Diagnostics Srl":                "Novartis AG",
    "Novartis Institutes for Biomedical Research, Inc.":    "Novartis AG",
    "Novartis Institutes For Biomedical Research, Inc.":    "Novartis AG",
    "Novartis Institute for Biomedical Research":           "Novartis AG",
    "Novartis Pharmaceuticals Corporation, .":              "Novartis AG",
    "Astrazeneca LP":                               "AstraZeneca PLC",
    "Astrazeneca UK, Ltd.":                         "AstraZeneca PLC",
    "Astrazeneca, LP":                              "AstraZeneca PLC",
    "Astrazeneca, Lp":                              "AstraZeneca PLC",
    "Astrazeneca, L.P.":                            "AstraZeneca PLC",
    "Astrazeneca Plc":                              "AstraZeneca PLC",
    "Astrazeneca PLC":                              "AstraZeneca PLC",
    "AstraZeneca Biotech AB":                       "AstraZeneca PLC",
    "AstraZeneca Biotech Holdings, Inc.":           "AstraZeneca PLC",
    "AstraZeneca Biopharmaceuticals, Inc.":         "AstraZeneca PLC",
    "AstraZeneca Pharmaceuticals, L.P.":            "AstraZeneca PLC",
    "Toshiba America Information Systems, Inc.,":               "Toshiba Corp.",
    "Toshiba America Electronic Components, Inc.,":             "Toshiba Corp.",
    "Toshiba America Consumer Products, Inc.":                  "Toshiba Corp.",
    "Toshiba America, Inc.,":                                   "Toshiba Corp.",
    "Toshiba America Consumer Products, LLC.,":                 "Toshiba Corp.",
    "Toshiba America Consumer Products LLC.,":                  "Toshiba Corp.",
    "Toshiba Medical Systems Corporation, Inc.":                "Toshiba Corp.",
    "Toshiba America MRI, Inc.":                                "Toshiba Corp.",
    "Toshiba TEC America Retail Information Systems, Inc.":     "Toshiba Corp.",
    "Toshiba America Consumer Products,":                       "Toshiba Corp.",
    "Toshiba America MRI, Inc., A California Corporation,":     "Toshiba Corp.",
    "Toshiba America Consumer Products, Inc., et al":           "Toshiba Corp.",
    "Toshiba International":                                    "Toshiba Corp.",
    "Toshiba America Electronic Components":                    "Toshiba Corp.",
    "Toshiba American Information Services, Inc.":              "Toshiba Corp.",
    "Toshiba America Consumer Products":                        "Toshiba Corp.",
    "Toshiba America , Inc.":                                   "Toshiba Corp.",
    "formerly known asToshiba America, Inc.":                   "Toshiba Corp.",
    "Toshiba Tec, Corp.":                                       "Toshiba Corp.",
    "Inc. Toshiba America Elec":                                "Toshiba Corp.",
    "Inc. Toshiba America Info":                                "Toshiba Corp.",
    "Toshiba Consumer Products, Inc.":                          "Toshiba Corp.",
    "Toshiba America, Corp.":                                   "Toshiba Corp.",
    "Toshiba Medical Systems, Corp.":                           "Toshiba Corp.",
    "Toshiba Electronics Asia, Ltd.":                           "Toshiba Corp.",
    "Toshiba Corporation.,":                                    "Toshiba Corp.",
    "Toshiba Asia Pacific Pte., Ltd.":                          "Toshiba Corp.",
    "Toshiba Information Equipment (Philippines), Inc.":        "Toshiba Corp.",
    "Toshiba Of Canada, Ltd.":                                  "Toshiba Corp.",
    "- Toshiba America Consumer Products":                      "Toshiba Corp.",
    "Toshiba Materials Co., Ltd.":                              "Toshiba Corp.",
    "Toshiba Logistics America, Inc.":                          "Toshiba Corp.",
    "Blackberry USA":                                               "BlackBerry Ltd.",
    "BlackBerry Limited f/k/a Research in Motion, Ltd.":           "BlackBerry Ltd.",
    "Blackberry Commerce, Corp.":                                   "BlackBerry Ltd.",
    "BlackBerry Corporation (f/k/a Research In Motion Corporation)": "BlackBerry Ltd.",
    "Blackberry Limited (f//k/a Research in Motion Limited)":       "BlackBerry Ltd.",
    "Sanofi-Aventis Deutschland GmbH":                              "Sanofi S.A.",
    "Bristol-Myers Squibb Sanofi Pharmaceuticals Holding Partnership": "Sanofi S.A.",
    "Sanofi-Aventis Deutschland GMBH":                              "Sanofi S.A.",
    "Sanofi-Aventis, U.S., LLC":                                    "Sanofi S.A.",
    "Sanofi-Synthelobo, Inc.":                                      "Sanofi S.A.",
    "Bristol-Myers Squibb/Sanofi Pharmaceuticals Partnership":      "Sanofi S.A.",
    "Sanofi-Aventis, S.A.":                                         "Sanofi S.A.",
    "Bristol-Myers Squibb Sanofi-Pharmaceuticals Holding Partnership": "Sanofi S.A.",
    "Sanofi-Aventis, Inc.":                                         "Sanofi S.A.",
    "Sanofi Animal Health, Inc.":                                   "Sanofi S.A.",
    "Sanofi-aventis U.S., LLC":                                     "Sanofi S.A.",
    "Sanofi-Synthelabo, LLC":                                       "Sanofi S.A.",
    "Sanofi-Synthelabo SA":                                         "Sanofi S.A.",
    "Sanofi-Aventis SA":                                            "Sanofi S.A.",
    "Koninklijke Philips Electronics, N.V.":                "Koninklijke Philips N.V.",
    "Koninklijke Phillips Electronics NV":                  "Koninklijke Philips N.V.",
    "formerly known asKoninklijke Philips Electronics NV":  "Koninklijke Philips N.V.",
    "Koninklijke Philips Electronics N. V.":                "Koninklijke Philips N.V.",
    "Lupin Atlantis Holdings SA":           "Lupin Ltd.",
    "Lupin Pharmaceuticals,Inc.":           "Lupin Ltd.",
    "Lupin Atlantis Holding Sa":            "Lupin Ltd.",
    "Lupin Pharmaceutcials, Inc.":          "Lupin Ltd.",
    "Lupin Atlantis Holdings, S.A.":        "Lupin Ltd.",
    "Lupin Pharmaceuticals Usa, Inc.":      "Lupin Ltd.",
    "Lupin Pharmaceuticals":               "Lupin Ltd.",
    "Panasonic Corporation of North America,":                      "Panasonic Corp.",
    "Panasonic Electronic Devices Corporation of America":          "Panasonic Corp.",
    "Panasonic Communications Corporation of America":              "Panasonic Corp.",
    "Panasonic Personal Computer, Co.":                             "Panasonic Corp.",
    "Panasonic Mobile Communications Development Corporation of USA": "Panasonic Corp.",
    "Panasonic Division of Matsushita Electric Corporation of America": "Panasonic Corp.",
    "Panasonic Communications Co., Ltd.":                           "Panasonic Corp.",
    "Panasonic Semiconductor Discrete Devices Co., Ltd.":           "Panasonic Corp.",
    "Panasonic Corportion Of North America":                        "Panasonic Corp.",
    "Panasonic Broadcast & Television Systems, Co.":                "Panasonic Corp.",
    "Panasonic Industrial, Co.":                                    "Panasonic Corp.",
    "Panasonic Corporation Of North America, Inc.":                 "Panasonic Corp.",
    "Panasonic Mobile Communications Co., Ltd.":                    "Panasonic Corp.",
    "Panasonic EV Energy Co., Ltd.":                                "Panasonic Corp.",
    "Panasonic Company Central":                                    "Panasonic Corp.",
    "Panasonic AVC Networks, Co.":                                  "Panasonic Corp.",
    "Panasonic Corporation, Ltd.":                                  "Panasonic Corp.",
    "Panasonic Kabushiki Kaisha":                                   "Panasonic Corp.",
    "Panasonic Automotive Systems Company of America":              "Panasonic Corp.",
    "Panasonic EV Energy Company, Ltd.":                            "Panasonic Corp.",
    "Panasonic Communications Co., LTD.,":                          "Panasonic Corp.",
    "Panasonic Mobile Communications Co., LTD.,":                   "Panasonic Corp.",
    "Panasonic Digital Communications & Security, Co.":             "Panasonic Corp.",
    "Google, Inc.,":        "Google Inc.",
    "Google Voice, Inc.":   "Google Inc.",
    "Google,Inc.":          "Google Inc.",
    "Htc B.V.I.":                                           "HTC Corp.",
    "Htc America, Inc.":                                    "HTC Corp.",
    "Htc, Corp.":                                           "HTC Corp.",
    "High Tech Computer Corp. A/K/A Htc, Corp.":            "HTC Corp.",
    "also known asHTC, Corp.":                              "HTC Corp.",
    "HTC Corp., a Foreign, Corp.":                          "HTC Corp.",
    "HTC Products, Inc.":                                   "HTC Corp.",
    "High Tech Computer Corp (Htc)":                        "HTC Corp.",
    "Htc (B.V.I.)":                                         "HTC Corp.",
    "High Tech Computer Corp., a/k/a HTC, Corp.":           "HTC Corp.",
    "High Tech Computer, Corp.":    "HTC Corp.",
    "Abbott Respiratory, LLC":                          "Abbott Laboratories",
    "Abbott Bioresearch Center, Inc.":                  "Abbott Laboratories",
    "Abbott Biotechnology, Ltd.":                       "Abbott Laboratories",
    "Abbott GmbH & Co., KG":                            "Abbott Laboratories",
    "Abbott Molecular, Inc.":                           "Abbott Laboratories",
    "Abbott Gmbh & Co. Kg":                             "Abbott Laboratories",
    "Abbott Pharmaceuticals PR, Ltd.":                  "Abbott Laboratories",
    "Abbott Products, Inc.":                            "Abbott Laboratories",
    "Abbott Cardiovascular Systems, Inc.,":             "Abbott Laboratories",
    "Abbott Cardiovascular Systems Inc..":              "Abbott Laboratories",
    "Abbott Pharmaceutical, Corp.":                     "Abbott Laboratories",
    "Abbott Sales, Marketing & Distribution, Co.":      "Abbott Laboratories",
    "Abbott Point of Care, Inc.":                       "Abbott Laboratories",
    "Abbott Medical Optics, Inc.":                      "Abbott Laboratories",
    "Abbott of England (1981), Ltd.":                   "Abbott Laboratories",
    "Abbott Laboratories Inc.,":                        "Abbott Laboratories",
    "Abbott Nutrition":                                 "Abbott Laboratories",
    "Abbott Laboratories, The":                         "Abbott Laboratories",
    "Actavis Laboratories UT, Inc.":                "Actavis Inc.",
    "Actavis Laboratories Ut, Inc.":                "Actavis Inc.",
    "Actavis Group Hf":                             "Actavis Inc.",
    "Actavis Pharma Manufacturing Pvt.":            "Actavis Inc.",
    "Actavis Laboratories Fl., Inc.":               "Actavis Inc.",
    "Actavis Group hf":                             "Actavis Inc.",
    "Actavis Pharma Manufacturing Pvt, Ltd.":       "Actavis Inc.",
    "Actavis Pharma Manufacturing Private, Ltd.":   "Actavis Inc.",
    "Actavis US Holding, LLC":                      "Actavis Inc.",
    "Actavis Group PTC ehf":                        "Actavis Inc.",
    "Actavis, PLC.":                                "Actavis Inc.",
    "Actavis Group Ptc Ehf":                        "Actavis Inc.",
    "Actavis Mid-Atlantic, LLC":                    "Actavis Inc.",
    "Actavis Pharma,INC.":                          "Actavis Inc.",
    "Actavis,INC.":                                 "Actavis Inc.",
    "Reddy-Cheminor, Inc.":                 "Dr. Reddy's Laboratories",
    "Reddy Cheminor, Inc.":                 "Dr. Reddy's Laboratories",
    "GlaxoSmithKline Intellectual Property (No. 2), Ltd.":          "GlaxoSmithKline PLC",
    "Glaxosmithkline Biologicals S.A.":                             "GlaxoSmithKline PLC",
    "Glaxosmithkline LLP":                                          "GlaxoSmithKline PLC",
    "doing business asGlaxoSmithKline":                             "GlaxoSmithKline PLC",
    "GlaxoSmithkline, PLC":                                         "GlaxoSmithKline PLC",
    "GlaxoSmithKline Intellectual Property Management, Ltd.":       "GlaxoSmithKline PLC",
    "Glaxosmithkline, P.L.C.":                                      "GlaxoSmithKline PLC",
    "GlaxoSmithKline, Inc.":                                        "GlaxoSmithKline PLC",
    "Glaxosmithkline Biologicals, S.A.":                            "GlaxoSmithKline PLC",
    "GlaxoSmithKline plc":                                          "GlaxoSmithKline PLC",
    "Glaxosmithkline Consumer Healthcare GMBH & Co. KG":            "GlaxoSmithKline PLC",
    "Glaxosmithkline Consumer Healthcare, Lp":                      "GlaxoSmithKline PLC",
    "Glaxosmithkline Healthcare, LP":                               "GlaxoSmithKline PLC",
    "a Pennsylvania Corporationdoing business asGlaxosmithkline, Glaxo Wellcome, Inc.": "GlaxoSmithKline PLC",
    "a Pennsylvania corporationdoing business asGlaxoSmithKline":   "GlaxoSmithKline PLC",
    "Glaxosmithkline Biologicals Manufacturing S.A.":               "GlaxoSmithKline PLC",
    "Glaxosmithkline Capital, Inc.":                                "GlaxoSmithKline PLC",
    "Glaxo Group Ltd. d/b/a Glaxosmithkline":                       "GlaxoSmithKline PLC",
    "Smithkline Beecham Corp. d/b/a Glaxosmithkline":               "GlaxoSmithKline PLC",
    "Sandoz Canada, Inc.":              "Sandoz Inc.",
    "Sandoz Nutrition, Corp.":          "Sandoz Inc.",
    "Sandoz International GmbH":        "Sandoz Inc.",
    "SANDOZ INTERNATIONAL GmbH":        "Sandoz Inc.",
    "Sandoz GmbH":                      "Sandoz Inc.",
    "Sandoz Ag":                        "Sandoz Inc.",
    "Sandoz AG":                        "Sandoz Inc.",
    "Sandoz Gmbh":                      "Sandoz Inc.",
    "Sandoz, Ltd.":                     "Sandoz Inc.",
    "Sandoz, Corp.":                    "Sandoz Inc.",
    "Sandoz Pharmaceuticals, Corp.":    "Sandoz Inc.",
    "Sandoz Private, Ltd.":             "Sandoz Inc.",
    "Sandoz International Gmbh":        "Sandoz Inc.",
    "Sandoz Industrial Products S.A.":  "Sandoz Inc.",
    "Sprint Communications Company, LP":                "Sprint Corp.",
    "Sprint Communications Company LP":                 "Sprint Corp.",
    "Sprint Spectrum, LP":                              "Sprint Corp.",
    "Sprint/United Management, Co.":                    "Sprint Corp.",
    "Sprint Communication Company LP":                  "Sprint Corp.",
    "Sprint Spectrum L P":                              "Sprint Corp.",
    "Sprint Spectrum L.P. d/b/a Sprint PCS":            "Sprint Corp.",
    "Sprint International, Inc.,":                      "Sprint Corp.",
    "doing business as Sprint PCS":                     "Sprint Corp.",
    "Sprint Communications Co. LP":                     "Sprint Corp.",
    "Sprint Spectrum Equipment Company, L.P.":          "Sprint Corp.",
    "Sprintcom, Inc.":                                  "Sprint Corp.",
    "SprintCom, Inc.":                                  "Sprint Corp.",
    "Sprint Spectrum Lp":                               "Sprint Corp.",
    "Sprint Communications":                            "Sprint Corp.",
    "Sprint International, Inc.":                       "Sprint Corp.",
    "Sprint PCS Group":                                 "Sprint Corp.",
    "Sprint Wireless Broadband Company LLC n/k/a Clear Wireless Broadband, LLC": "Sprint Corp.",
    "d/b/a Sprint PCS":                                 "Sprint Corp.",
    "Sprint Communications, Co.":                       "Sprint Corp.",
    "Sprint Wireless Broadband Company, LLC":           "Sprint Corp.",
    "Sprint Nextell, Corp.":                            "Sprint Corp.",
    "Sprint Nextel Corp. D/B/A Sprint":                 "Sprint Corp.",
    "formerly known asSprint Communications, Inc.":     "Sprint Corp.",
    "Sprint FON Group":                                 "Sprint Corp.",
    "US Sprint Communications Company Limited Partnership": "Sprint Corp.",
    "Sprint Communication, Co.":                        "Sprint Corp.",
    "ZTE (USA) Inc.,":      "ZTE Corp.",
    "Huawei Investment & Holding Co., Ltd.":                    "Huawei Technologies",
    "Huawei Device (Hong Kong) Co., Ltd.":                      "Huawei Technologies",
    "Huawei Enterprise USA, Inc.":                              "Huawei Technologies",
    "Huawei Technologies Cooperatief U.A.":                     "Huawei Technologies",
    "Huawei Device Co, Ltd.":                                   "Huawei Technologies",
    "Huawei Technologies Company, Ltd.":                        "Huawei Technologies",
    "Futurewei Technologies, Inc. D/B/A Huawei Technologies (USA)": "Huawei Technologies",
    "Huawei Investment and Holding Co., Ltd.":                  "Huawei Technologies",
    "Huawei Technologies Cooperatif U.A.":                      "Huawei Technologies",
    "Huawei Devce, USA, Inc.":                                  "Huawei Technologies",
    "Huawei Technology Co., Ltd.":                              "Huawei Technologies",
    "Huawei Technologies (USA America)":                        "Huawei Technologies",
    "doing business asHuawei":                                  "Huawei Technologies",
    "Huawei America, Inc.,":                                    "Huawei Technologies",
    "Huawei Network USA, Inc.":                                 "Huawei Technologies",
    "Huawei USA R&D Center":                                    "Huawei Technologies",
    "Huawei North America":                                     "Huawei Technologies",
    "doing business asHuawei Technologies (USA)":               "Huawei Technologies",
    "Huawei Network Usa, Inc.":                                 "Huawei Technologies",
    "Huawei Technologies, USA, Inc.":                           "Huawei Technologies",
    "3M Purification, Inc.":                                    "3M Co.",
    "3M Unitek, Corp.":                                         "3M Co.",
    "Minnesota Mining and Manufacturing Company (3M)":          "3M Co.",
    "3M Cogent, Inc.":                                          "3M Co.",
    "Minnesota Mining & Manufacturing Co(3M)":                  "3M Co.",
    "3M Precision Optics, Inc.":                                "3M Co.",
    "3M Health Information Systems":                            "3M Co.",
    "3m Innovative Properties, Co.":                            "3M Co.",
    "3m, Co.":                                                  "3M Co.",
    "A Division of 3M, Co.":                                    "3M Co.",
    "Minnesota Mining and Manufacturing Company a/k/a 3M":      "3M Co.",
    "a Delaware Corporation having a principal place of business at 3M Center, St. Paul, Minnesota": "3M Co.",
    "3M, Minnesota Mining and Manufacturing, Co.":              "3M Co.",
    "3M":                                                       "3M Co.",
    "Boston Scientific Corporation,":                                                               "Boston Scientific Corp.",
    "Boston Scientific Technology, Inc.":                                                           "Boston Scientific Corp.",
    "Boston Scientific Scimed, Inc. formerly known as Scimed Life Systems, Inc.also known asScimed Life Systems, Inc.": "Boston Scientific Corp.",
    "Boston Scientific Sales, Inc.":                                                                "Boston Scientific Corp.",
    "Boston Scientific Scimed":                                                                     "Boston Scientific Corp.",
    "Kyocera Sanyo Telecom, Inc.":                              "Kyocera Corp.",
    "Kyocera Kabushiki Kaisha":                                 "Kyocera Corp.",
    "Kyocera Corporation,":                                     "Kyocera Corp.",
    "Kyocera Document Solutions Development America, Inc.":     "Kyocera Corp.",
    "Kyocera Electronic Devices, LLC":                          "Kyocera Corp.",
    "Kyocera Industrial Ceramics, Corp.":                       "Kyocera Corp.",
    "Kyocera Corporation (Japan)":                              "Kyocera Corp.",
    "Ronald A. Katz Technology Licensing, L.P.,":   "Ronald A. Katz Technology Licensing L.P.",
    "Ronald A. Katz Technology Licensing L P":       "Ronald A. Katz Technology Licensing L.P.",
    "Katz Technology, Ronald A":                     "Ronald A. Katz Technology Licensing L.P.",
    "Walmart Stores, Inc.":                     "Walmart Inc.",
    "WalMart Stores, Inc.":                     "Walmart Inc.",
    "Walmart.com USA, LLC":                     "Walmart Inc.",
    "a division of WalMart Stores, Inc.":       "Walmart Inc.",
    "Inc Walmart Stores":                       "Walmart Inc.",
    "Walmart Stores Inc a Delaware, Corp.":     "Walmart Inc.",
    "ASUS Computer International (America)":                    "ASUSTeK Computer Inc.",
    "Asus Technology Pte, Ltd.":                                "ASUSTeK Computer Inc.",
    "Asustek Computer (Suzhou) Co., Ltd.":                      "ASUSTeK Computer Inc.",
    "Asustech (Suzhou) Co., Ltd.":                              "ASUSTeK Computer Inc.",
    "ASUSTeK Computer International America":                   "ASUSTeK Computer Inc.",
    "Asustek Computer International America":                   "ASUSTeK Computer Inc.",
    "Asus Computer International Inc, a California, Corp.":     "ASUSTeK Computer Inc.",
    "Asus Cloud, Corp.":                                        "ASUSTeK Computer Inc.",
    "ASUSTek Computer International, Inc.":                     "ASUSTeK Computer Inc.",
    "Merck, Sharp & Dohme, Corp.":                              "Merck & Co.",
    "Merck Sharp & Dohme Pharmaceuticals Srl":                  "Merck & Co.",
    "Merck &, Co.":                                             "Merck & Co.",
    "Merck Frosst Canada &, Co.":                               "Merck & Co.",
    "Merck Sharp and Dohme, Corp.":                             "Merck & Co.",
    "Merck-Medco Managed Care, LLC":                            "Merck & Co.",
    "Merck And Co, Inc.":                                       "Merck & Co.",
    "Merck Canada, Inc.":                                       "Merck & Co.",
    "Merck Sharp & Dohme Pharmaceuticals":                      "Merck & Co.",
    "Merck-Frosst Center for Therapeutic Research":             "Merck & Co.",
    "Merck & Co., Ltd.":                                        "Merck & Co.",
    "Merck Sharp & Dohme Research, Ltd.":                       "Merck & Co.",
    "doing business asMerck Animal Health":                     "Merck & Co.",
    "Inc. Merck &, Co.":                                        "Merck & Co.",
    "Merck & Co.,Inc.":                                         "Merck & Co.",
    "Merck and Co, Inc.":                                       "Merck & Co.",

    "Merck Kgaa":                                               "Merck KGaA",
    "Merck Patent Gmbh":                                        "Merck KGaA",
    "Merck Patent GmbH":                                        "Merck KGaA",
    "Merck Serono S.A.":                                        "Merck KGaA",
    "Merck Patent Gesellschaft mit beschrankter Haftung":       "Merck KGaA",
    "Merck & Cie":                                              "Merck KGaA",
    "f/n/a Monsanto, Co.":          "Monsanto Co.",
    "Monsanto Ag Products, LLC":    "Monsanto Co.",
    "Monsanto Company, Inc.":       "Monsanto Co.",
    "Monsanto Chemical, Co.":       "Monsanto Co.",
    "Monsanto, Corp.":              "Monsanto Co.",
    "Target National Bank":                         "Target Corp.",
    "Target Bank":                                  "Target Corp.",
    "Target Stores, Inc.":                          "Target Corp.",
    "Target Brands, Inc.":                          "Target Corp.",
    "Target Stores":                                "Target Corp.",
    "Target Corporation, Inc.":                     "Target Corp.",
    "Target Department Stores, Inc.":               "Target Corp.",
    "doing business asTarget Stores":               "Target Corp.",
    "Dayton Hudson Corporation Target Stores":      "Target Corp.",
    "Target Corporation and Doe Corporations 1-10": "Target Corp.",
    "fictitious names for corporations unknown to Plaintiffs, who supply Target with its Trutech brand MPEG-2-complaint products": "Target Corp.",
    "Ranbaxy Pharmaceuticals,Inc.,":            "Ranbaxy Laboratories",
    "Ranbaxy laboratories, Ltd.":               "Ranbaxy Laboratories",
    "Ranbaxy Pharmaceutials, Inc.":             "Ranbaxy Laboratories",
    "Ranbaxy Schein Pharmaceuticals, LLC":      "Ranbaxy Laboratories",
    "Ranbaxy Schein Pharma, LLC":               "Ranbaxy Laboratories",
    "Ranbaxy Laboratories,Ltd.":                "Ranbaxy Laboratories",
    "Ranbaxy Holdings (Uk), Ltd.":              "Ranbaxy Laboratories",
    "ArrivalStar SA":           "ArrivalStar S.A.",
    "Arrivalstar S.A.":         "ArrivalStar S.A.",
    "ArrivalStar S A":          "ArrivalStar S.A.",
    "ArrivalStar s.a.":         "ArrivalStar S.A.",
    "Arrivalstar SA":           "ArrivalStar S.A.",
    "Arrivalstar USA, Corp.":   "ArrivalStar S.A.",
    "Arrivalstar, Inc.":        "ArrivalStar S.A.",
    "Arrival Star, Inc.":           "ArrivalStar S.A.",
    "Arrival Star S.A.":            "ArrivalStar S.A.",
    "Arrival Star, SA":             "ArrivalStar S.A.",
    "Arrival Star SA":              "ArrivalStar S.A.",
    "Arrival Star (Jersey), Ltd.":  "ArrivalStar S.A.",
    "Arrival Star S. A.":           "ArrivalStar S.A.",
    "Arrival Stars SA":             "ArrivalStar S.A.",
    "Fujitsu Microelectronics America, Inc.":                           "Fujitsu Ltd.",
    "Fujitsu PC, Corp.":                                                "Fujitsu Ltd.",
    "Fujitsu Frontech North America, Inc.":                             "Fujitsu Ltd.",
    "Fujitsu Hitachi Plasma Display, Ltd.":                             "Fujitsu Ltd.",
    "Fujitsu Computer Systems Corp.,":                                  "Fujitsu Ltd.",
    "Fujitsu Media Devices, Ltd.":                                      "Fujitsu Ltd.",
    "formerly known asFujitsu Hitachi Plasma Display, Ltd.":            "Fujitsu Ltd.",
    "Fujitsu Personal Systems":                                         "Fujitsu Ltd.",
    "Fujitsu General America, Inc.":                                    "Fujitsu Ltd.",
    "Fujitsu Business Communication Systems, Inc.":                     "Fujitsu Ltd.",
    "Fujitsu Ten Corp. of America, Inc.":                               "Fujitsu Ltd.",
    "Fujitsu Computer Systems":                                         "Fujitsu Ltd.",
    "Fujitsu Computer Products of America, Inc.,":                      "Fujitsu Ltd.",
    "Fujitsu America, Inc.,":                                           "Fujitsu Ltd.",
    "Fujitsu Computer Products of America":                             "Fujitsu Ltd.",
    "Fujitsu GTE Business Systems, Inc.":                               "Fujitsu Ltd.",
    "Fujitsu Pc, Corp.":                                                "Fujitsu Ltd.",
    "Fujitsu Display Technologies, Corp.":                              "Fujitsu Ltd.",
    "Fujitsu-Siemens Computers, Inc.":                                  "Fujitsu Ltd.",
    "Fujitsu-Siemens Computers, LLC":                                   "Fujitsu Ltd.",
    "Fujitsu General, Ltd.":                                            "Fujitsu Ltd.",
    "Fujitsu Ten Corporation of America,":                              "Fujitsu Ltd.",
    "Fujitsu America, Inc., n/k/a Fujitsu Management Services of America, Inc.": "Fujitsu Ltd.",
    "Fujitsu Components America, Inc.":                                 "Fujitsu Ltd.",
    "Fujitsu Ten. Corp. of America, Inc.":                              "Fujitsu Ltd.",
    "Fujitsu Siemens Computers GmbH":                                   "Fujitsu Ltd.",
    "Fujitsu America":                                                  "Fujitsu Ltd.",
    "Fujitsu-siemens computers, Inc.":                                  "Fujitsu Ltd.",
    "Fujitsu-siemens computers, LLC":                                   "Fujitsu Ltd.",
    "Fujitsu america, Inc.":                                            "Fujitsu Ltd.",
    "Fujitsu Ten, Ltd.":                                                "Fujitsu Ltd.",
    "Fujitsu Micoelectronics, Inc.":                                    "Fujitsu Ltd.",
    "Fujitsu General America, Corp.":                                   "Fujitsu Ltd.",
    "Fujitsu (Thailand) Co, Ltd.":                                      "Fujitsu Ltd.",
    "Fujitsu Computer Products Of America, Inc.":                       "Fujitsu Ltd.",
    "Fujitsu General America, Inc.,":                                   "Fujitsu Ltd.",
    "Fujitsu Siemens Computers, Inc.":                                  "Fujitsu Ltd.",
    "Fujitsu Network Transmission Systems, Inc.":                       "Fujitsu Ltd.",
    "Fujitsu Ten Corporation of America":                               "Fujitsu Ltd.",
    "Fujitsu It Holdings, Inc.":                                        "Fujitsu Ltd.",
    "Fujitsu Limited and Futijsu Media Devices, Ltd.":                  "Fujitsu Ltd.",
    "Fujitsu Japan, Ltd.":                                              "Fujitsu Ltd.",
    "Fujitu, Ltd.":     "Fujitsu Ltd.",
    "Fujifilm, Corp.":                                          "Fujifilm Corp.",
    "Fujifilm North America, Corp.":                            "Fujifilm Corp.",
    "Fuji Photo Film USA, Inc.":                                "Fujifilm Corp.",
    "Fujifilm Holdings, Corp.":                                 "Fujifilm Corp.",
    "Fuji Photo Film U.S.A., Inc.":                             "Fujifilm Corp.",
    "Fuji Photo Film Co., Ltd.":                                "Fujifilm Corp.",
    "Fujifilm U.S.A., Inc.":                                    "Fujifilm Corp.",
    "Fujifilm Holdings America, Corp.":                         "Fujifilm Corp.",
    "Fujinon, Corp.":                                           "Fujifilm Corp.",
    "Fujifilm Medical Systems U.S.A., Inc.":                    "Fujifilm Corp.",
    "FUJIFILM Holdings America, Corp.":                         "Fujifilm Corp.",
    "FUJIFILM, Corp.":                                          "Fujifilm Corp.",
    "Fujifilm Recording Media U.S.A., Inc.":                    "Fujifilm Corp.",
    "Fuji Photo Film, Co.":                                     "Fujifilm Corp.",
    "Fujifilm USA, Inc.":                                       "Fujifilm Corp.",
    "FujiFilm America, Inc.":                                   "Fujifilm Corp.",
    "Fujifilm Usa":                                             "Fujifilm Corp.",
    "Fujifilm America, Inc.":                                   "Fujifilm Corp.",
    "Fujifilm North America f/k/a Fujifilm U.S.A., Inc.":       "Fujifilm Corp.",
    "Fuji Photo Film Co. LTD.,":                                "Fujifilm Corp.",
    "Fuji America, Inc.":                                       "Fujifilm Corp.",
    "Fuji Health Science, Inc.":                                "Fujifilm Corp.",
    "Fujifilm Electronic Materials U.S.A., Inc.":               "Fujifilm Corp.",
    "Fuji Photo Film, Ltd.":                                    "Fujifilm Corp.",
    "Fujifilm Life Science, a New York, Corp.":                 "Fujifilm Corp.",
    "Fujifilm Manufacturing USA, Inc.":                         "Fujifilm Corp.",
    "Fujifilm Sericol USA, Inc.":                               "Fujifilm Corp.",
    "Fuji Medical Systems Usa, Inc.":                           "Fujifilm Corp.",
    "Fuji Photo File Company, Ltd.":                            "Fujifilm Corp.",
    "Fujifilm Usa, Inc.":                                       "Fujifilm Corp.",
    "Fuji America, Corp.":                                      "Fujifilm Corp.",
    "Fuji America Corporaton,":                                 "Fujifilm Corp.",
    "Fujifilm America, In":                                     "Fujifilm Corp.",
    "Fujifilm America, Inc.,":                                  "Fujifilm Corp.",
    "IBM, Corp.":                                           "IBM Corp.",
    "Ibm":                                                  "IBM Corp.",
    "IBM Telecommunications Holdings, Inc.":                "IBM Corp.",
    "International Business Machines Corporation (IBM)":    "IBM Corp.",
    "Ibm, Corp.":                                           "IBM Corp.",
    "doing business asIBM, Corp.":                          "IBM Corp.",
    "International Business Machine, Corp.":                "IBM Corp.",
    "International Business MacHines, Corp.":               "IBM Corp.",
    "International Business Machines, Inc.":                "IBM Corp.",
    "also known asInternational Business Machines, Corp.":  "IBM Corp.",
    "International business machines, Corp.":               "IBM Corp.",
    "International Business Machines, Co.":                 "IBM Corp.",
    "International Business Machines":                      "IBM Corp.",
    "For an Order Pursuant to 28 U.S.C. ? 1782 Granting Leave to Obtain Discovery from Qualcomm Incorporated for Use in Foreign Proceedings.": "Qualcomm Inc.",
    "Qualcomm Personal Electronics":                        "Qualcomm Inc.",
    "Qualcomm, Inc. - Non-Party":                           "Qualcomm Inc.",
    "QUALCOMM Personal Electronics":                        "Qualcomm Inc.",
    "Subpoena on Qualcomm, Inc.":                           "Qualcomm Inc.",
    "Qualcomm CDMA Technologies Asia Pacific PTE, Ltd.":    "Qualcomm Inc.",
    "Non-Party Qualcomm, Inc.":                             "Qualcomm Inc.",
    "Qualcomm Techonologies, Inc.":                         "Qualcomm Inc.",
    "Interdigital Technology, Corp.":       "InterDigital Inc.",
    "InterDigital Technology, Corp.":       "InterDigital Inc.",
    "Interdigital Communications, Corp.":   "InterDigital Inc.",
    "InterDigital Holdings, Inc.":          "InterDigital Inc.",
    "InterDigital Communications, Inc.":    "InterDigital Inc.",
    "Interdigital Communications, LLC":     "InterDigital Inc.",
    "InterDigital Communications, LLC":     "InterDigital Inc.",
    "Interdigital Patents, Corp.":          "InterDigital Inc.",
    "Ericsson Inc.,":                                       "Ericsson AB",
    "Ericsson Radio Systems, Inc.":                         "Ericsson AB",
    "Ericsson Network Systems, Inc.":                       "Ericsson AB",
    "Ericsson-GE Mobile Communications, Inc.":              "Ericsson AB",
    "Telefonaktiebolaget LM Ericsson, Inc.":                "Ericsson AB",
    "Ericsson Television, Inc.":                            "Ericsson AB",
    "Ericsson GE Mobile Communications, Inc.":              "Ericsson AB",
    "Ericsson Messaging Systems, Inc.":                     "Ericsson AB",
    "Ericsson Mobile Communications":                       "Ericsson AB",
    "Ericsson, Corp.":                                      "Ericsson AB",
    "Ericsson Technical Services, Inc.":                    "Ericsson AB",
    "Ericsson Holding II, Inc.":                            "Ericsson AB",
    "Ericsson Canada, Inc.":                                "Ericsson AB",
    "Ericsson Radio Systems, BV":                           "Ericsson AB",
    "Ericsson Components AB":                               "Ericsson AB",
    "Ericsson Radio Systems AB":                            "Ericsson AB",
    "Sweden and Ericsson, Inc.":                            "Ericsson AB",
    "Non-Party Ericsson, Inc.":                             "Ericsson AB",
    "Media Broadcom, LLC":                              "Broadcom Corp.",
    "Broadcom Homemetworking, Inc.":                    "Broadcom Corp.",
    "Broadcom Homenetworking, Inc.":                    "Broadcom Corp.",
    "Non-Party Broadcom, Corp.":                        "Broadcom Corp.",
    "Cisco Technology, Inc.":                   "Cisco Systems Inc.",
    "Cisco-Linksys, LLC":                       "Cisco Systems Inc.",
    "CISCO Systems, Inc.":                      "Cisco Systems Inc.",
    "Cisco Consumer Products, LLC":             "Cisco Systems Inc.",
    "Cisco Webex, LLC":                         "Cisco Systems Inc.",
    "Cisco Ironport Systems, LLC":              "Cisco Systems Inc.",
    "Cisco IronPort Systems, LLC":              "Cisco Systems Inc.",
    "Cisco Linksys, LLC":                       "Cisco Systems Inc.",
    "CISCO Technology, Inc.":                   "Cisco Systems Inc.",
    "Cisco System, Inc.":                       "Cisco Systems Inc.",
    "Cisco Sytems, Inc.":                       "Cisco Systems Inc.",
    "Cisco Systems, Inc.,":                     "Cisco Systems Inc.",
    "Cisco Systems (USA) Pte., Ltd.":           "Cisco Systems Inc.",
    "Cisco Systems International BV":           "Cisco Systems Inc.",
    "Cisco Systems,inc (ctr-clm Pltf)":         "Cisco Systems Inc.",
    "Cisco Technology,inc(ctr-clm Plt":         "Cisco Systems Inc.",
    "Cisco Systems (Puerto Rico), Corp.":       "Cisco Systems Inc.",
    "Intel Corporation, Inc.":      "Intel Corp.",
    "Intel, Inc.":          "Intel Corp.",
    "Nonparty Intel, Corp.": "Intel Corp.",
    "Maxim Integrated Products,Inc.":      "Maxim Integrated Products, Inc.",
    "Maxim Integrated Products":           "Maxim Integrated Products, Inc.",
    "Maxim Integrated Products, Corp.":    "Maxim Integrated Products, Inc.",
    "Maxim Integrated Prod, Inc.":         "Maxim Integrated Products, Inc.",
    "BellSouth Intellectual Property, Corp.":   "AT&T Inc.",
    "Bellsouth Intellectual Property Corp.,":   "AT&T Inc.",
    "Bayer Intellectual Property GMBH":         "Bayer AG",
    "Bayer Intellectual Property GmbH":         "Bayer AG",
    "Bayer Intellectual Property Gmbh":         "Bayer AG",
    "Intellectual Ventures Ii, LLC":            "Intellectual Ventures II, LLC",
    "Intellecutal ventures I, LLC":             "Intellectual Ventures I, LLC",
    "Intellictual Venture, LLC":                "Intellectual Ventures I, LLC",
    "Intellectual Ventures Fund 83, LLC":       "Intellectual Ventures II, LLC",
    "Intellectual Ventures, LLC":               "Intellectual Ventures II, LLC",
    "Intellectual Ventures Funding, LLC":       "Intellectual Ventures II, LLC",
    "Johnson & Johnson Professional, Inc.":                     "Johnson & Johnson",
    "Johnson and Johnson":                                      "Johnson & Johnson",
    "Johnson & Johnson Services, Inc.":                         "Johnson & Johnson",
    "Johnson & Johnson Interventional Systems, Co.":            "Johnson & Johnson",
    "Johnson & Johnson Interventional Systems, Inc.":           "Johnson & Johnson",
    "Johnson & Johnson Medical, Inc.":                          "Johnson & Johnson",
    "Johnson & Johnson, Corp.":                                 "Johnson & Johnson",
    "Johnson & Johnson Orthopaedics, Inc.":                     "Johnson & Johnson",
    "Johnson and Johnson, Inc.":                                "Johnson & Johnson",
    "Johnson & Johnson Consumer Products Company, Inc.":        "Johnson & Johnson",
    "Johnson & Johnson Consumer Co, Inc.":                      "Johnson & Johnson",
    "Johnson & Johnson, Co.":                                   "Johnson & Johnson",
    "Johnson & Johnson Consumer Companies, Inc.":               "Johnson & Johnson",
    "Johnson & Johnson Interventional Systems":                 "Johnson & Johnson",
    "Johnson & Johnson Products, Inc.":                         "Johnson & Johnson",
    "Johnson & Johnson Consumer, Inc.":                         "Johnson & Johnson",
    "Johnson & Johnson Development, Corp.":                     "Johnson & Johnson",
    "Johnson & Johnson Pharmaceutical Research & Develpment, LLC": "Johnson & Johnson",
    "a Johnson & Johnson, Co.":                                 "Johnson & Johnson",
    "Johnson & Johnson Consumer Products, Inc.":                "Johnson & Johnson",
    "a division of Johnson & Johnson Medical, Inc., a New Jersey, Corp.": "Johnson & Johnson",
    "Johnson & Johnson Hospital Services, Inc.":                "Johnson & Johnson",
    "' Johnson & Johnson Consumer Companies, Inc.":             "Johnson & Johnson",
    "Roche Molecular Systems":                                  "Roche Holding AG",
    "F. Hoffmann-LaRoche, Ltd.":                                "Roche Holding AG",
    "Roche Diagnostic Systems, Inc.":                           "Roche Holding AG",
    "Hoffmann-LaRoche, Inc.":                                   "Roche Holding AG",
    "Roche Palo Alto Lc":                                       "Roche Holding AG",
    "Roche Biomedical Laboratories, Inc.":                      "Roche Holding AG",
    "Roche Holdings, Inc.":                                     "Roche Holding AG",
    "Roche Diagnostics GmbH":                                   "Roche Holding AG",
    "Roche Diagnostics GMBH":                                   "Roche Holding AG",
    "Hoffmann LaRoche, Inc.":                                   "Roche Holding AG",
    "Roche NimbleGen, Inc.":                                    "Roche Holding AG",
    "F. Hoffman-La Roche AG":                                   "Roche Holding AG",
    "Hoffman-Laroche, Inc.":                                    "Roche Holding AG",
    "Hoffmann-LA Roche, Inc.":                                  "Roche Holding AG",
    "Roche Holding, Ltd.":                                      "Roche Holding AG",
    "Roche Colorado, Corp.":                                    "Roche Holding AG",
    "E. Hoffmann-La Roche, Ltd.":                               "Roche Holding AG",
    "Roche Vitamins, Inc.":                                     "Roche Holding AG",
    "Roche Vitamins, Ltd.":                                     "Roche Holding AG",
    "F. Hoffmann-La Roche LTD, Roche Diagnostics GmbH, and Hoffmann-La Roche, Inc.": "Roche Holding AG",
    "F Hoffman-La Roche, Inc.":                                 "Roche Holding AG",
    "F. Hoffmann-La Roche AG":                                  "Roche Holding AG",
    "'Roche Diagnostics, Corp.":                                "Roche Holding AG",
    "Roche Diagnostics Corporation d/b/a Roche Diagnostics":    "Roche Holding AG",
    "Hoffmann-La Roche, Ltd.":                                  "Roche Holding AG",
    "F. Hoffman-La Roche, Ltd.":                                "Roche Holding AG",
    "Roche Diagnostics, GmbH":                                  "Roche Holding AG",
    "Roche Diagnosics, Corp.":                                  "Roche Holding AG",
    "Bayer Ag":                                         "Bayer AG",
    "Bayer Cropscience LP":                             "Bayer AG",
    "Bayer Cropscience, Lp":                            "Bayer AG",
    "Bayer BioScience N.V.":                            "Bayer AG",
    "Bayer Healthcare, LLC,":                           "Bayer AG",
    "Bayer Healthcare AG":                              "Bayer AG",
    "Bayer S.A.S.":                                     "Bayer AG",
    "Bayer Pharmaceuticals, Corp.":                     "Bayer AG",
    "Bayer Schering Pharma Ag":                         "Bayer AG",
    "Bayer HealthCare, LLC":                            "Bayer AG",
    "Bayer CropScience AG":                             "Bayer AG",
    "Bayer Healthcare Pharamaceuticals, Inc.":          "Bayer AG",
    "Bayer Pharma Ag":                                  "Bayer AG",
    "Bayer BioScience N V":                             "Bayer AG",
    "Bayer SAS":                                        "Bayer AG",
    "Bayer Healthcare Pharmaceuticals , Inc.":          "Bayer AG",
    "Bayer AG,":                                        "Bayer AG",
    "Bayer CropScience S.A.":                           "Bayer AG",
    "Bayer Cropscience Ag":                             "Bayer AG",
    "Bayer Properties, LLC":                            "Bayer AG",
    "Bayer Ag, Bayer, Corp.":                           "Bayer AG",
    "Bayer Corporation/Consumer Care Division":         "Bayer AG",
    "Bayer Aktiengescellschaft, Corp.":                 "Bayer AG",
    "Bayer Cropscience AG":                             "Bayer AG",
    "Bayer Cropscience NV":                             "Bayer AG",
    "Bayer Bioscience NV":                              "Bayer AG",
    "Bayer Corporation, successor to Miles, Inc.":      "Bayer AG",
    "Bayer CropScience, S.A.":                          "Bayer AG",
    "BAYER CROPSCIENCE (New Jersey), Inc.":             "Bayer AG",
    "Bayer Pharma Chemicals, Inc.":                     "Bayer AG",
    "Bayer Materialscience, LLC":                       "Bayer AG",
    "Bayer Animal Health GMBH":                         "Bayer AG",
    "Bayer Cropscience Holdings, Inc.":                 "Bayer AG",
    "Bristol-Myers Squibb and, Co.":                    "Bristol-Myers Squibb Co.",
    "Bristol-Myers Squibb &, Co.":                      "Bristol-Myers Squibb Co.",
    "Bristol Myers Squibb Pharma, Co.":                 "Bristol-Myers Squibb Co.",
    "Bristol Myers Squibb Medical Imaging, Inc.":       "Bristol-Myers Squibb Co.",
    "Bristol-Meyers Squibbg, Co.":                      "Bristol-Myers Squibb Co.",
    "Bristol-Myers, Inc.":                              "Bristol-Myers Squibb Co.",
    "Bristol Meyers Squibb, Co.":                       "Bristol-Myers Squibb Co.",
    "Bristol-Myers Squibb":                             "Bristol-Myers Squibb Co.",
    "Melvino Technologies, Ltd.":   "ArrivalStar S.A.",
    "Melvino Technologies, Inc.":   "ArrivalStar S.A.",
    "Melvino Technology, Ltd.":     "ArrivalStar S.A.",
    "Shipping and Transit, LLC":    "ArrivalStar S.A.",
    "Pepsico, Inc.":                                 "PepsiCo, Inc.",
    "PepsiCo, Inc.":                                 "PepsiCo, Inc.",
    "Pepsi-Cola, Co.":                               "PepsiCo, Inc.",
    "A Division of Pepsico, Inc.":                   "PepsiCo, Inc.",
    "The Pepsi Bottling Group, Inc.":                "PepsiCo, Inc.",
    "Pepsiamericas, Inc.":                           "PepsiCo, Inc.",
    "Pepsi Bottling Ventures, LLC":                  "PepsiCo, Inc.",
    "Pepsi-Cola Metropolitan Bottling Company, Inc.":"PepsiCo, Inc.",
    "Pepsico Wine & Spirits, Co.":                   "PepsiCo, Inc.",
    "Pepsi Bottling Group, Inc.":                    "PepsiCo, Inc.",
    "Pepsi Cola, Co.":                               "PepsiCo, Inc.",
    "Pepsi-Cola Technical Operations, Inc.":         "PepsiCo, Inc.",
    "CVS Pharmacy Inc.":                             "CVS Health, Corp.",
    "CVS Caremark, Corp.":                           "CVS Health, Corp.",
    "Cvs Caremark, Corp.":                           "CVS Health, Corp.",
    "CVS Distribution, Inc.":                        "CVS Health, Corp.",
    "CVS Meridian, Inc.":                            "CVS Health, Corp.",
    "CVS Caremark Corporation, f/k/a CVS, Corp.":    "CVS Health, Corp.",
    "Cvs Caremark Corp. D/B/A Cvs D/B/A Cvs.Com":    "CVS Health, Corp.",
    "CVS of Virginia, Inc.":                         "CVS Health, Corp.",
    "CVS Pharmacy, Inc. (Identified as CVS Corporation)":   "CVS Health, Corp.",
    "CVS Transportation, LLC":                       "CVS Health, Corp.",
    "Toyota Motor Corp.":                            "Toyota Motor Corp.",
    "Toyota Motor Sales U.S.A., Inc.":               "Toyota Motor Corp.",
    "Toyota Motor Manufacturing Mississippi, Inc.":  "Toyota Motor Corp.",
    "Toyota Motor Manufacturing Kentucky, Inc.":     "Toyota Motor Corp.",
    "Toyota Motor Manufacturing Texas, Inc.":        "Toyota Motor Corp.",
    "Toyota Motor Manufacturing Indiana, Inc.":      "Toyota Motor Corp.",
    "Toyota Motor Manufacturing Alabama, Inc.":      "Toyota Motor Corp.",
    "Toyota Motor Manufacturing West Virgina, Inc.": "Toyota Motor Corp.",
    "Toyota Motor Engineering & Manufacturing":      "Toyota Motor Corp.",
    "Toyota Motor North America, Inc.":              "Toyota Motor Corp.",
    "Toyota Financial Savings Bank":                 "Toyota Motor Corp.",
    "Toyota Tsusho America, Inc.":                   "Toyota Motor Corp.",
    "Toyota Technical Center":                       "Toyota Motor Corp.",
    "Gulf States Toyota, Inc.":                      "Gulf States Toyota, Inc.",
    "Southeast Toyota Distributors, LLC":            "Southeast Toyota Distributors, LLC",
    "Lafontaine Toyota":                             "Lafontaine Toyota",
    "Kinsel Toyota":                                 "Kinsel Toyota",
    "Toyota Motor Corp.":                                     "Toyota Motor Corp.",
    "Toyota Motor Sales, USA, Inc.":                          "Toyota Motor Corp.",
    "Toyota Motor Sales, Usa, Inc.":                          "Toyota Motor Corp.",
    "Gulf States Toyota, Inc.":                               "Toyota Motor Corp.",
    "Toyota Motor Manufacturing, Indiana, Inc.":              "Toyota Motor Corp.",
    "Toyota Motor Manufacturing, Texas, Inc.":                 "Toyota Motor Corp.",
    "Toyota Motor Sales Usa, Inc.":                           "Toyota Motor Corp.",
    "Southeast Toyota Distributors, LLC":                     "Toyota Motor Corp.",
    "Toyota":                                                 "Toyota Motor Corp.",
    "Toyota Motor Sales North America":                       "Toyota Motor Corp.",
    "sued as Toyota Motor, Corp.":                            "Toyota Motor Corp.",
    "Toyota Motor Sales, U. S. A., Inc.":                     "Toyota Motor Corp.",
    "Toyota Motor Engineering & Manufacturing North America": "Toyota Motor Corp.",
    "Toyota Motor Sales, U.S.A.":                             "Toyota Motor Corp.",
    "Toyota Motor Manufacturing, U.S.A., Inc.":               "Toyota Motor Corp.",
    "Intellectual Poperty DIV 1 Toyota":                      "Toyota Motor Corp.",
    "Toyota Motor CO, Ltd.":                                  "Toyota Motor Corp.",
    "Lafontaine Toyota":                                      "Toyota Motor Corp.",
    "Toyota Motor Corporate Service of North America, Inc.":  "Toyota Motor Corp.",
    "dba Alford Toyota":                                      "Toyota Motor Corp.",
    "A Division of Toyota Motor Sales, USA, Inc.":            "Toyota Motor Corp.",
    "Toyota Caelum, Inc.":                                    "Toyota Motor Corp.",
    "Toyota Caelum USA, Inc.":                                "Toyota Motor Corp.",
    "Marshall Toyota":                                        "Toyota Motor Corp.",
    "Toyota of Longview":                                     "Toyota Motor Corp.",
    "Toyota Motor Sales U S A, Inc.":                         "Toyota Motor Corp.",
    "Toyota Motor North America, Inc. et al":                 "Toyota Motor Corp.",
    "Toyota North America, Inc.":                             "Toyota Motor Corp.",
    "Toyota Motor Distributors, Inc.":                        "Toyota Motor Corp.",
    "d/b/a Smart Motors Toyota":                              "Toyota Motor Corp.",
    "Toyota Motor Sales, USA":                                "Toyota Motor Corp.",
    "Kinsel Toyota":                                          "Toyota Motor Corp.",
    "Clearwater Toyota":                                      "Toyota Motor Corp.",
    "Toyota Motor Engineering & Manufacturing North America, Inc.,": "Toyota Motor Corp.",
    "Costco Wholesale Corp.":           "Costco Wholesale Corp.",
    "Costco Wholesale Corp.,":          "Costco Wholesale Corp.",
    "Costco Wholesale, Inc.":           "Costco Wholesale Corp.",
    "Costco Wholesale Membership, Inc.":"Costco Wholesale Corp.",
    "CostCo Wholesale, Corp.":          "Costco Wholesale Corp.",
    "Costco Company, Inc.":             "Costco Wholesale Corp.",
    "Nvidia, Corp.":             "Nvidia, Corp.",
    "NVIDIA, Corp.":             "Nvidia, Corp.",
    "nVidia, Corp.":             "Nvidia, Corp.",
    "NVIDIA Singapore Pte, Ltd.": "Nvidia, Corp.",
    "Advanced Micro Devices, Inc.,":                      "Advanced Micro Devices, Inc.",
    "Advanced Micro Devices, Inc., A Delaware, Corp.":    "Advanced Micro Devices, Inc.",
    "Advanced Micro Devices, Inc., A California, Corp.":  "Advanced Micro Devices, Inc.",
    "AMD Inc.": "Advanced Micro Devices, Inc.",
    "Nintendo Co. Ltd.":         "Nintendo Co., Ltd.",
    "Nintendo Corp. of America": "Nintendo Co., Ltd.",
    "Nintendo CO., Ltd.":        "Nintendo Co., Ltd.",
    "Nintendo, Co.":             "Nintendo Co., Ltd.",
    "Nintendo America, Inc.":    "Nintendo Co., Ltd.",
    
    "Oakley, Inc.":       "Oakley, Inc.",
    "Oakley":             "Oakley, Inc.",
    "Oakley Sales, Corp.": "Oakley, Inc.",
    "Oakley Direct, Inc.": "Oakley, Inc.",
    "doing business asBest Buy Co., Inc.": "Best Buy, Inc.",
    "Shanghai Lenovo Electronic Co., Ltd.": "Lenovo, Inc.",
    # ── US Government remaining variants ──────────────────
    "U.S. Attorney UNITED STATES OF AMERICA":                   "US Government",
    "United States UNITED STATES OF AMERICA":                   "US Government",
    "The United States":                                        "US Government",
    "Attorney General of the United States":                    "US Government",
    "Commerce Department of the United States":                 "US Government",
    "United States of America, Ex. Rel.":                       "US Government",
    "The United States of America, -":                          "US Government",
    "Intervenor United States of America":                      "US Government",
    "The Executive Branch of the United States Government":     "US Government",
    "The United States Senate":                                 "US Government",
    "The United States House of Representatives":               "US Government",
    "in his official capacity as Attorney General of the United States": "US Government",
    "in his official capacity as Director of the United States Bureau of Land Management": "US Government",
    "United States of America/C-IP":                            "US Government",
    "United States of America, ex rel":                         "US Government",
    "Office of the President of the United States of America":  "US Government",
    "The United States District Court":                         "US Government",
    "United States Court of Appeals for the Federal Circuit":   "US Government",
    "U.S. Doj United States Of America":                        "US Government",
    "United States of America Ex Rel":                          "US Government",
    "for itself and on behalf of the United States of America": "US Government",
    "United States Army":                                       "US Government",
    "United States International Trade Commission":             "US Government",
    # ── General Electric remaining variants ───────────────
    "General Electric Capital Services, Inc.":                          "General Electric Co.",
    "General Electric Consumer Finance, Inc.":                          "General Electric Co.",
    "General Electric Company d/b/a GE Healthcare":                     "General Electric Co.",
    "General Electric Superabrasives, Inc.":                            "General Electric Co.",
    "General Electric Enterprise Solutions, A Division Of General Electric, Co.": "General Electric Co.",
    "A division of General Electric, Co.":                              "General Electric Co.",
    "a division of General Electric Company, a New York corporation,":  "General Electric Co.",
    "General Electric Company, Inc.":                                   "General Electric Co.",
    # ── GE remaining variants ──────────────────────────────
    "GE Lighting Solutions, LLC":                                   "General Electric Co.",
    "GE Healthcare Bio-Sciences, Corp.":                            "General Electric Co.",
    "GE Technology Development, Inc.":                              "General Electric Co.",
    "GE Healthcare, Inc.":                                          "General Electric Co.",
    "GE Money Bank":                                                "General Electric Co.",
    "GE Fanuc Intelligent Platforms, Inc.":                         "General Electric Co.",
    "GE Fanuc Automation Americas, Inc.":                           "General Electric Co.",
    "GE Lighting, LLC":                                             "General Electric Co.",
    "GE Interlogix, Inc.":                                          "General Electric Co.",
    "GE Healthcare":                                                "General Electric Co.",
    "GE Homeland Protection, Inc.":                                 "General Electric Co.",
    "GE Lighting, Inc.":                                            "General Electric Co.",
    "GE Power Systems":                                             "General Electric Co.",
    "GE Healthcare UK, Ltd.":                                       "General Electric Co.",
    "GE Intelligent Platforms Embedded Systems, Inc.":              "General Electric Co.",
    "GE Capital Retail Bank":                                       "General Electric Co.",
    "GE Capital Bank":                                              "General Electric Co.",
    "GE Energy, LLC":                                               "General Electric Co.",
    "GE Capital Fleet Services":                                    "General Electric Co.",
    "GE Specialty Chemicals, Inc.":                                 "General Electric Co.",
    "GE Fanuc Automation, Corp.":                                   "General Electric Co.",
    "GE Medical Systems":                                           "General Electric Co.",
    "GE Lighting":                                                  "General Electric Co.",
    "GE Lighting Systems, LLC":                                     "General Electric Co.",
    "GE Lighting Systems, Inc.":                                    "General Electric Co.",
    "GE Healthcare, Ltd.":                                          "General Electric Co.",
    "GE Energy and Industrial Services, Inc.":                      "General Electric Co.",
    "GE Intelligent Platforms, Inc.":                               "General Electric Co.",
    "GE Asset Intelligence, LLC":                                   "General Electric Co.",
    "GE Energy Management Services, Inc.":                          "General Electric Co.",
    "GE Medical Systems Ultrasound and Primary Care Diagnostics, LLC": "General Electric Co.",
    "GE Healthcare Austria GMBH & CO OG":                          "General Electric Co.",
    "GE Medical Systems Information Technologies, Inc.":            "General Electric Co.",
    "GE Home Electric Products, Inc.":                              "General Electric Co.",
    "GE Transportation Systems Global Signaling, LLC":              "General Electric Co.",
    "GE Plastics":                                                  "General Electric Co.",
    "a division of GE, Inc.":                                       "General Electric Co.",
    "GE, Inc.":                                                     "General Electric Co.",
    "GE Harris Railway Electronics, LLC":                           "General Electric Co.",
    "GE-Harris Railway Electronics Services, LLC":                  "General Electric Co.",
    "GE Superabrasives, Inc.":                                      "General Electric Co.",
    "CIF Licensing, LLC d/b/a GE Licensing":                        "General Electric Co.",
    "GE Capital Financial, Inc.":                                   "General Electric Co.",
    "GE Consumer Finance, Inc.":                                    "General Electric Co.",
    "GE Money Bank F S B":                                          "General Electric Co.",
    "GE Medical Systems Kretztechnik GMBH & Co. OHG":              "General Electric Co.",
    "GE Grid Solutions, LLC":                                       "General Electric Co.",
    "Ge Intelligent Platforms Embedded Systems, Inc.":              "General Electric Co.",
    # ── Flextronics remaining variants ────────────────────
    "Flextronics America, LLC":             "Flextronics International, Ltd.",
    "Flextronics Global Services":          "Flextronics International, Ltd.",
    "Flextronics International USA, Inc.":  "Flextronics International, Ltd.",
    "Flextronics Automotive USA, Inc.":     "Flextronics International, Ltd.",

}

    








def apply_org_map_3b(name: str) -> str:
    if not isinstance(name, str):
        return name
    if name in DIRECT_MAP_3b:
        return DIRECT_MAP_3b[name]
    for pattern, replacement in REGEX_RULES_3b:
        if re.match(pattern, name, flags=re.IGNORECASE):
            return replacement
    return name

names['standardized_org_name'] = names['standardized_org_name'].apply(apply_org_map_3b)

In [ ]:
# Create a mask for anything containing "Hitachi"
hitachi_mask = names['standardized_org_name'].str.contains(r'^Hitachi\b', case=False, na=False)

# Standardize the whole group at once
names.loc[hitachi_mask, 'standardized_org_name'] = 'Hitachi, Ltd.'

In [ ]:
# Create a mask for anything containing "Hitachi"
medtronic_mask = names['standardized_org_name'].str.contains(r'^Medtronic\b', case=False, na=False)

# Standardize the whole group at once
names.loc[medtronic_mask, 'standardized_org_name'] = 'Medtronic, Inc.'

In [ ]:
bestbuy_mask = names['standardized_org_name'].str.contains(r'^Best Buy\b', case=False, na=False)

# Standardize the whole group at once
names.loc[bestbuy_mask, 'standardized_org_name'] = 'Best Buy, Inc.'

In [ ]:
lenovo_mask = names['standardized_org_name'].str.contains(r'^Lenovo\b', case=False, na=False)

# Standardize the whole group at once
names.loc[lenovo_mask, 'standardized_org_name'] = 'Lenovo, Inc.'

In [ ]:
# \b ensures "Acer" is a whole word
# ^ ensures it is at the start of the string (optional, depending on your data)
acer_mask = names['standardized_org_name'].str.contains(r'\bAcer\b', case=False, na=False)

# Standardize the entries
names.loc[acer_mask, 'standardized_org_name'] = 'Acer, Inc.'

In [ ]:
medtronic_mask = names['standardized_org_name'].str.contains(
    r'par', case=False, na=False
)
print(names[medtronic_mask]['standardized_org_name'].value_counts().to_string())

standardized_org_name
Par Pharmaceutical Inc.                                                                                                                                                                                                                                           365
Parallel Networks, LLC                                                                                                                                                                                                                                            219
Kohl's Department Stores, Inc.                                                                                                                                                                                                                                     62
United Parcel Service, Inc.                                                                                                                                                                     

In [ ]:
# 1. Calculate the value counts
counts = names['standardized_org_name'].value_counts()

# 2. Filter for entries between 250 and 300 (inclusive)
filtered_counts = counts[(counts >= 350) & (counts <= 400)]

# 3. Display the results
print(filtered_counts)

standardized_org_name
TQP Development, LLC       400
Lenovo, Inc.               394
Monsanto Co.               388
Johnson & Johnson          381
IBM Corp.                  369
Par Pharmaceutical Inc.    365
eDekka, LLC                365
Name: count, dtype: int64


In [ ]:

# List of entity types that should result in a NaN standardized name
null_types = ['Independent', 'Consolidated', 'Unknown', 'Role in Case Terminated']

# Apply NaN to standardized_org_name where entity_type is in our list
names.loc[names['entity_type'].isin(null_types), 'standardized_org_name'] = np.nan

In [ ]:
print(names['standardized_org_name'].nunique())

93522


In [ ]:
names['entity_type'].value_counts()

entity_type
Organization               423919
Independent                 75643
Role in Case Terminated     50728
Consolidated                 5484
Unknown                      3958
Name: count, dtype: int64

In [ ]:
names.to_csv('names_cleaned_v4.csv', index=False)

In [ ]:
with open('similar_groups.txt4', 'w') as f:
    for base, variants in list(similar_groups.items())[:200]:
        f.write(f"Base: {base}\n")
        f.write(f"Variants: {list(variants)}\n")
        f.write("---\n")

In [ ]:
counts = names['standardized_org_name'].value_counts()
rare = counts[counts < 5].index
sample = names[names['standardized_org_name'].isin(rare)]['standardized_org_name'].drop_duplicates().sample(100, random_state=42)
print(sample.to_string())

60161                                             The Coca-Cola Company,
470144                                      Giles Environmental Services
526526                                              Greenway Health, LLC
483338                           GE Energy and Industrial Services, Inc.
374843                                                    J&D Associates
462204    a New York Corporationdoing business asElectronic Mailbox, The
559691                                               Actavis Pharma,INC.
454796                              BeautyBank Inc. d/b/a Good Skin Labs
177492                                           Inc. Medical Components
165803                               Flextronics International USA, Inc.
165115                                                 K2 Concepts, Inc.
442561                     a Division of Pierceton Rubber Products, Inc.
403143                                                        NGHT, Inc.
280686                 O'Reilly Automotive, Inc. Db

In [ ]:
names['standardized_org_name'].nunique()

94006

In [ ]:
targets = [
    'The Walt Disney Company',
    'Adidas',
    'DirecTV LLC.',
    'Daewoo'
]

names['standardized_org_name'].value_counts().loc[targets]

standardized_org_name
The Walt Disney Company    167
Adidas                     204
DirecTV LLC.               199
Daewoo                      53
Name: count, dtype: int64

In [ ]:
usgov_mask = names['standardized_org_name'].str.contains(
    r'United States Department of', case=False, na=False
)
names.loc[usgov_mask, 'standardized_org_name'] = 'US Government'

In [ ]:
usoa_mask = names['standardized_org_name'].str.match(
    r'^United States of America$', case=False, na=False
)
names.loc[usoa_mask, 'standardized_org_name'] = 'US Government'

In [ ]:
uspto_pattern = (
    r'(?i)\b('
    r'(united\s+states|us)\b.*patent.*trademark.*office'
    r'|patent.*trademark.*office.*(united\s+states|us)'
    r')\b'
)

uspto_mask = names['standardized_org_name'].str.contains(
    uspto_pattern,
    na=False,
    regex=True
)

names.loc[uspto_mask, 'standardized_org_name'] = 'US Government'

/var/folders/51/1bmpv68x1h3gvxqw6fktjqh80000gn/T/ipykernel_64430/2593689295.py:8: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  uspto_mask = names['standardized_org_name'].str.contains(


In [ ]:
usa_mask = names['standardized_org_name'].str.contains(
    r'^\s*united\s+states\s+of\s+america\s*$',
    case=False,
    na=False,
    regex=True
)

names.loc[usa_mask, 'standardized_org_name'] = 'US Government'

In [ ]:
names['standardized_org_name'] = (
    names['standardized_org_name']
    .str.replace(r'\s+', ' ', regex=True)  # collapse weird whitespace
    .str.strip()
)

In [ ]:
usa_mask = names['standardized_org_name'].str.fullmatch(
    r'(the\s+)?united\s+states\s+of\s+america',
    case=False
)

names.loc[usa_mask, 'standardized_org_name'] = 'US Government'

In [ ]:
us_mask = names['standardized_org_name'].str.contains(r'flextronics', case=False, na=False)
print(names[us_mask]['standardized_org_name'].value_counts().to_string())

standardized_org_name
Flextronics International, Ltd.    13


In [ ]:
print(names['standardized_org_name'].nunique())

93841


In [ ]:
# Clean up any list entries first
names['standardized_org_name'] = names['standardized_org_name'].apply(
    lambda x: x[0] if isinstance(x, list) else x
)

unique_names = sorted(names['standardized_org_name'].dropna().unique().tolist())
print(f"Total unique names: {len(unique_names)}")

Total unique names: 93827


In [ ]:
unique_names = sorted(names['standardized_org_name'].dropna().unique().tolist())

In [ ]:
print(names['standardized_org_name'].nunique())

93065


In [ ]:
import json
import time
import re
import anthropic
from collections import Counter

client = anthropic.Anthropic()

# -----------------------------
# DATA PREP
# -----------------------------
unique_names = sorted(
    names['standardized_org_name']
    .dropna()
    .unique()
    .tolist()
)

print(f"Unique names remaining: {len(unique_names)}")

BATCH_SIZE = 300

# -----------------------------
# OUTPUT STRUCTURES
# -----------------------------
consolidation_map = {}      # final lookup: variant -> canonical
merge_decisions = []        # full audit trail for validation
failed_batches = []

total_input_tokens = 0
total_output_tokens = 0


# -----------------------------
# SAFE JSON PARSER
# -----------------------------
def parse_json_response(text: str) -> dict:
    """Strict + recoverable JSON parser for LLM output."""
    
    text = text.strip()

    # remove markdown fences if present
    text = re.sub(r'^```[\w]*\n?', '', text)
    text = re.sub(r'\n?```$', '', text).strip()

    # 1. strict parse
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # 2. try extracting JSON block
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass

    # 3. fail loudly (prevents silent corruption)
    raise ValueError(f"Invalid JSON from model:\n{text[:500]}")


# -----------------------------
# MAIN LOOP
# -----------------------------
for i, batch_start in enumerate(range(0, len(unique_names), BATCH_SIZE)):
    
    batch_names = unique_names[batch_start:batch_start + BATCH_SIZE]
    names_text = "\n".join(batch_names)

    print(f"\nBatch {i+1} | names {batch_start}–{batch_start + len(batch_names)}")

    # cost estimate (rough but fine for monitoring)
    estimated_cost = (
        (total_input_tokens / 1_000_000 * 3) +
        (total_output_tokens / 1_000_000 * 15)
    )
    print(f"Estimated cost so far: ${estimated_cost:.2f}")

    if estimated_cost > 9.00:
        print("Stopping early due to budget threshold.")
        break

    try:
        response = client.messages.create(
            model="claude-sonnet-4-5",
            max_tokens=4096,
            messages=[{
                "role": "user",
                "content": f"""
You are performing entity resolution on corporate names from US patent litigation records.

Task:
Identify names that refer to the SAME legal corporate entity.

Rules:
- Only merge when HIGH confidence they are identical entities
- Do NOT merge by industry similarity
- Do NOT merge parent/subsidiary unless clearly identical entity
- Keep distinct brands separate even under same parent company
- Avoid over-consolidation

For each group, assign a confidence score (0.0–1.0) reflecting how certain you are
that all variants refer to the same entity. Use these guidelines:
  1.0 — Exact match or trivial punctuation/abbreviation difference (e.g. "IBM" / "I.B.M.")
  0.9 — Very high confidence, minor formatting variation (e.g. "Inc." vs "Incorporated")
  0.7–0.8 — High confidence, slightly ambiguous (e.g. abbreviated vs full name)
  0.5–0.6 — Moderate confidence, some uncertainty
  below 0.5 — Do not merge; keep as separate entities

Return ONLY valid JSON in this format:

{{
  "Canonical Company Name": {{
    "variants": ["variant1", "variant2"],
    "confidence": 0.95
  }}
}}

No markdown. No explanation.

Names:
{names_text}
"""
            }]
        )

        total_input_tokens += response.usage.input_tokens
        total_output_tokens += response.usage.output_tokens

        text = response.content[0].text
        batch_map = parse_json_response(text)

        # -----------------------------
        # PROCESS RESULTS
        # -----------------------------
        for canonical, data in batch_map.items():

            # support both new format {variants: [], confidence: float}
            # and old flat format {canonical: [variants]} for safety
            if isinstance(data, list):
                variants = data
                confidence = None
            elif isinstance(data, dict):
                variants = data.get("variants", [])
                confidence = data.get("confidence", None)
            elif isinstance(data, str):
                variants = [data]
                confidence = None
            else:
                variants = []
                confidence = None

            for v in variants:

                # final mapping
                consolidation_map[v] = canonical

                # audit trail with confidence score
                merge_decisions.append({
                    "source": v,
                    "target": canonical,
                    "batch_id": i,
                    "confidence": confidence,
                })

        batch_merges = sum(
            len(d.get("variants", d) if isinstance(d, dict) else d)
            for d in batch_map.values()
        )
        print(f"  Merges this batch: {batch_merges}")

        # confidence score summary for the batch
        batch_confidences = [
            d["confidence"] for d in merge_decisions
            if d["batch_id"] == i and d["confidence"] is not None
        ]
        if batch_confidences:
            avg_conf = sum(batch_confidences) / len(batch_confidences)
            min_conf = min(batch_confidences)
            print(f"  Confidence — avg: {avg_conf:.2f}, min: {min_conf:.2f}")

        # -----------------------------
        # CHECKPOINTING
        # -----------------------------
        with open("consolidation_map_checkpoint.json", "w") as f:
            json.dump(consolidation_map, f, indent=2)

        with open("merge_decisions_checkpoint.json", "w") as f:
            json.dump(merge_decisions, f, indent=2)

        time.sleep(1)

    except Exception as e:
        print(f"  Batch {i+1} failed: {e}")
        failed_batches.append(i)

        time.sleep(5)


# -----------------------------
# FINAL OUTPUTS
# -----------------------------
final_cost = (
    (total_input_tokens / 1_000_000 * 3) +
    (total_output_tokens / 1_000_000 * 15)
)

print("\nDONE")
print(f"Total merge decisions: {len(merge_decisions)}")
print(f"Estimated cost: ${final_cost:.2f}")

if failed_batches:
    print(f"Failed batches: {failed_batches}")

# confidence score summary across all batches
all_confidences = [d["confidence"] for d in merge_decisions if d["confidence"] is not None]
if all_confidences:
    conf_counter = Counter(round(c, 1) for c in all_confidences)
    print(f"\nConfidence score distribution:")
    for score in sorted(conf_counter.keys(), reverse=True):
        print(f"  {score:.1f}: {conf_counter[score]} decisions")
    print(f"  Overall avg: {sum(all_confidences) / len(all_confidences):.2f}")

with open("consolidation_map.json", "w") as f:
    json.dump(consolidation_map, f, indent=2)

with open("merge_decisions.json", "w") as f:
    json.dump(merge_decisions, f, indent=2)

print("\nSaved:")
print("- consolidation_map.json")
print("- merge_decisions.json")
print("- checkpoint files")

Unique names remaining: 92298

Batch 1 | names 0–300
Estimated cost so far: $0.00
  Merges this batch: 2
  Confidence — avg: 1.00, min: 1.00

Batch 2 | names 300–600
Estimated cost so far: $0.01


KeyboardInterrupt: 

In [ ]:
print(f"Consolidations so far: {len(consolidation_map)}")
consolidation_map

Consolidations so far: 2


{'21St Century Insurance And Financial Services, Inc.': '21st Century Insurance and Financial Services, Inc.',
 '21st Century Insurance and Financial Services, Inc.': '21st Century Insurance and Financial Services, Inc.'}

In [ ]:
names=pd.read_csv('names_cleaned_v4.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'names_cleaned_v4.csv'

In [ ]:
unique_names = sorted(names['standardized_org_name'].dropna().unique().tolist())

In [ ]:
# Remove the bad consolidation
bad_keys = [k for k in consolidation_map if 'Abraxis' in k or 'Abrika' in k]
for k in bad_keys:
    del consolidation_map[k]
print(f"Removed {len(bad_keys)} bad entries")

# Flatten the map — some values are lists, need to handle both formats
flat_map = {}
for key, value in consolidation_map.items():
    variants = key.split('|')
    if isinstance(value, str):
        canonical = value
    else:
        canonical = variants[0]
    
    for variant in variants:
        variant = variant.strip()
        if variant != canonical:
            flat_map[variant] = canonical

print(f"Total variant→canonical mappings: {len(flat_map)}")

# Apply in place
names['standardized_org_name'] = names['standardized_org_name'].map(flat_map).fillna(names['standardized_org_name'])

# Sanity check
changed = (names['standardized_org_name'] != names['standardized_org_name'].map(flat_map).fillna(names['standardized_org_name'])).sum()
print(f"Rows changed: {changed}")

Removed 1 bad entries
Total variant→canonical mappings: 811
Rows changed: 135813


In [ ]:
print(f"\nDone. Total consolidations: {len(consolidation_map)}")


Done. Total consolidations: 1581


In [ ]:
import json

# Parse the response
text = response.content[0].text.strip()

try:
    test_map = json.loads(text)
except:
    test_map = eval(text)

print(f"Consolidations found: {len(test_map)}")
for k, v in test_map.items():
    print(f"  {k} -> {v}")

# Apply
names['standardized_org_name'] = names['standardized_org_name'].map(test_map).fillna(names['standardized_org_name'])
print("Applied.")

Consolidations found: 14
  1 & 1 Internet, Inc. -> ['1&1 Internet, Inc.']
  02 Micro Inc., Ltd. -> ['02 Micro International, Ltd.', '02 Micro, Inc.']
  20Th Century Plastics, Inc. -> ['20th Century Plastic, Inc.', '21st Century Plastics, Inc.']
  21 Srl -> ['21 srl']
  21st Century Insurance Company of California -> ['21St Century Insurance Company Of California']
  21st Century Insurance Company of the Southwest -> ['21St Century Insurance Company Of The Southwest']
  21st Century Insurance and Financial Services, Inc. -> ['21St Century Insurance And Financial Services, Inc.', '21st Century Insurance and Financial Service, Inc.']
  21st Century National Insurance, Co. -> ['21St Century National Insurance, Co.']
  21st Century North America Insurance, Co. -> ['21St Century North America Insurance, Co.']
  24/7 Real Media, Inc. -> ['24/7 Real Media,Inc.,']
  2K Sports, Inc. -> ['2KSports, Inc.']
  2Wire, Inc. -> ['2Wire,Inc.,']
  1st Technology, LLC -> ['1ST Technology, LLC']
  3COM, Co

In [ ]:
print(names['standardized_org_name'].nunique())

92298


In [ ]:
print(names['standardized_org_name'].value_counts())

standardized_org_name
Non-Specific Entity       45414
Samsung Electronics        2237
Actavis Inc.               1795
Sony Corp.                 1701
LG Electronics             1397
                          ...  
Suntek Systems, Inc.          1
Trans-Expedite, Inc.          1
OSMI, Inc.                    1
Stellar Materials, LLC        1
L & L Wings, Inc.             1
Name: count, Length: 92298, dtype: int64


In [ ]:
names.to_csv('names_cleaned_v5.csv', index=False)

In [29]:
names.head(20)


,case_row_id,case_number,party_row_count,party_type,name_row_count,name,party_type_original,case_number_clean,entity_type,standardized_org_name,gics_sector
0,1,0:79-cv-06704-JCP,1,Plaintiff,1,Burroghs Wellcome Co.,Plaintiff,0:1979-cv-06704,Organization,"Burroghs Wellcome, Co.",Unknown
1,1,0:79-cv-06704-JCP,2,Defendant,2,Generix Drug Corp.,Defendant,0:1979-cv-06704,Organization,"Generix Drug, Corp.",Health Care
2,3,0:83-cv-06860-JAG,3,Plaintiff,3,Kenneth R. Cornwall,Plaintiff,0:1983-cv-06860,Independent,NaN,Unknown
3,3,0:83-cv-06860-JAG,4,Defendant,4,"U. S. COnstruction Manufacturing, Inc.",Defendant,0:1983-cv-06860,Organization,"U. S. COnstruction Manufacturing, Inc.",Industrials
4,4,0:84-cv-06456-KLR,5,Plaintiff,5,"Monte Carlo Hairpieces, Inc.",Plaintiff,0:1984-cv-06456,Organization,"Monte Carlo Hairpieces, Inc.",Unknown
5,4,0:84-cv-06456-KLR,6,Plaintiff,6,James L. Waters,Plaintiff,0:1984-cv-06456,Independent,NaN,Unknown
6,4,0:84-cv-06456-KLR,7,Defendant,7,On-Rite Hairpiece Company,Defendant,0:1984-cv-06456,Organization,"On-Rite Hairpiece, Co.",Unknown
7,4,0:84-cv-06456-KLR,8,Defendant,8,Andrew O. Wright,Defendant,0:1984-cv-06456,Independent,NaN,Unknown
8,5,0:84-cv-06726-WMH,9,Plaintiff,9,"Monaco Del, Rocco A. Sr.",Plaintiff,0:1984-cv-06726,Independent,NaN,Unknown
9,5,0:84-cv-06726-WMH,10,Plaintiff,10,Frances Monaco Del,Plaintiff,0:1984-cv-06726,Independent,NaN,Unknown


In [30]:
names = names.rename(columns={'standardized_org_name': 'brand'})

In [31]:
names.head(5)

,case_row_id,case_number,party_row_count,party_type,name_row_count,name,party_type_original,case_number_clean,entity_type,brand,gics_sector
0,1,0:79-cv-06704-JCP,1,Plaintiff,1,Burroghs Wellcome Co.,Plaintiff,0:1979-cv-06704,Organization,"Burroghs Wellcome, Co.",Unknown
1,1,0:79-cv-06704-JCP,2,Defendant,2,Generix Drug Corp.,Defendant,0:1979-cv-06704,Organization,"Generix Drug, Corp.",Health Care
2,3,0:83-cv-06860-JAG,3,Plaintiff,3,Kenneth R. Cornwall,Plaintiff,0:1983-cv-06860,Independent,NaN,Unknown
3,3,0:83-cv-06860-JAG,4,Defendant,4,"U. S. COnstruction Manufacturing, Inc.",Defendant,0:1983-cv-06860,Organization,"U. S. COnstruction Manufacturing, Inc.",Industrials
4,4,0:84-cv-06456-KLR,5,Plaintiff,5,"Monte Carlo Hairpieces, Inc.",Plaintiff,0:1984-cv-06456,Organization,"Monte Carlo Hairpieces, Inc.",Unknown
